
# PSU Esports Local RAG - Qwen3 4B

Notebook นี้เป็น pipeline เริ่มต้นสำหรับทำ Local RAG Chatbot โดยใช้ `qwen3:4b` ผ่าน Ollama และใช้ข้อมูล chunks ที่ scrape จากเว็บ PSU Esports แล้ว

เป้าหมายของ notebook นี้:

1. โหลดข้อมูล chunks
2. ตรวจ/ซ่อม encoding ภาษาไทยที่เพี้ยน
3. สร้าง local embeddings
4. เก็บลง Chroma vector database
5. retrieve context
6. ให้ Qwen3 4B ตอบจาก context
7. ทดสอบด้วย ground truth เบื้องต้น


## 0. ติดตั้ง dependencies

รัน cell นี้ครั้งแรกครั้งเดียว ถ้าติดตั้งแล้วข้ามได้

In [1]:
%pip install -U pandas requests tqdm chromadb sentence-transformers scikit-learn numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Config

ตรวจว่า Ollama เปิดอยู่ และมีโมเดล `qwen3:4b` แล้ว

In [2]:
from pathlib import Path
import json
import re
import time
from datetime import datetime

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_CHUNKS_PATH = PROJECT_DIR / "data" / "raw" / "all_sections_rag_chunks.jsonl"
OPTIMIZED_CHUNKS_PATH = PROJECT_DIR / "data" / "processed" / "optimized_chunks.jsonl"
PROCESSED_CHUNKS_PATH = PROJECT_DIR / "data" / "processed" / "chunks_clean.jsonl"
VECTOR_DIR = PROJECT_DIR / "data" / "vector_db" / "chroma_psu_esports"
GROUND_TRUTH_PATH = PROJECT_DIR / "ground_truth" / "ground_truth_full.jsonl"
LOG_PATH = PROJECT_DIR / "logs" / "chat_log.jsonl"
RULES_PATH = PROJECT_DIR / "data" / "curated" / "rule_patterns.jsonl"

OLLAMA_BASE_URL = "http://localhost:11434"
FAST_LOCAL_MODEL = "qwen2.5:3b"
QUALITY_LOCAL_MODEL = "qwen3:4b"
LLM_MODEL = FAST_LOCAL_MODEL
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"
COLLECTION_NAME = "psu_esports_local_rag_optimized"

TOP_K = 4
MAX_CONTEXT_CHARS = 3200
MAX_DOC_CHARS = 750
LLM_KEEP_ALIVE = "30m"
LLM_NUM_CTX = 2048
LLM_NUM_PREDICT = 120
LLM_TEMPERATURE = 0.0

print("PROJECT_DIR:", PROJECT_DIR)
print("RAW_CHUNKS_PATH exists:", RAW_CHUNKS_PATH.exists())
print("OPTIMIZED_CHUNKS_PATH exists:", OPTIMIZED_CHUNKS_PATH.exists())

PROJECT_DIR: c:\Users\Chokhun\Downloads\Learn-LLM\15_PSU_Esports_Local_RAG_Qwen3_4B
RAW_CHUNKS_PATH exists: True
OPTIMIZED_CHUNKS_PATH exists: True


c:\Users\Chokhun\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def check_ollama():
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        r.raise_for_status()
        models = [m.get("name") for m in r.json().get("models", [])]
        print("Ollama models:", models)
        if LLM_MODEL not in models:
            print(f"ไม่พบ {LLM_MODEL} ใน Ollama ให้รัน: ollama pull {LLM_MODEL}")
        return models
    except Exception as e:
        print("ยังเชื่อม Ollama ไม่ได้")
        print("ให้เปิด Ollama ก่อน หรือรัน:")
        print(r'& "C:\Users\Chokhun\AppData\Local\Programs\Ollama\ollama.exe" serve')
        print("error:", e)
        return []

models = check_ollama()

Ollama models: ['qwen2.5:3b', 'qwen3:4b']


## 2. Load + repair encoding

ถ้าข้อความไทยแสดงเป็น `à¸...` แปลว่ามี mojibake จากการ decode ผิด ขั้นนี้จะพยายามซ่อมให้ก่อนเอาไป embedding

In [4]:
def count_thai(text: str) -> int:
    return sum(1 for ch in text if "\u0E00" <= ch <= "\u0E7F")

def mojibake_score(text: str) -> int:
    markers = ["à¸", "à¹", "Â", "â€", "ðŸ"]
    return sum(text.count(m) for m in markers)

def repair_mojibake(text: str) -> str:
    if not isinstance(text, str):
        return ""
    original = text.replace("\xa0", " ")
    before_thai = count_thai(original)
    before_moji = mojibake_score(original)
    if before_moji <= 2:
        return original
    try:
        fixed = original.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
        if count_thai(fixed) > before_thai:
            return fixed.replace("\xa0", " ")
    except Exception:
        pass
    return original

def infer_category(record: dict) -> str:
    hay = " ".join([
        str(record.get("id", "")),
        str(record.get("title", "")),
        str(record.get("source_url", "")),
        str(record.get("text", ""))[:500],
    ]).lower()
    if any(k in hay for k in ["ค่าปรับ", "เสียหาย", "ชดเชย", "ระงับสิทธิ์", "penalty", "damage", "suspension"]):
        return "penalty"
    if any(k in hay for k in ["reservation", "booking", "จอง", "check-in", "เช็คอิน"]):
        return "reservation"
    if any(k in hay for k in ["regulation", "rule", "กฎ", "ห้าม", "ค่าปรับ"]):
        return "rules"
    if any(k in hay for k in ["playstation", "nintendo", "vr", "game", "เกม", "equipment"]):
        return "services_games"
    if any(k in hay for k in ["contact", "ติดต่อ"]):
        return "contact"
    if any(k in hay for k in ["competition", "tournament", "แข่งขัน"]):
        return "competition"
    if any(k in hay for k in ["knowledge", "article", "ความรู้"]):
        return "knowledge"
    if any(k in hay for k in ["news", "event", "activity", "ข่าว", "กิจกรรม"]):
        return "events_news"
    return "general"

def parse_jsonl_loose(path: Path):
    records = []
    buffer = ""
    decoder = json.JSONDecoder()
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line.strip() and not buffer:
            continue
        buffer += line + "\n"
        try:
            obj, idx = decoder.raw_decode(buffer.strip())
            records.append(obj)
            buffer = ""
        except json.JSONDecodeError:
            continue
    return records

source_chunks_path = OPTIMIZED_CHUNKS_PATH if OPTIMIZED_CHUNKS_PATH.exists() else RAW_CHUNKS_PATH
print("using chunks:", source_chunks_path)

records = []
for obj in parse_jsonl_loose(source_chunks_path):
    raw_text = obj.get("text", "")
    clean_text = repair_mojibake(raw_text)
    clean_text = re.sub(r"\n{3,}", "\n\n", clean_text).strip()
    if len(clean_text) < 40:
        continue
    obj["text_raw"] = raw_text
    obj["text"] = clean_text
    obj["category"] = obj.get("category") or infer_category(obj)
    obj["priority"] = int(obj.get("priority", 5) or 5)
    obj["source_type"] = obj.get("source_type", "webscraping")
    obj["thai_chars"] = count_thai(clean_text)
    obj["mojibake_score"] = mojibake_score(clean_text)
    records.append(obj)

print("records:", len(records))
df = pd.DataFrame(records)
df[["id", "title", "category", "thai_chars", "mojibake_score"]].head()

using chunks: c:\Users\Chokhun\Downloads\Learn-LLM\15_PSU_Esports_Local_RAG_Qwen3_4B\data\processed\optimized_chunks.jsonl
records: 325


,id,title,category,thai_chars,mojibake_score
0,curated_overview_identity,PSU Esports Studio - Phuket คืออะไร,overview,171,0
1,curated_overview_mission,Mission ของ PSU Esports Studio - Phuket,overview,222,0
2,curated_reservation_advance_time,ต้องจองล่วงหน้า,reservation,84,0
3,curated_reservation_max_sessions,จำนวน session สูงสุดต่อการจอง,reservation,49,0
4,curated_payment_10_minutes,ชำระเงินหลังจอง,reservation,106,0


In [5]:
print(records[0]["text"][:1000])

PSU Esports Studio - Phuket คืออะไร
PSU Esports Studio - Phuket คือศูนย์พัฒนาการเรียนรู้ด้านอีสปอร์ตเพื่อความเป็นเลิศและขับเคลื่อนเศรษฐกิจในพื้นที่ภาคใต้ สาขาภูเก็ต เป็นศูนย์การเรียนรู้ผ่านเกมและอีสปอร์ตของมหาวิทยาลัยสงขลานครินทร์


In [6]:
PROCESSED_CHUNKS_PATH.parent.mkdir(parents=True, exist_ok=True)
with PROCESSED_CHUNKS_PATH.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("saved:", PROCESSED_CHUNKS_PATH)

saved: c:\Users\Chokhun\Downloads\Learn-LLM\15_PSU_Esports_Local_RAG_Qwen3_4B\data\processed\chunks_clean.jsonl


## 3. Create embeddings + Chroma index

ใช้ `multilingual-e5-small` เพราะเบาและเหมาะกับไทย/อังกฤษใน MVP

In [7]:
from sentence_transformers import SentenceTransformer
import chromadb

embedder = SentenceTransformer(EMBEDDING_MODEL)
client = chromadb.PersistentClient(path=str(VECTOR_DIR))
collection = client.get_or_create_collection(name=COLLECTION_NAME)
print("current collection count:", collection.count())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3624.87it/s]


current collection count: 325


In [8]:
import hashlib

def to_chroma_metadata(r: dict) -> dict:
    return {
        "source_url": str(r.get("source_url", "")),
        "title": str(r.get("title", "")),
        "category": str(r.get("category", "general")),
        "source_type": str(r.get("source_type", "webscraping")),
        "section": str(r.get("section", "")),
        "priority": int(r.get("priority", 5) or 5),
        "chunk_index": int(r.get("chunk_index", 0) or 0),
        "tags": ";".join(r.get("tags", []) or []),
        "source_ids": ";".join(r.get("source_ids", []) or []),
    }

def reset_collection():
    global collection
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass
    collection = client.get_or_create_collection(name=COLLECTION_NAME)

def current_data_fingerprint() -> str:
    if source_chunks_path.exists():
        return hashlib.sha256(source_chunks_path.read_bytes()).hexdigest()
    payload = "\n".join(json.dumps(r, ensure_ascii=False, sort_keys=True) for r in records)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

FINGERPRINT_PATH = VECTOR_DIR / f"{COLLECTION_NAME}.fingerprint"
DATA_FINGERPRINT = current_data_fingerprint()
OLD_FINGERPRINT = FINGERPRINT_PATH.read_text(encoding="utf-8").strip() if FINGERPRINT_PATH.exists() else ""

# rebuild ???????????? collection ????, ????? chunk ??????, ??????????? optimized data ???????
REBUILD_INDEX = collection.count() == 0 or collection.count() != len(records) or OLD_FINGERPRINT != DATA_FINGERPRINT

if REBUILD_INDEX:
    reset_collection()
    batch_size = 32
    for start in tqdm(range(0, len(records), batch_size)):
        batch = records[start:start + batch_size]
        ids = [str(r.get("id") or f"chunk-{start+i}") for i, r in enumerate(batch)]
        docs = [r["text"] for r in batch]
        # E5 convention: passages ????? prefix 'passage: '
        embeddings = embedder.encode(["passage: " + d for d in docs], normalize_embeddings=True).tolist()
        metadatas = [to_chroma_metadata(r) for r in batch]
        collection.add(ids=ids, documents=docs, embeddings=embeddings, metadatas=metadatas)
    FINGERPRINT_PATH.write_text(DATA_FINGERPRINT, encoding="utf-8")
    print("rebuilt collection count:", collection.count())
else:
    print("skip rebuild, collection count:", collection.count())


skip rebuild, collection count: 325


## 4. Retriever: vector search + category route + lexical rerank

เทคนิคที่ใส่เพิ่ม:

- route หมวดคำถามก่อนค้น
- vector search ด้วย embeddings
- lexical rerank แบบง่าย เพื่อ boost chunk ที่มีคำตรงกับคำถาม


In [9]:
import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from scripts.rag_runtime_overrides import (
    SERVICE_SCHEDULE_DOC_IDS,
    direct_curated_max_items,
    expand_query,
    is_service_schedule_query,
    lexical_score,
    route_category,
)

def retrieve(query: str, top_k: int = TOP_K):
    search_query = expand_query(query)
    category = route_category(query)
    where = {"category": category} if category else None
    q_emb = embedder.encode(["query: " + search_query], normalize_embeddings=True).tolist()[0]
    n_results = max(top_k * 2, top_k)
    try:
        res = collection.query(query_embeddings=[q_emb], n_results=n_results, where=where)
        # fallback to unfiltered search if the category filter is too narrow
        if not res.get("ids", [[]])[0]:
            res = collection.query(query_embeddings=[q_emb], n_results=n_results)
    except Exception:
        res = collection.query(query_embeddings=[q_emb], n_results=n_results)

    items = []
    for i, doc_id in enumerate(res["ids"][0]):
        doc = res["documents"][0][i]
        meta = res["metadatas"][0][i]
        distance = res["distances"][0][i] if "distances" in res and res["distances"] else 0.0
        priority_boost = 0.02 * float(meta.get("priority", 5) or 5)
        if category:
            curated_boost = 0.12 if meta.get("source_type") == "curated_fact" else 0.0
        else:
            curated_boost = 0.03 if meta.get("source_type") == "curated_fact" else 0.0
        schedule_boost = 3.0 if is_service_schedule_query(search_query) and doc_id in SERVICE_SCHEDULE_DOC_IDS else 0.0
        score = -float(distance) + 0.25 * lexical_score(search_query, doc) + priority_boost + curated_boost + schedule_boost
        items.append({"id": doc_id, "text": doc, "metadata": meta, "distance": distance, "score": score})

    items = sorted(items, key=lambda x: x["score"], reverse=True)[:top_k]
    return items

test_hits = retrieve("PS5 games", top_k=5)
[(h["id"], h["metadata"].get("category"), h["metadata"].get("title"), round(h["score"], 3)) for h in test_hits]


[('curated_games_ps5', 'games', 'เกมบน PlayStation 5', 0.175),
 ('services-our-games-01-037', 'games', 'Services - Our Games', 0.028),
 ('services-our-games-01-022', 'games', 'Services - Our Games', 0.004),
 ('services-our-games-01-032', 'games', 'Services - Our Games', -0.016),
 ('services-our-games-01-031', 'games', 'Services - Our Games', -0.019)]

## 5. Ask Qwen3 4B with RAG context

In [10]:
TH_ANSWER_LABEL = "\u0e04\u0e33\u0e15\u0e2d\u0e1a"
TH_DETAIL_LABEL = "\u0e23\u0e32\u0e22\u0e25\u0e30\u0e40\u0e2d\u0e35\u0e22\u0e14"
TH_SOURCE_LABEL = "\u0e41\u0e2b\u0e25\u0e48\u0e07\u0e02\u0e49\u0e2d\u0e21\u0e39\u0e25"
TH_NOT_FOUND = "\u0e44\u0e21\u0e48\u0e1e\u0e1a\u0e02\u0e49\u0e2d\u0e21\u0e39\u0e25\u0e19\u0e35\u0e49\u0e43\u0e19\u0e10\u0e32\u0e19\u0e02\u0e49\u0e2d\u0e21\u0e39\u0e25\u0e17\u0e35\u0e48\u0e21\u0e35"

SYSTEM_PROMPT = f"""
You are an AI Chatbot for PSU Esports Studio - Phuket.
Use only the provided CONTEXT. Do not guess or invent missing details.

Rules:
1. Answer only in the same language as the user question. For Thai questions, use Thai only. For English questions, use English only. Do not mix in Chinese or any third language.
2. If the exact answer is not in CONTEXT, politely say that this information was not found in the available knowledge base. If the CONTEXT contains a related confirmed policy or alternative, mention it briefly without guessing.
3. Start immediately with "{TH_ANSWER_LABEL}:" for Thai or "Answer:" for English.
4. Do not write analysis, chain-of-thought, or phrases like "Okay, let's see".
5. Keep the answer concise. For MVP latency, use at most 3 short bullets unless the user asks for details.
6. Include sources from the provided context.
""".strip()

# Rule-based FAQ fast path: repeated FAQ should answer before RAG/LLM.
import sys
SCRIPTS_DIR = PROJECT_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

from rule_matcher import load_rules, match_rule

RULES = load_rules(RULES_PATH)
print("loaded rule patterns:", len(RULES))
print("active LLM model:", LLM_MODEL)
print("fast config:", {"top_k": TOP_K, "num_ctx": LLM_NUM_CTX, "num_predict": LLM_NUM_PREDICT, "max_context_chars": MAX_CONTEXT_CHARS})

def trim_doc_text(text: str, max_chars: int = MAX_DOC_CHARS) -> str:
    text = " ".join((text or "").split())
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "..."

def build_context(hits: list[dict], max_chars: int = MAX_CONTEXT_CHARS) -> str:
    blocks = []
    total = 0
    for i, h in enumerate(hits, 1):
        meta = h["metadata"]
        header = f"[SOURCE {i}] title={meta.get('title')} | category={meta.get('category')} | url={meta.get('source_url')} | id={h.get('id')}"
        body = trim_doc_text(h["text"])
        block = header + "\n" + body
        if total + len(block) > max_chars:
            break
        blocks.append(block)
        total += len(block)
    return "\n\n---\n\n".join(blocks)

def clean_qwen_answer(text: str) -> str:
    if not text:
        return ""
    if "</think>" in text:
        text = text.split("</think>", 1)[-1]
    text = re.sub(r"<think>.*", "", text, flags=re.DOTALL).strip()
    markers = [f"{TH_ANSWER_LABEL}:", "Answer:", f"{TH_DETAIL_LABEL}:", "Final:"]
    marker_positions = [text.find(m) for m in markers if text.find(m) >= 0]
    if marker_positions:
        text = text[min(marker_positions):]
    text = re.sub(r"^(Okay|Let me|I need|First,|Looking at|The user|We need).*?(?=Answer:|Final:)", "", text, flags=re.DOTALL | re.I).strip()
    return text.strip()

def prepare_messages_for_model(messages: list[dict]) -> list[dict]:
    # Qwen3 may still think aloud; /no_think is harmless for other Qwen models.
    prepared = []
    for msg in messages:
        content = msg.get("content", "")
        if msg.get("role") == "user" and LLM_MODEL.startswith("qwen3"):
            content = "/no_think\n" + content
        prepared.append({**msg, "content": content})
    return prepared

def ask_ollama(messages, temperature: float | None = None, num_ctx: int | None = None, num_predict: int | None = None):
    payload = {
        "model": LLM_MODEL,
        "messages": prepare_messages_for_model(messages),
        "stream": False,
        "think": False,
        "keep_alive": LLM_KEEP_ALIVE,
        "options": {
            "temperature": LLM_TEMPERATURE if temperature is None else temperature,
            "num_ctx": LLM_NUM_CTX if num_ctx is None else num_ctx,
            "num_predict": LLM_NUM_PREDICT if num_predict is None else num_predict,
        },
    }
    r = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=payload, timeout=120)
    r.raise_for_status()
    data = r.json()
    content = data.get("message", {}).get("content", "")
    answer = clean_qwen_answer(content)
    if not answer:
        answer = "The model did not produce a final answer. Try increasing LLM_NUM_PREDICT or using a shorter context."
    return answer

def save_chat_log(row: dict):
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

def is_thai_question(text: str) -> bool:
    return any("\u0E00" <= ch <= "\u0E7F" for ch in text)

def format_rule_answer(question: str, rule_match: dict) -> str:
    source_url = rule_match.get("source_url")
    source_ids = ", ".join(rule_match.get("source_ids") or [])
    source_note = f"{source_url or 'ฐานข้อมูลที่มี'} ({rule_match.get('rule_id')}"
    if source_ids:
        source_note += f" / {source_ids}"
    source_note += ")"

    if is_thai_question(question):
        return f"""{TH_ANSWER_LABEL}:
{rule_match.get('answer', '')}

{TH_SOURCE_LABEL}:
- {source_note}""".strip()

    return f"""Answer:
{rule_match.get('answer', '')}

Sources:
- {source_note}""".strip()

def rule_match_to_hit(rule_match: dict) -> dict:
    return {
        "id": rule_match.get("rule_id"),
        "text": rule_match.get("answer", ""),
        "metadata": {
            "title": rule_match.get("intent"),
            "category": rule_match.get("category"),
            "source_url": rule_match.get("source_url"),
            "source_type": "rule_fast_path",
            "matched_pattern": rule_match.get("matched_pattern"),
            "source_ids": ";".join(rule_match.get("source_ids") or []),
        },
        "distance": 0.0,
        "score": 999.0,
    }

def is_not_found_answer(answer: str) -> bool:
    if not answer:
        return True
    a = answer.lower()
    return TH_NOT_FOUND in answer or "not found" in a or "available knowledge base" in a

def clean_direct_text(hit: dict) -> str:
    text = (hit.get("text") or "").strip()
    title = str(hit.get("metadata", {}).get("title", "")).strip()
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if len(lines) > 1 and title and lines[0] == title:
        lines = lines[1:]
    return " ".join(lines).strip()

BOOKING_STEP_IDS = (
    "curated_booking_steps",
    "curated_reservation_advance_time",
    "curated_payment_10_minutes",
    "curated_user_info_required",
)
BOOKING_STEP_KEYWORDS = ["ขั้นตอน", "วิธีจอง", "จองยังไง", "จองอย่างไร", "สรุป", "กรอกข้อมูล", "แนบสลิป", "สลิป", "โอนเงิน", "how to", "steps", "booking process", "slip", "transfer"]

def is_booking_steps_question(question: str) -> bool:
    q = question.lower()
    return (any(k in q for k in ["จอง", "booking", "reservation"]) and any(k in q for k in BOOKING_STEP_KEYWORDS)) or ("กรอกข้อมูล" in q and "สลิป" in q)

SERVICE_FEE_TABLE_IDS = ("curated_service_fee_2026_full_table",)
SERVICE_FEE_SUMMARY_KEYWORDS = ["ราคาทั้งหมด", "ค่าบริการทั้งหมด", "ค่าใช้จ่าย", "ค่าเล่นแต่ละ", "ค่าเล่นทั้งหมด", "service fee", "ตารางราคา", "ตารางค่าบริการ", "ราคาแต่ละ", "เรทราคา", "rates"]

def is_service_fee_summary_question(question: str) -> bool:
    q = question.lower()
    return any(k in q for k in SERVICE_FEE_SUMMARY_KEYWORDS) or ("ราคา" in q and "ทั้งหมด" in q)

SERVICE_SCHEDULE_IDS = (
    "curated_schedule_morning",
    "curated_schedule_afternoon",
    "curated_reservation_schedule_monday_morning",
    "curated_reservation_schedule_friday_maintenance",
)
SERVICE_SCHEDULE_KEYWORDS = ["เปิดถึง", "เปิดกี่โมง", "ปิดถึงกี่โมง", "ปิดกี่โมง", "เปิดปิดกี่โมง", "เปิด-ปิดกี่โมง", "เวลาเปิด", "เวลาปิด", "เวลาทำการ", "เวลาให้บริการ", "ตารางบริการ", "รอบเช้า", "ช่วงเช้า", "ตอนเช้า", "รอบบ่าย", "ช่วงบ่าย", "24 ชั่วโมง", "เปิด 24", "service hours", "opening hours", "closing hours", "morning", "afternoon", "24 hours"]

def is_service_schedule_question(question: str) -> bool:
    q = question.lower()
    return any(k in q for k in SERVICE_SCHEDULE_KEYWORDS)

def curated_hits_from_records(record_ids: tuple[str, ...]) -> list[dict]:
    lookup = {r.get("id"): r for r in globals().get("records", [])}
    hits = []
    for rid in record_ids:
        r = lookup.get(rid)
        if not r:
            continue
        hits.append({
            "id": r.get("id"),
            "text": r.get("text", ""),
            "metadata": {
                "title": r.get("title", ""),
                "category": r.get("category", ""),
                "source_url": r.get("source_url", ""),
                "source_type": r.get("source_type", "curated_fact"),
            },
            "distance": 0.0,
            "score": 100.0,
        })
    return hits

def force_direct_hits_for_question(question: str, hits: list[dict]) -> list[dict]:
    if is_service_fee_summary_question(question):
        fee_hits = curated_hits_from_records(SERVICE_FEE_TABLE_IDS)
        if fee_hits:
            return fee_hits
    if is_booking_steps_question(question):
        booking_hits = curated_hits_from_records(BOOKING_STEP_IDS)
        if booking_hits:
            return booking_hits
    if is_service_schedule_question(question):
        schedule_hits = curated_hits_from_records(SERVICE_SCHEDULE_IDS)
        if schedule_hits:
            return schedule_hits
    return hits

from scripts.rag_runtime_overrides import direct_curated_max_items

def format_direct_rag_answer(question: str, hits: list[dict]) -> str | None:
    if not hits:
        return None
    if is_service_schedule_question(question):
        schedule_hits = curated_hits_from_records(SERVICE_SCHEDULE_IDS)
        if schedule_hits:
            hits = schedule_hits
        else:
            hits = [h for h in hits if h.get("id") in SERVICE_SCHEDULE_IDS] or hits
    if is_booking_steps_question(question):
        booking_hits = curated_hits_from_records(BOOKING_STEP_IDS)
        if booking_hits:
            hits = booking_hits
    top_meta = hits[0].get("metadata", {})
    top_category = top_meta.get("category")
    routed_category = route_category(question) if "route_category" in globals() else None
    if not routed_category or top_category != routed_category:
        return None
    if top_meta.get("source_type") != "curated_fact":
        return None
    if top_category not in {"overview", "contact", "reservation", "rules", "games", "penalty", "equipment", "services", "service_fee", "events_news", "about_us", "knowledge"}:
        return None

    same_category_hits = [
        h for h in hits
        if h.get("metadata", {}).get("source_type") == "curated_fact"
        and h.get("metadata", {}).get("category") == top_category
    ]
    max_items = direct_curated_max_items(question, top_category)
    selected = same_category_hits[:max_items]
    if not selected:
        return None

    bullets = []
    sources = []
    for h in selected:
        text = clean_direct_text(h)
        if text:
            bullets.append(f"- {text}")
        meta = h.get("metadata", {})
        sources.append(f"- {meta.get('source_url')} ({h.get('id')})")

    # dedupe while preserving order
    bullets = list(dict.fromkeys(bullets))
    if is_service_schedule_question(question):
        direct_schedule = "- ไม่ได้เปิด 24 ชั่วโมง ตารางบริการแบ่งเป็น Morning 09:00–12:00 และ Afternoon 13:00–16:00 โดยดูตามวัน: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* เล่นไม่ได้ และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดตามรอบปกติ 09:00–12:00 และ 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น" if is_thai_question(question) else "- The studio is not open 24 hours. Service sessions are Morning 09:00-12:00 and Afternoon 13:00-16:00. By day: Monday morning 09:00-12:00 is Maintenance* and not playable, while Monday afternoon 13:00-16:00 is open; Tuesday-Thursday follow the regular slots 09:00-12:00 and 13:00-16:00; Friday morning 09:00-12:00 is open, but Friday afternoon 13:00-16:00 is Maintenance** for weekly hardware inspection and cleaning, so it is not playable."
        bullets = [direct_schedule] + [b for b in bullets if b != direct_schedule]
    sources = list(dict.fromkeys(sources))
    if not bullets:
        return None

    if is_thai_question(question):
        return f"""{TH_ANSWER_LABEL}:
{chr(10).join(bullets)}

{TH_SOURCE_LABEL}:
{chr(10).join(sources)}""".strip()

    return f"""Answer:
{chr(10).join(bullets)}

Sources:
{chr(10).join(sources)}""".strip()

def sanitize_model_answer(answer: str, question: str) -> str:
    if not answer:
        return answer
    if is_thai_question(question):
        answer = re.sub(r"[\u4E00-\u9FFF]+[。｡.]?", "", answer)
        answer = re.sub(r"[ \t]{2,}", " ", answer)
        answer = re.sub(r"\n{3,}", "\n\n", answer)
        answer = re.sub(r"\s+[。｡.]\s*$", "", answer).strip()
    return answer.strip()

import math
from difflib import SequenceMatcher

SERVICE_FEE_SOURCE_URL = "https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png"
SERVICE_PRESET_SOURCE_URL = "https://esports.computing.psu.ac.th/wp-json/wbk/v2/get-preset"

SERVICE_PRICING = {
    "pc": {
        "label_th": "PC",
        "duration_min": 60,
        "rates": {},
        "note_th": "หน้าเว็บมีบริการ PC 60 นาที แต่ยังไม่พบราคาค่าบริการ PC ในข้อมูลที่ดึงมาและในรูป Service Fee 2026",
    },
    "ps5": {
        "label_th": "PlayStation 5",
        "duration_min": 60,
        "rates": {"psu": 0, "alumni_student": 50, "adult": 150},
    },
    "nintendo_1_2": {
        "label_th": "Nintendo Switch 1-2 คน",
        "duration_min": 60,
        "rates": {"psu": 0, "alumni_student": 50, "adult": 140},
    },
    "nintendo_3_4": {
        "label_th": "Nintendo Switch 3-4 คน",
        "duration_min": 60,
        "rates": {"psu": 0, "alumni_student": 100, "adult": 280},
    },
    "cockpit": {
        "label_th": "Cockpit",
        "duration_min": 60,
        "rates": {"psu": 0, "alumni_student": 65, "adult": 200},
    },
    "vr_30": {
        "label_th": "VR 30 นาที",
        "duration_min": 30,
        "rates": {"psu": 0, "alumni_student": 190, "adult": 525},
    },
    "vr_60": {
        "label_th": "VR 1 ชั่วโมง",
        "duration_min": 60,
        "rates": {"psu": 0, "alumni_student": 375, "adult": 1050},
    },
}

CUSTOMER_GROUP_LABELS_TH = {
    "psu": "นักศึกษา/บุคลากร PSU",
    "alumni_student": "ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)",
    "adult": "บุคคลทั่วไป",
}

def normalize_thai_digits(text: str) -> str:
    return (text or "").translate(str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789"))

def normalize_alias_text(text: str) -> str:
    q = normalize_thai_digits(text or "").lower()
    q = re.sub(r"p\s*\.?\s*s\s*\.?\s*u\.?", "psu", q)
    replacements = {
        "ม.อ.": "มอ",
        "ม.อ": "มอ",
        "ม. อ.": "มอ",
        "กีโมง": "กี่โมง",
        "เช็คอิน": "เชคอิน",
        "check-in": "checkin",
        "check in": "checkin",
        "ช.ม.": "ชม",
        "ชม.": "ชม",
        "ชั่วโมง": "ชม",
        "ชัวโมง": "ชม",
        "ครึ่งชั่วโมง": "ครึ่งชม",
        "ครึ่ง ชม": "ครึ่งชม",
        "นาที": "นาที",
        "น.ท.": "นาที",
        "เพลย์สเตชั่น": "playstation",
        "เพลสเตชั่น": "playstation",
        "เพลย์ห้า": "ps5",
        "เพลห้า": "ps5",
        "พีซี": "pc",
        "คอมพิวเตอร์": "คอม",
        "วีอาร์": "vr",
        "นินเทนโด": "nintendo",
    }
    for old, new in replacements.items():
        q = q.replace(old, new)
    q = re.sub(r"\bhrs?\b|\bhours?\b", "ชม", q)
    q = re.sub(r"\bmins?\b|\bminutes?\b", "นาที", q)
    return re.sub(r"[\s\._\-/()]+", "", q)

def fuzzy_contains_alias(normalized_query: str, normalized_alias: str, threshold: float = 0.88) -> bool:
    if not normalized_query or not normalized_alias:
        return False
    if normalized_alias in normalized_query:
        return True
    if len(normalized_alias) < 4 or len(normalized_query) < 4:
        return False
    min_len = max(4, len(normalized_alias) - 2)
    max_len = min(len(normalized_query), len(normalized_alias) + 2)
    for size in range(min_len, max_len + 1):
        for start in range(0, len(normalized_query) - size + 1):
            segment = normalized_query[start:start + size]
            if SequenceMatcher(None, segment, normalized_alias).ratio() >= threshold:
                return True
    return False

def alias_match(question: str, aliases: list[str], threshold: float = 0.88) -> bool:
    nq = normalize_alias_text(question)
    for alias in aliases:
        na = normalize_alias_text(alias)
        if fuzzy_contains_alias(nq, na, threshold=threshold):
            return True
    return False

PRICE_WORD_ALIASES = [
    "กี่บาท", "กีบาท", "กี่บาด", "ราคา", "ค่าบริการ", "ค่าเล่น", "ค่าใช้จ่าย",
    "ต้องจ่าย", "จ่ายเท่าไหร่", "เท่าไหร่", "เท่าไร", "เท่ารัย", "คิดเงิน",
    "ฟรี", "price", "cost", "fee", "free", "per hour", "hourly", "ต่อชั่วโมง",
    "ต่อชม", "ต่อรอบ",
]

CUSTOMER_GROUP_ALIASES = {
    "adult": ["บุคคลทั่วไป", "คนทั่วไป", "คนนอก", "ข้างนอก", "ผู้ใหญ่ทั่วไป", "general adult", "adult"],
    "alumni_student": [
        "ศิษย์เก่า", "alumni", "general student", "นักศึกษาทั่วไป", "นักเรียนทั่วไป",
        "นักศึกษาจาก", "นักศึกษาต่าง", "นักศึกษานอก", "นักศึกษามหาลัยอื่น",
        "นักศึกษาต่างมหาลัย", "นักเรียนต่างมหาลัย", "เด็กต่างมหาลัย",
        "นักศึกษาต่างมหาวิทยาลัย", "นักเรียนต่างมหาวิทยาลัย",
        "นักศึกษามหาวิทยาลัยอื่น", "นักเรียนจาก", "ต่างมหาวิทยาลัย", "ต่างมหาลัย",
        "มหาวิทยาลัยอื่น", "มหาลัยอื่น", "ต่างมอ", "ต่างม.อ", "ต่างสถาบัน", "จากกระบี่",
    ],
    "psu": ["นักศึกษา มอ", "นักเรียน มอ", "เด็ก มอ", "นิสิต มอ", "นักศึกษา psu", "นักเรียน psu", "psu student", "psu staff", "บุคลากร psu", "staff psu"],
}

SERVICE_ALIASES = {
    "pc": ["pc", "พีซี", "คอม", "คอมพิวเตอร์", "computer", "gaming pc"],
    "ps5": ["ps5", "playstation", "play station", "เพลย์", "เพลย์ห้า", "เพลห้า", "playstation 5"],
    "nintendo": ["nintendo", "switch", "สวิตช์", "สวิทช์", "นินเทนโด"],
    "cockpit": ["cockpit", "พวงมาลัย", "ขับรถ", "racing", "race"],
    "vr": ["vr", "วีอาร์", "แว่น vr", "virtual reality", "playstation vr2"],
}

def parse_time_range_minutes(question: str) -> tuple[int, int] | None:
    q = normalize_thai_digits(question.lower())
    patterns = [
        r"(\d{1,2})(?:[:.](\d{1,2}))?\s*(?:โมง|น\.?|am|pm)?\s*(?:ถึง|จนถึง|[-–—]|to)\s*(\d{1,2})(?:[:.](\d{1,2}))?\s*(?:โมง|น\.?|am|pm)?",
        r"(\d{1,2})(?:[:.](\d{1,2}))?\s*(?:โมง|น\.?)\s*(\d{1,2})(?:[:.](\d{1,2}))?\s*(?:โมง|น\.?)",
    ]
    for pattern in patterns:
        m = re.search(pattern, q)
        if not m:
            continue
        after_match = q[m.end():m.end() + 16]
        if re.match(r"\s*(คน|ท่าน|persons?|people|players?)", after_match):
            continue
        sh = int(m.group(1))
        sm = int(m.group(2) or 0)
        eh = int(m.group(3))
        em = int(m.group(4) or 0)
        start = sh * 60 + sm
        end = eh * 60 + em
        if end <= start:
            end += 24 * 60
        if 0 < end - start <= 24 * 60:
            return start, end
    return None

def detect_pricing_service(question: str) -> str | None:
    q = normalize_thai_digits(question.lower())
    if alias_match(q, SERVICE_ALIASES["pc"], threshold=0.9):
        return "pc"
    if alias_match(q, SERVICE_ALIASES["ps5"], threshold=0.88):
        return "ps5"
    if alias_match(q, SERVICE_ALIASES["nintendo"], threshold=0.88):
        if any(k in q for k in ["3-4", "3 ถึง 4", "4 คน", "3 คน"]):
            return "nintendo_3_4"
        return "nintendo_1_2"
    if alias_match(q, SERVICE_ALIASES["cockpit"], threshold=0.88):
        return "cockpit"
    if alias_match(q, SERVICE_ALIASES["vr"], threshold=0.92):
        q_alias = normalize_alias_text(q)
        if "30" in q_alias or "ครึ่ง" in q_alias:
            return "vr_30"
        return "vr_60"
    return None

def count_pricing_service_mentions(question: str) -> int:
    return sum(1 for variants in SERVICE_ALIASES.values() if alias_match(question, variants, threshold=0.9))

def detect_customer_group(question: str) -> str | None:
    q = normalize_thai_digits(question.lower())
    if alias_match(q, CUSTOMER_GROUP_ALIASES["alumni_student"], threshold=0.86):
        return "alumni_student"
    if alias_match(q, CUSTOMER_GROUP_ALIASES["adult"], threshold=0.88):
        return "adult"
    psu_ref = re.search(r"psu|p\.?s\.?u\.?|มหาวิทยาลัยสงขลานครินทร์|ม\.?\s*อ\.?|(?<![\u0E00-\u0E7F])มอ\.?(?![\u0E00-\u0E7F])", q)
    psu_person_words = ["นักศึกษา", "นักเรียน", "เด็ก", "นิสิต", "student", "staff", "บุคลากร"]
    if psu_ref and (alias_match(q, psu_person_words, threshold=0.86) or "มอ" in q or "ม.อ" in q or "psu" in q):
        return "psu"
    if alias_match(q, CUSTOMER_GROUP_ALIASES["psu"], threshold=0.86):
        return "psu"
    if alias_match(q, ["นักเรียน", "นักศึกษา", "student"], threshold=0.9):
        return "alumni_student"
    return None

def is_price_calculation_question(question: str) -> bool:
    q = normalize_thai_digits(question.lower())
    damage_words = ["พัง", "แตก", "เสียหาย", "ขาด", "เปียก", "รอย", "ค่าปรับ", "ชดเชย", "ซ่อม"]
    if any(w in q for w in damage_words):
        return False
    money_words = PRICE_WORD_ALIASES + ["เสีย", "เสียค่า", "คำนวณ", "calculate"]
    if count_pricing_service_mentions(question) > 1:
        return False
    return detect_pricing_service(question) is not None and alias_match(q, money_words, threshold=0.86)

def format_minutes(minutes: int) -> str:
    hours = minutes // 60
    mins = minutes % 60
    if hours and mins:
        return f"{hours} ชั่วโมง {mins} นาที"
    if hours:
        return f"{hours} ชั่วโมง"
    return f"{mins} นาที"

def calculate_service_price_answer(question: str) -> tuple[str, list[dict]] | None:
    if not is_price_calculation_question(question):
        return None
    time_range = parse_time_range_minutes(question)
    service_key = detect_pricing_service(question)
    if not service_key:
        return None
    service = SERVICE_PRICING[service_key]
    session_min = int(service["duration_min"])
    if time_range:
        start, end = time_range
        duration = end - start
        start_text = f"{(start // 60) % 24:02d}:{start % 60:02d}"
        end_text = f"{(end // 60) % 24:02d}:{end % 60:02d}"
        first_line = f"ช่วงเวลาที่ถามคือ {start_text}-{end_text} = {format_minutes(duration)}"
    else:
        duration = session_min
        first_line = f"คำถามเป็นราคาแบบต่อรอบ/ต่อชั่วโมง จึงคิดเป็น 1 session = {format_minutes(session_min)}"
    sessions = math.ceil(duration / session_min)
    group = detect_customer_group(question)
    rates = service.get("rates", {})

    base_lines = [
        first_line,
        f"บริการ {service['label_th']} คิดเป็นรอบละ {session_min} นาที ดังนั้นต้องใช้ {sessions} session(s)",
    ]

    if not rates:
        group_line = f"- กลุ่มผู้ใช้ที่ถาม: {CUSTOMER_GROUP_LABELS_TH[group]}" if group else "- ยังไม่ทราบกลุ่มผู้ใช้ จึงยังเทียบเรตราคาเฉพาะกลุ่มไม่ได้"
        answer = f"""{TH_ANSWER_LABEL}:
- ราคา PC: ยังไม่พบราคาค่าบริการ PC ในฐานข้อมูล/Service Fee 2026 ที่ดึงมา จึงยังไม่ควรคำนวณยอดเงินบาทแบบฟันธง
{chr(10).join('- ' + line for line in base_lines)}
{group_line}
- จากภาพ Service Fee 2026 ที่มีตอนนี้ มีราคา PlayStation 5, Nintendo Switch, Cockpit และ VR แต่ไม่ปรากฏราคา PC
- ถ้าได้รับราคา PC ต่อ 1 session แล้ว ระบบจะคำนวณได้ทันทีด้วยสูตร: จำนวน session × ราคาต่อ session

{TH_SOURCE_LABEL}:
- {SERVICE_PRESET_SOURCE_URL} (service duration)
- {SERVICE_FEE_SOURCE_URL} (service fee image; PC price not shown)""".strip()
    elif group:
        unit_price = rates[group]
        total = unit_price * sessions
        if not time_range and sessions == 1:
            answer = f"""{TH_ANSWER_LABEL}:
- ราคา: {unit_price:,} บาทต่อ {session_min} นาที ({service['label_th']})
- กลุ่มผู้ใช้: {CUSTOMER_GROUP_LABELS_TH[group]}

{TH_SOURCE_LABEL}:
- {SERVICE_FEE_SOURCE_URL}""".strip()
        else:
            answer = f"""{TH_ANSWER_LABEL}:
- ราคา: {total:,} บาท ({unit_price:,} บาท/session × {sessions} session(s))
- กลุ่มผู้ใช้: {CUSTOMER_GROUP_LABELS_TH[group]}
{chr(10).join('- ' + line for line in base_lines)}

{TH_SOURCE_LABEL}:
- {SERVICE_FEE_SOURCE_URL}""".strip()
    else:
        price_lines = []
        for key in ["psu", "alumni_student", "adult"]:
            unit_price = rates[key]
            total = unit_price * sessions
            price_lines.append(f"- {CUSTOMER_GROUP_LABELS_TH[key]}: {unit_price:,} บาท/session × {sessions} = {total:,} บาท")
        answer = f"""{TH_ANSWER_LABEL}:
- ยังไม่ทราบกลุ่มผู้ใช้ จึงแสดงราคาทุกกลุ่มให้เทียบก่อน
{chr(10).join(price_lines)}
{chr(10).join('- ' + line for line in base_lines)}

{TH_SOURCE_LABEL}:
- {SERVICE_FEE_SOURCE_URL}""".strip()

    hits = [{
        "id": f"calculator_{service_key}",
        "text": answer,
        "metadata": {
            "title": "reservation_price_calculator",
            "category": "service_fee",
            "source_url": SERVICE_FEE_SOURCE_URL if rates else SERVICE_PRESET_SOURCE_URL,
            "source_type": "calculator",
        },
        "distance": 0.0,
        "score": 999.0,
    }]
    return answer, hits

def llm_num_predict_for_question(question: str) -> int:
    q = normalize_thai_digits(question.lower())
    long_answer_words = [
        "สรุป", "ขั้นตอน", "รายละเอียด", "ทั้งหมด", "มีอะไรบ้าง", "อธิบาย", "เปรียบเทียบ",
        "summary", "summarize", "steps", "detail", "explain", "list", "compare",
    ]
    if any(word in q for word in long_answer_words):
        return max(LLM_NUM_PREDICT, 320)
    return LLM_NUM_PREDICT

def answer_question(question: str, top_k: int = TOP_K, use_rules: bool = True, use_direct: bool = True):
    start = time.time()

    calc_result = calculate_service_price_answer(question)
    if calc_result:
        answer, hits = calc_result
        elapsed = round(time.time() - start, 3)
        log_row = {
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "mode": "deterministic_calculator",
            "question": question,
            "answer": answer,
            "retrieved_ids": [h["id"] for h in hits],
            "sources": [h["metadata"] for h in hits],
            "model": "calculator",
            "latency_sec": elapsed,
        }
        save_chat_log(log_row)
        return answer, hits, elapsed

    if use_rules:
        rule_match = match_rule(question, RULES)
        if rule_match:
            answer = format_rule_answer(question, rule_match)
            hits = [rule_match_to_hit(rule_match)]
            elapsed = round(time.time() - start, 3)
            log_row = {
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "mode": "rule_fast_path",
                "question": question,
                "answer": answer,
                "rule_id": rule_match.get("rule_id"),
                "matched_pattern": rule_match.get("matched_pattern"),
                "retrieved_ids": [h["id"] for h in hits],
                "sources": [h["metadata"] for h in hits],
                "model": "rule_based",
                "latency_sec": elapsed,
            }
            save_chat_log(log_row)
            return answer, hits, elapsed

    hits = retrieve(question, top_k=top_k)
    direct_hits = force_direct_hits_for_question(question, hits) if use_direct else hits
    direct_answer = format_direct_rag_answer(question, direct_hits) if use_direct else None
    if direct_answer:
        hits = direct_hits
        elapsed = round(time.time() - start, 3)
        log_row = {
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "mode": "rag_direct_curated",
            "question": question,
            "answer": direct_answer,
            "retrieved_ids": [h["id"] for h in hits],
            "sources": [h["metadata"] for h in hits],
            "model": "direct_from_retrieved_curated_fact",
            "top_k": top_k,
            "latency_sec": elapsed,
        }
        save_chat_log(log_row)
        return direct_answer, hits, elapsed

    context = build_context(hits)
    user_prompt = f"""
CONTEXT:
{context}

QUESTION:
{question}
""".strip()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    llm_num_predict = llm_num_predict_for_question(question)
    answer = sanitize_model_answer(ask_ollama(messages, num_predict=llm_num_predict), question)
    elapsed = round(time.time() - start, 3)
    log_row = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "mode": "rag_llm",
        "question": question,
        "answer": answer,
        "retrieved_ids": [h["id"] for h in hits],
        "sources": [h["metadata"] for h in hits],
        "model": LLM_MODEL,
        "top_k": top_k,
        "num_ctx": LLM_NUM_CTX,
        "num_predict": llm_num_predict,
        "latency_sec": elapsed,
    }
    save_chat_log(log_row)
    return answer, hits, elapsed


loaded rule patterns: 77
active LLM model: qwen2.5:3b
fast config: {'top_k': 4, 'num_ctx': 2048, 'num_predict': 120, 'max_context_chars': 3200}


In [11]:
question = "PSU Esports Studio Phuket คืออะไร"
question = "Mission ของศูนย์คืออะไร"
question = "เช็คอินล่วงหน้าได้กี่นาที"
question = "ถ้าทำจอแตกต้องชดเชยยังไง"
question = "ติดต่อศูนย์ได้ทางไหน"
answer, hits, elapsed = answer_question(question)
print("latency_sec:", elapsed)
print(answer)

latency_sec: 0.11
คำตอบ:
- Facebook ของศูนย์คือ https://www.facebook.com/psuesportsphuket
- อีเมลติดต่อศูนย์คือ psuesportspkt@gmail.com
- PSU Esports Studio - Phuket ตั้งอยู่ที่มหาวิทยาลัยสงขลานครินทร์ วิทยาเขตภูเก็ต 80 หมู่ 1 ถ.วิชิตสงคราม อ.กะทู้ จ.ภูเก็ต 83120
- เบอร์ติดต่อที่ปรากฏในระบบจองคือ +66 7627 6004 และ +66 7627 6045

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/Contact-Us (curated_contact_facebook)
- https://esports.phuket.psu.ac.th/Contact-Us (curated_contact_email)
- https://esports.phuket.psu.ac.th/Contact-Us (curated_contact_location)
- https://esports.computing.psu.ac.th/ (curated_contact_phone)


In [12]:
question = "ศูนย์อีสปอร์ตห้ามสูบบุหรี่ไหม"
answer, hits, elapsed = answer_question(question)
print("latency_sec:", elapsed)
print(answer)

latency_sec: 0.062
คำตอบ:
ศูนย์ห้ามสูบบุหรี่ เสพสารเสพติด หรือดื่มเครื่องดื่มแอลกอฮอล์ภายในศูนย์

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_smoking_alcohol / Reservation, curated_rule_smoking_alcohol_drugs)


## 6. Ground truth evaluation เบื้องต้น

ขั้นนี้เป็น keyword check แบบง่ายก่อน ยังไม่ใช่ evaluation สมบูรณ์ แต่พอใช้วัด MVP ได้

In [13]:
def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

ground_truth = load_jsonl(GROUND_TRUTH_PATH)
pd.DataFrame(ground_truth).head()

,id,category,question,expected_keywords,expected_source_keywords,answer_type,difficulty
0,overview_001,overview,PSU Esports Studio Phuket คืออะไร,"[ศูนย์พัฒนาการเรียนรู้, อีสปอร์ต, ภูเก็ต]",[home],summary,easy
1,overview_002,overview,ศูนย์อีสปอร์ตนี้ก่อตั้งโดยหน่วยงานไหน,"[มหาวิทยาลัยสงขลานครินทร์, วิทยาลัยการคอมพิวเต...",[home],fact,easy
2,overview_003,overview,What is the mission of PSU Esports Studio Phuket?,"[esports education, College of Computing]",[home],summary,medium
3,overview_004,overview,ศูนย์นี้เน้นช่วยกลุ่มผู้ใช้แบบไหนบ้าง,"[นักเล่นเกม, นักศึกษา, ผู้สนใจ]",[home],summary,medium
4,equipment_001,equipment,ศูนย์มี Gaming PC กี่เครื่อง,"[MSI MAG Infinite S3 14th, 10 Units]",[home],fact,easy


In [14]:
def keyword_hit(answer: str, expected_keywords: list[str]) -> bool:
    if not expected_keywords:
        return True
    a = answer.lower()
    return all(k.lower() in a for k in expected_keywords)

def source_hit(hits: list[dict], expected_source_keywords: list[str]) -> bool:
    if not expected_source_keywords:
        return True
    text = " ".join([
        h["id"] + " " +
        str(h["metadata"].get("title", "")) + " " +
        str(h["metadata"].get("category", "")) + " " +
        str(h["metadata"].get("source_url", ""))
        for h in hits
    ]).lower()
    return any(k.lower() in text for k in expected_source_keywords)

def run_eval(limit: int | None = 5):
    rows = ground_truth[:limit] if limit else ground_truth
    results = []    
    for item in tqdm(rows):
        ans, hits, elapsed = answer_question(item["question"])
        kw_ok = keyword_hit(ans, item.get("expected_keywords", []))
        src_ok = source_hit(hits, item.get("expected_source_keywords", []))
        results.append({
            "id": item["id"],
            "category": item.get("category"),
            "question": item["question"],
            "keyword_ok": kw_ok,
            "source_ok": src_ok,
            "latency_sec": elapsed,
            "answer": ans,
            "retrieved_ids": [h["id"] for h in hits],
        })
    return pd.DataFrame(results)

# ระวัง: cell นี้เรียก LLM หลายครั้ง เริ่มจาก 3-5 ข้อก่อน
eval_df = run_eval(limit=3)
eval_df[["id", "keyword_ok", "source_ok", "latency_sec", "retrieved_ids"]]

100%|██████████| 3/3 [00:00<00:00, 12.41it/s]


,id,keyword_ok,source_ok,latency_sec,retrieved_ids
0,overview_001,True,True,0.075,[rule_overview_identity]
1,overview_002,True,True,0.054,[rule_overview_founder]
2,overview_003,True,True,0.110,[rule_overview_mission]


## 6.1 Ground Truth Verbose Check

Use this cell to inspect Ground Truth one item at a time: question, AI answer, expected answer/keywords, and pass/fail status.


In [15]:
from IPython.display import Markdown, display
from datetime import datetime
from pathlib import Path
import json
import pandas as pd

GROUND_TRUTH_V2_PATH = PROJECT_DIR / "ground_truth" / "ground_truth_v2_360.jsonl"
VERBOSE_GT_REPORT_DIR = PROJECT_DIR.parent / "16_PSU_Esports_RAG_Experiment_Timeline"


def _u(text: str) -> str:
    if "\\u" not in text:
        return text
    return text.encode("ascii").decode("unicode_escape")


TH_QUESTION = _u("\u0e04\u0e33\u0e16\u0e32\u0e21")
TH_AI_ANSWER = _u("\u0e04\u0e33\u0e15\u0e2d\u0e1a(\u0e08\u0e32\u0e01 AI)")
TH_EXPECTED = _u("\u0e40\u0e09\u0e25\u0e22/\u0e40\u0e01\u0e13\u0e11\u0e4c\u0e17\u0e35\u0e48\u0e16\u0e39\u0e01")
TH_CHECK_RESULT = _u("\u0e1c\u0e25\u0e15\u0e23\u0e27\u0e08")
TH_STATUS = _u("\u0e2a\u0e16\u0e32\u0e19\u0e30")
TH_PASS = _u("\u0e16\u0e39\u0e01")
TH_FAIL = _u("\u0e1c\u0e34\u0e14")
TH_EXPECTED_KEYWORDS = _u("\u0e15\u0e49\u0e2d\u0e07\u0e21\u0e35\u0e04\u0e33\u0e2a\u0e33\u0e04\u0e31\u0e0d")
TH_CATEGORY = _u("\u0e2b\u0e21\u0e27\u0e14")
TH_ANSWER_TYPE = _u("\u0e0a\u0e19\u0e34\u0e14\u0e04\u0e33\u0e15\u0e2d\u0e1a")
TH_NO_EXPECTED = _u("\u0e44\u0e21\u0e48\u0e21\u0e35\u0e40\u0e09\u0e25\u0e22/keyword \u0e43\u0e19\u0e44\u0e1f\u0e25\u0e4c Ground Truth")
TH_NO_SOURCES = _u("\u0e44\u0e21\u0e48\u0e21\u0e35 retrieved sources")
TH_FILE = _u("\u0e44\u0e1f\u0e25\u0e4c")
TH_RANGE = _u("\u0e02\u0e49\u0e2d\u0e17\u0e35\u0e48\u0e23\u0e31\u0e19")
TH_TOTAL = _u("\u0e23\u0e27\u0e21")


def _gt_load_jsonl(path: Path):
    path = Path(path)
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def _gt_read_last_log():
    path = Path(LOG_PATH)
    if not path.exists():
        return {}
    lines = [line for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not lines:
        return {}
    try:
        return json.loads(lines[-1])
    except json.JSONDecodeError:
        return {}


def _gt_keyword_check(answer: str, expected_keywords: list[str]):
    def norm(text):
        return str(text).lower().replace(",", "")
    answer_lower = norm(answer)
    missing = [kw for kw in expected_keywords if norm(kw) not in answer_lower]
    return len(missing) == 0, missing


def _gt_source_check(hits: list[dict], expected_source_keywords: list[str]):
    if not expected_source_keywords:
        return True, []
    source_aliases = []
    for h in hits:
        meta = h.get('metadata', {})
        source_url = str(meta.get('source_url', '')).lower()
        category = str(meta.get('category', '')).lower()
        source_id = str(h.get('id', '')).lower()
        if 'esports.computing.psu.ac.th' in source_url:
            source_aliases.append('Reservation')
        if 'service-fee' in source_url or 'service_fee' in category or 'service_fee' in source_id:
            source_aliases.append('service_fee')
            source_aliases.append('Service Fee')
        if '/home' in source_url:
            source_aliases.append('home')
        if '/knowledge' in source_url:
            source_aliases.append('Knowledge')
        if '/events-news/news' in source_url:
            source_aliases.append('News')
        if '/members' in source_url:
            source_aliases.append('Members')
    source_text = " ".join(
        f"{h.get('id', '')} "
        f"{h.get('metadata', {}).get('title', '')} "
        f"{h.get('metadata', {}).get('category', '')} "
        f"{h.get('metadata', {}).get('source_url', '')} "
        f"{h.get('metadata', {}).get('source_ids', '')}"
        for h in hits
    ) + " " + " ".join(source_aliases)
    source_text = source_text.lower()
    matched = [kw for kw in expected_source_keywords if str(kw).lower() in source_text]
    # Source check uses any-match because one correct answer may come from several valid chunks.
    return len(matched) > 0, matched


def _gt_expected_text(item: dict):
    if item.get("expected_answer"):
        return str(item["expected_answer"])

    lines = []
    if item.get("expected_keywords"):
        lines.append(TH_EXPECTED_KEYWORDS + ": " + ", ".join(map(str, item["expected_keywords"])))
    if item.get("expected_source_keywords"):
        lines.append("Expected source keywords: " + ", ".join(map(str, item["expected_source_keywords"])))
    if item.get("category"):
        lines.append(f"{TH_CATEGORY}: {item['category']}")
    if item.get("answer_type"):
        lines.append(f"{TH_ANSWER_TYPE}: {item['answer_type']}")
    return "\n".join(lines) if lines else TH_NO_EXPECTED


def _gt_clip(text: str, max_chars: int | None = 1600):
    text = str(text).strip()
    if max_chars is None or len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "\n... [trimmed output]"


def _gt_sources_text(hits: list[dict], max_sources: int = 4):
    if not hits:
        return TH_NO_SOURCES
    lines = []
    for i, hit in enumerate(hits[:max_sources], 1):
        meta = hit.get("metadata", {})
        lines.append(
            f"{i}. `{hit.get('id', '-')}` | category: `{meta.get('category', '-')}` | "
            f"title: {meta.get('title', '-')} | url: {meta.get('source_url', '-')}"
        )
    return "\n".join(lines)


def run_ground_truth_verbose(
    ground_truth_path: Path = GROUND_TRUTH_PATH,
    limit: int | None = 10,
    start: int = 0,
    use_rules: bool = True,
    use_direct: bool = True,
    show_sources: bool = False,
    max_answer_chars: int | None = 1600,
    save_report: bool = True,
    label: str = "manual_verbose",
):
    """
    Run Ground Truth in verbose mode.

    Examples:
    - run_ground_truth_verbose(limit=10)
    - run_ground_truth_verbose(limit=None)
    - run_ground_truth_verbose(GROUND_TRUTH_V2_PATH, limit=20, label="v2_first20")
    - run_ground_truth_verbose(GROUND_TRUTH_V2_PATH, limit=None, label="v2_360")
    """
    gt_path = Path(ground_truth_path)
    rows = _gt_load_jsonl(gt_path)
    selected = rows[start:] if limit is None else rows[start:start + limit]

    results = []
    blocks = []
    total = len(selected)

    for offset, item in enumerate(selected, 1):
        display_no = start + offset
        question = item["question"]

        try:
            answer, hits, elapsed = answer_question(
                question,
                use_rules=use_rules,
                use_direct=use_direct,
            )
            log_row = _gt_read_last_log()
            mode = log_row.get("mode", "unknown")
            model = log_row.get("model", "-")

            kw_ok, missing_keywords = _gt_keyword_check(answer, item.get("expected_keywords", []))
            src_ok, matched_sources = _gt_source_check(hits, item.get("expected_source_keywords", []))
            passed = kw_ok and src_ok
            status = TH_PASS if passed else TH_FAIL
            badge = "[PASS]" if passed else "[FAIL]"

            result = {
                "no": display_no,
                "id": item.get("id"),
                "category": item.get("category"),
                "question": question,
                "status": status,
                "pass": passed,
                "keyword_ok": kw_ok,
                "source_ok": src_ok,
                "missing_keywords": missing_keywords,
                "matched_sources": matched_sources,
                "mode": mode,
                "model": model,
                "latency_sec": elapsed,
                "answer": answer,
                "expected": _gt_expected_text(item),
                "retrieved_ids": [h.get("id") for h in hits],
            }

            check_lines = [
                f"- {TH_STATUS}: **{status}**",
                f"- keyword_ok: `{kw_ok}`" + (f" | missing: `{missing_keywords}`" if missing_keywords else ""),
                f"- source_ok: `{src_ok}`" + (f" | matched: `{matched_sources}`" if matched_sources else ""),
                f"- route: `{mode}` | model: `{model}` | elapsed: `{elapsed:.3f}` sec",
            ]
            if show_sources:
                check_lines.append("\n**Retrieved Sources:**\n" + _gt_sources_text(hits))

            block = f"""## {display_no}. {badge} {status}

**{TH_QUESTION}:** {question}

**{TH_AI_ANSWER}:**

{_gt_clip(answer, max_answer_chars)}

**{TH_EXPECTED}:**

{_gt_expected_text(item)}

**{TH_CHECK_RESULT}:**
{chr(10).join(check_lines)}
"""

        except Exception as exc:
            result = {
                "no": display_no,
                "id": item.get("id"),
                "category": item.get("category"),
                "question": question,
                "status": "error",
                "pass": False,
                "keyword_ok": False,
                "source_ok": False,
                "missing_keywords": item.get("expected_keywords", []),
                "matched_sources": [],
                "mode": "error",
                "model": "-",
                "latency_sec": None,
                "answer": repr(exc),
                "expected": _gt_expected_text(item),
                "retrieved_ids": [],
            }
            block = f"""## {display_no}. [ERROR]

**{TH_QUESTION}:** {question}

**{TH_AI_ANSWER}:**

`{repr(exc)}`

**{TH_EXPECTED}:**

{_gt_expected_text(item)}
"""

        results.append(result)
        blocks.append(block)
        print(f"[{offset}/{total}] item {display_no} | {result['status']} | {question}")

    df = pd.DataFrame(results)
    passed_count = int(df["pass"].sum()) if len(df) else 0
    failed_count = len(df) - passed_count
    avg_latency = df["latency_sec"].dropna().mean() if "latency_sec" in df else 0

    summary = f"""# Ground Truth Verbose Report

- {TH_FILE}: `{gt_path}`
- {TH_RANGE}: {start + 1} - {start + total}
- {TH_TOTAL}: {len(df)}
- {TH_PASS}: {passed_count}
- {TH_FAIL}/error: {failed_count}
- Accuracy: {(passed_count / len(df) * 100) if len(df) else 0:.2f}%
- Average latency: {avg_latency:.3f} sec
- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

---
"""
    report_md = summary + "\n\n---\n\n".join(blocks)
    display(Markdown(report_md))

    if save_report:
        VERBOSE_GT_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        safe_label = "".join(ch if ch.isalnum() or ch in {"_", "-"} else "_" for ch in label)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        md_path = VERBOSE_GT_REPORT_DIR / f"ground_truth_verbose_{safe_label}_{timestamp}.md"
        jsonl_path = VERBOSE_GT_REPORT_DIR / f"ground_truth_verbose_{safe_label}_{timestamp}.jsonl"
        md_path.write_text(report_md, encoding="utf-8")
        with jsonl_path.open("w", encoding="utf-8") as f:
            for row in results:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
        print("saved markdown:", md_path)
        print("saved jsonl:", jsonl_path)

    return df


# Quick test: first 5 items from the original 105-question Ground Truth.
# gt_verbose_df = run_ground_truth_verbose(limit=5)

# Ground Truth v2: first 20 items from the 360-question file.
# gt_verbose_v2_df = run_ground_truth_verbose(GROUND_TRUTH_V2_PATH, limit=20, label="v2_first20")

# Full original set: 105 items.
# gt_verbose_all_df = run_ground_truth_verbose(limit=None, label="full105")

# Full v2 set: 360 items. This can take a long time because some questions call the LLM.
# gt_verbose_v2_all_df = run_ground_truth_verbose(GROUND_TRUTH_V2_PATH, limit=None, label="v2_360")


In [16]:
gt_verbose_v2_df = run_ground_truth_verbose(
    GROUND_TRUTH_V2_PATH,
    limit=None,          # รันทั้งหมด 360 ข้อ
    label="v2_full",
    save_report=True     # เซฟเป็น .md และ .jsonl ให้อัตโนมัติ
)

[1/360] item 1 | ถูก | วันจันทร์เปิดให้เล่นกีโมง ปิดกี่โมง
[2/360] item 2 | ถูก | วันจันทร์เปิดให้เล่นกี่โมงถึงกี่โมง
[3/360] item 3 | ถูก | จันทร์เปิดปิดยังไง
[4/360] item 4 | ถูก | วันจันทร์เล่นได้ตั้งแต่กี่โมง
[5/360] item 5 | ถูก | วันจันทร์มีรอบเล่นช่วงไหนบ้าง
[6/360] item 6 | ถูก | Monday open close time?
[7/360] item 7 | ถูก | monday hours for play
[8/360] item 8 | ถูก | ถ้าไปวันจันทร์เช้าเล่นได้ไหม แล้วเปิดจริงกี่โมง
[9/360] item 9 | ถูก | วันจันทร์ morning เล่นได้ไหม afternoon เปิดไหม
[10/360] item 10 | ถูก | จันทร์เช้า maintenance แล้วบ่ายเปิดกี่โมง
[11/360] item 11 | ถูก | วันจันทร์ช่วงเช้าเปิดไหม
[12/360] item 12 | ถูก | จันทร์ 9 โมงเล่นได้ไหม
[13/360] item 13 | ถูก | Monday morning available?
[14/360] item 14 | ถูก | วันจันทร์ 09:00-12:00 เป็นอะไร
[15/360] item 15 | ถูก | เช้าวันจันทร์เป็น maintenance ใช่ไหม
[16/360] item 16 | ถูก | ศูนย์เปิดกีโมงปิดกี่โมง
[17/360] item 17 | ถูก | เปิดปิดกี่โมงนะ
[18/360] item 18 | ถูก | เวลาเปิดให้บริการคืออะไร
[19/360] item 19 | ถูก | se

# Ground Truth Verbose Report

- ไฟล์: `c:\Users\Chokhun\Downloads\Learn-LLM\15_PSU_Esports_Local_RAG_Qwen3_4B\ground_truth\ground_truth_v2_360.jsonl`
- ข้อที่รัน: 1 - 360
- รวม: 360
- ถูก: 360
- ผิด/error: 0
- Accuracy: 100.00%
- Average latency: 0.215 sec
- Generated: 2026-07-01 23:39:59

---
## 1. [PASS] ถูก

**คำถาม:** วันจันทร์เปิดให้เล่นกีโมง ปิดกี่โมง

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* จึงยังไม่ใช่ช่วงเล่น/เปิดให้บริการ ส่วนช่วง Afternoon 13:00–16:00 เป็น Open for Service ดังนั้นวันจันทร์เปิดให้เล่นช่วง 13:00 น. และปิด 16:00 น.

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_open_close / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.071` sec


---

## 2. [PASS] ถูก

**คำถาม:** วันจันทร์เปิดให้เล่นกี่โมงถึงกี่โมง

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* จึงยังไม่ใช่ช่วงเล่น/เปิดให้บริการ ส่วนช่วง Afternoon 13:00–16:00 เป็น Open for Service ดังนั้นวันจันทร์เปิดให้เล่นช่วง 13:00 น. และปิด 16:00 น.

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_open_close / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.057` sec


---

## 3. [PASS] ถูก

**คำถาม:** จันทร์เปิดปิดยังไง

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* จึงยังไม่ใช่ช่วงเล่น/เปิดให้บริการ ส่วนช่วง Afternoon 13:00–16:00 เป็น Open for Service ดังนั้นวันจันทร์เปิดให้เล่นช่วง 13:00 น. และปิด 16:00 น.

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_open_close / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.023` sec


---

## 4. [PASS] ถูก

**คำถาม:** วันจันทร์เล่นได้ตั้งแต่กี่โมง

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* ยังไม่ใช่ช่วงเล่น/เปิดให้บริการ ส่วนช่วง Afternoon 13:00–16:00 เป็น Open for Service จึงเล่นได้ช่วง 13:00–16:00

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_play_window / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.048` sec


---

## 5. [PASS] ถูก

**คำถาม:** วันจันทร์มีรอบเล่นช่วงไหนบ้าง

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* ยังไม่ใช่ช่วงเล่น/เปิดให้บริการ ส่วนช่วง Afternoon 13:00–16:00 เป็น Open for Service จึงเล่นได้ช่วง 13:00–16:00

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_play_window / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.054` sec


---

## 6. [PASS] ถูก

**คำถาม:** Monday open close time?

**คำตอบ(จาก AI):**

Answer:
On Monday, Morning 09:00-12:00 is Maintenance*, not an open service slot. Afternoon 13:00-16:00 is Open for Service, so Monday play/service hours are 13:00-16:00.

Sources:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_open_close / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.042` sec


---

## 7. [PASS] ถูก

**คำถาม:** monday hours for play

**คำตอบ(จาก AI):**

Answer:
On Monday, Morning 09:00-12:00 is Maintenance*, not an open service slot. Afternoon 13:00-16:00 is Open for Service, so Monday play/service hours are 13:00-16:00.

Sources:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_open_close / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.020` sec


---

## 8. [PASS] ถูก

**คำถาม:** ถ้าไปวันจันทร์เช้าเล่นได้ไหม แล้วเปิดจริงกี่โมง

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* จึงยังไม่ใช่ช่วงเล่น/เปิดให้บริการ ส่วนช่วง Afternoon 13:00–16:00 เป็น Open for Service ดังนั้นวันจันทร์เปิดให้เล่นช่วง 13:00 น. และปิด 16:00 น.

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_open_close / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.082` sec


---

## 9. [PASS] ถูก

**คำถาม:** วันจันทร์ morning เล่นได้ไหม afternoon เปิดไหม

**คำตอบ(จาก AI):**

คำตอบ:
- ไม่ได้เปิด 24 ชั่วโมง ตารางบริการแบ่งเป็น Morning 09:00–12:00 และ Afternoon 13:00–16:00 โดยดูตามวัน: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* เล่นไม่ได้ และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดตามรอบปกติ 09:00–12:00 และ 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น
- ตารางบริการช่วง Morning คือ 09:00 – 12:00
- ตารางบริการช่วง Afternoon คือ 13:00 – 16:00

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_schedule_morning)
- https://esports.computing.psu.ac.th/ (curated_schedule_afternoon)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.184` sec


---

## 10. [PASS] ถูก

**คำถาม:** จันทร์เช้า maintenance แล้วบ่ายเปิดกี่โมง

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* จึงยังไม่ใช่ช่วงเล่น/เปิดให้บริการ ส่วนช่วง Afternoon 13:00–16:00 เป็น Open for Service ดังนั้นวันจันทร์เปิดให้เล่นช่วง 13:00 น. และปิด 16:00 น.

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_open_close / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.116` sec


---

## 11. [PASS] ถูก

**คำถาม:** วันจันทร์ช่วงเช้าเปิดไหม

**คำตอบ(จาก AI):**

คำตอบ:
วัน Monday ช่วง Morning 09:00–12:00 เป็น Maintenance* ไม่ใช่ช่วง Open for Service

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_morning / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 12:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.039` sec


---

## 12. [PASS] ถูก

**คำถาม:** จันทร์ 9 โมงเล่นได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* ไม่ใช่ช่วง Open for Service ดังนั้นช่วง 9 โมงเช้าวันจันทร์ยังเล่นไม่ได้

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_morning_specific / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 12:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.023` sec


---

## 13. [PASS] ถูก

**คำถาม:** Monday morning available?

**คำตอบ(จาก AI):**

Answer:
On Monday morning, 09:00-12:00 is Maintenance*, not Open for Service.

Sources:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_morning / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 12:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.043` sec


---

## 14. [PASS] ถูก

**คำถาม:** วันจันทร์ 09:00-12:00 เป็นอะไร

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* ไม่ใช่ช่วง Open for Service ดังนั้นช่วง 9 โมงเช้าวันจันทร์ยังเล่นไม่ได้

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_morning_specific / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 12:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.040` sec


---

## 15. [PASS] ถูก

**คำถาม:** เช้าวันจันทร์เป็น maintenance ใช่ไหม

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* ไม่ใช่ช่วง Open for Service ดังนั้นช่วง 9 โมงเช้าวันจันทร์ยังเล่นไม่ได้

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_morning_specific / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 12:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.071` sec


---

## 16. [PASS] ถูก

**คำถาม:** ศูนย์เปิดกีโมงปิดกี่โมง

**คำตอบ(จาก AI):**

คำตอบ:
ไม่ได้เปิด 24 ชั่วโมง ตารางบริการแบ่งเป็น Morning 09:00–12:00 และ Afternoon 13:00–16:00 โดยดูตามวันดังนี้: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* เล่นไม่ได้ และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดตามรอบปกติ 09:00–12:00 และ 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_service_schedule / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 16:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.038` sec


---

## 17. [PASS] ถูก

**คำถาม:** เปิดปิดกี่โมงนะ

**คำตอบ(จาก AI):**

คำตอบ:
ไม่ได้เปิด 24 ชั่วโมง ตารางบริการแบ่งเป็น Morning 09:00–12:00 และ Afternoon 13:00–16:00 โดยดูตามวันดังนี้: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* เล่นไม่ได้ และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดตามรอบปกติ 09:00–12:00 และ 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_service_schedule / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 16:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.025` sec


---

## 18. [PASS] ถูก

**คำถาม:** เวลาเปิดให้บริการคืออะไร

**คำตอบ(จาก AI):**

คำตอบ:
ไม่ได้เปิด 24 ชั่วโมง ตารางบริการแบ่งเป็น Morning 09:00–12:00 และ Afternoon 13:00–16:00 โดยดูตามวันดังนี้: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* เล่นไม่ได้ และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดตามรอบปกติ 09:00–12:00 และ 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_service_schedule / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 16:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.040` sec


---

## 19. [PASS] ถูก

**คำถาม:** service hours คืออะไร

**คำตอบ(จาก AI):**

คำตอบ:
ไม่ได้เปิด 24 ชั่วโมง ตารางบริการแบ่งเป็น Morning 09:00–12:00 และ Afternoon 13:00–16:00 โดยดูตามวันดังนี้: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* เล่นไม่ได้ และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดตามรอบปกติ 09:00–12:00 และ 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_service_schedule / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 16:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.021` sec


---

## 20. [PASS] ถูก

**คำถาม:** เปิดถึง 16:00 ใช่ไหม

**คำตอบ(จาก AI):**

คำตอบ:
- ไม่ได้เปิด 24 ชั่วโมง ตารางบริการแบ่งเป็น Morning 09:00–12:00 และ Afternoon 13:00–16:00 โดยดูตามวัน: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* เล่นไม่ได้ และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดตามรอบปกติ 09:00–12:00 และ 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น
- ตารางบริการช่วง Morning คือ 09:00 – 12:00
- ตารางบริการช่วง Afternoon คือ 13:00 – 16:00

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_schedule_morning)
- https://esports.computing.psu.ac.th/ (curated_schedule_afternoon)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 16:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.093` sec


---

## 21. [PASS] ถูก

**คำถาม:** เปิด 24 ชม ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ไม่ได้เปิด 24 ชั่วโมง ให้ยึดตาม timeline การให้บริการ: วันจันทร์ (Monday) เช้า 09:00–12:00 เป็น Maintenance* และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิด 09:00–12:00 และ 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_not_24_hours / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 16:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.015` sec


---

## 22. [PASS] ถูก

**คำถาม:** ช่วงเช้าของตารางคือเวลาไหน

**คำตอบ(จาก AI):**

คำตอบ:
ใช่ รอบเช้า Morning คือ 09:00–12:00 แต่ถ้าไม่ได้ระบุวันควรดู timeline ทั้งสัปดาห์ด้วย: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* เล่นไม่ได้ และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดตามรอบปกติ 09:00–12:00 และ 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_morning_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 12:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.067` sec


---

## 23. [PASS] ถูก

**คำถาม:** Morning คือกี่โมงถึงกี่โมง

**คำตอบ(จาก AI):**

คำตอบ:
ใช่ รอบเช้า Morning คือ 09:00–12:00 แต่ถ้าไม่ได้ระบุวันควรดู timeline ทั้งสัปดาห์ด้วย: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* เล่นไม่ได้ และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดตามรอบปกติ 09:00–12:00 และ 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_morning_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 12:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.040` sec


---

## 24. [PASS] ถูก

**คำถาม:** รอบเช้า 09 ถึง 12 ใช่ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ใช่ รอบเช้า Morning คือ 09:00–12:00 แต่ถ้าไม่ได้ระบุวันควรดู timeline ทั้งสัปดาห์ด้วย: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* เล่นไม่ได้ และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดตามรอบปกติ 09:00–12:00 และ 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_morning_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 12:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.020` sec


---

## 25. [PASS] ถูก

**คำถาม:** ช่วงเช้าเปิดตั้งแต่กี่โมงถึงเที่ยงไหม

**คำตอบ(จาก AI):**

คำตอบ:
ใช่ รอบเช้า Morning คือ 09:00–12:00 แต่ถ้าไม่ได้ระบุวันควรดู timeline ทั้งสัปดาห์ด้วย: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* เล่นไม่ได้ และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดตามรอบปกติ 09:00–12:00 และ 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_morning_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 12:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.053` sec


---

## 26. [PASS] ถูก

**คำถาม:** ตอนเช้าเปิดกี่โมง

**คำตอบ(จาก AI):**

คำตอบ:
ใช่ รอบเช้า Morning คือ 09:00–12:00 แต่ถ้าไม่ได้ระบุวันควรดู timeline ทั้งสัปดาห์ด้วย: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* เล่นไม่ได้ และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดตามรอบปกติ 09:00–12:00 และ 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_morning_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 12:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.024` sec


---

## 27. [PASS] ถูก

**คำถาม:** รอบเช้าเริ่มตอนไหน

**คำตอบ(จาก AI):**

คำตอบ:
ใช่ รอบเช้า Morning คือ 09:00–12:00 แต่ถ้าไม่ได้ระบุวันควรดู timeline ทั้งสัปดาห์ด้วย: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* เล่นไม่ได้ และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดตามรอบปกติ 09:00–12:00 และ 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_morning_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 12:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.013` sec


---

## 28. [PASS] ถูก

**คำถาม:** morning session time?

**คำตอบ(จาก AI):**

Answer:
Yes, the Morning slot is 09:00-12:00, but if no day is specified you should check the weekly timeline: Monday morning 09:00-12:00 is Maintenance* and not playable, while Monday afternoon 13:00-16:00 is open; Tuesday-Thursday follow the regular slots 09:00-12:00 and 13:00-16:00; Friday morning 09:00-12:00 is open, but Friday afternoon 13:00-16:00 is Maintenance** for weekly hardware inspection and cleaning, so it is not playable.

Sources:
- https://esports.computing.psu.ac.th/ (rule_schedule_morning_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 12:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.042` sec


---

## 29. [PASS] ถูก

**คำถาม:** morning slot time

**คำตอบ(จาก AI):**

Answer:
Yes, the Morning slot is 09:00-12:00, but if no day is specified you should check the weekly timeline: Monday morning 09:00-12:00 is Maintenance* and not playable, while Monday afternoon 13:00-16:00 is open; Tuesday-Thursday follow the regular slots 09:00-12:00 and 13:00-16:00; Friday morning 09:00-12:00 is open, but Friday afternoon 13:00-16:00 is Maintenance** for weekly hardware inspection and cleaning, so it is not playable.

Sources:
- https://esports.computing.psu.ac.th/ (rule_schedule_morning_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 09:00, 12:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.016` sec


---

## 30. [PASS] ถูก

**คำถาม:** ช่วงบ่ายของตารางคือเวลาไหน

**คำตอบ(จาก AI):**

คำตอบ:
รอบบ่าย Afternoon คือ 13:00–16:00 แต่ถ้าไม่ได้ระบุวันต้องดู timeline ด้วย: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดทั้งรอบเช้า 09:00–12:00 และรอบบ่าย 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_afternoon_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.034` sec


---

## 31. [PASS] ถูก

**คำถาม:** Afternoon คือกี่โมงถึงกี่โมง

**คำตอบ(จาก AI):**

คำตอบ:
รอบบ่าย Afternoon คือ 13:00–16:00 แต่ถ้าไม่ได้ระบุวันต้องดู timeline ด้วย: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดทั้งรอบเช้า 09:00–12:00 และรอบบ่าย 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_afternoon_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.051` sec


---

## 32. [PASS] ถูก

**คำถาม:** รอบบ่าย 13 ถึง 16 ใช่ไหม

**คำตอบ(จาก AI):**

คำตอบ:
รอบบ่าย Afternoon คือ 13:00–16:00 แต่ถ้าไม่ได้ระบุวันต้องดู timeline ด้วย: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดทั้งรอบเช้า 09:00–12:00 และรอบบ่าย 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_afternoon_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.028` sec


---

## 33. [PASS] ถูก

**คำถาม:** ช่วงบ่ายเปิดกี่โมงปิดกี่โมง

**คำตอบ(จาก AI):**

คำตอบ:
รอบบ่าย Afternoon คือ 13:00–16:00 แต่ถ้าไม่ได้ระบุวันต้องดู timeline ด้วย: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดทั้งรอบเช้า 09:00–12:00 และรอบบ่าย 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_afternoon_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.043` sec


---

## 34. [PASS] ถูก

**คำถาม:** รอบบ่ายปิดกี่โมง

**คำตอบ(จาก AI):**

คำตอบ:
รอบบ่าย Afternoon คือ 13:00–16:00 แต่ถ้าไม่ได้ระบุวันต้องดู timeline ด้วย: วันจันทร์ (Monday) ช่วงเช้า 09:00–12:00 เป็น Maintenance* และเปิดเล่นช่วงบ่าย 13:00–16:00; วันอังคาร-พฤหัสบดี (Tuesday-Thursday) เปิดทั้งรอบเช้า 09:00–12:00 และรอบบ่าย 13:00–16:00; วันศุกร์ (Friday) เปิดช่วงเช้า 09:00–12:00 แต่ช่วงบ่าย 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_afternoon_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.017` sec


---

## 35. [PASS] ถูก

**คำถาม:** afternoon session time?

**คำตอบ(จาก AI):**

Answer:
The Afternoon slot is 13:00-16:00, but if no day is specified you should check the weekly timeline: Monday morning 09:00-12:00 is Maintenance* and Monday afternoon 13:00-16:00 is open; Tuesday-Thursday are open in both slots 09:00-12:00 and 13:00-16:00; Friday morning 09:00-12:00 is open, but Friday afternoon 13:00-16:00 is Maintenance** for weekly hardware inspection and cleaning, so it is not playable.

Sources:
- https://esports.computing.psu.ac.th/ (rule_schedule_afternoon_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.045` sec


---

## 36. [PASS] ถูก

**คำถาม:** afternoon slot time

**คำตอบ(จาก AI):**

Answer:
The Afternoon slot is 13:00-16:00, but if no day is specified you should check the weekly timeline: Monday morning 09:00-12:00 is Maintenance* and Monday afternoon 13:00-16:00 is open; Tuesday-Thursday are open in both slots 09:00-12:00 and 13:00-16:00; Friday morning 09:00-12:00 is open, but Friday afternoon 13:00-16:00 is Maintenance** for weekly hardware inspection and cleaning, so it is not playable.

Sources:
- https://esports.computing.psu.ac.th/ (rule_schedule_afternoon_time / Reservation, curated_schedule_morning, curated_schedule_afternoon, curated_reservation_schedule_monday_morning, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Monday, Friday, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.034` sec


---

## 37. [PASS] ถูก

**คำถาม:** วันศุกร์ maintenance คืออะไร

**คำตอบ(จาก AI):**

คำตอบ:
วันศุกร์ช่วง Morning 09:00–12:00 เปิดตามรอบเช้าได้ แต่ช่วง Afternoon 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น/ให้บริการ

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_friday_maintenance / Reservation, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance, Weekly hardware inspection, cleaning
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.078` sec


---

## 38. [PASS] ถูก

**คำถาม:** Friday maintenance ทำอะไร

**คำตอบ(จาก AI):**

คำตอบ:
วันศุกร์ช่วง Morning 09:00–12:00 เปิดตามรอบเช้าได้ แต่ช่วง Afternoon 13:00–16:00 เป็น Maintenance** สำหรับ Weekly hardware inspection and cleaning จึงไม่ใช่ช่วงเล่น/ให้บริการ

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_friday_maintenance / Reservation, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance, Weekly hardware inspection, cleaning
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.062` sec


---

## 39. [PASS] ถูก

**คำถาม:** maintenance weekly hardware inspection คืออะไร

**คำตอบ(จาก AI):**

คำตอบ:
วันศุกร์ช่วง Afternoon 13:00–16:00 เป็น Maintenance** หมายถึง Weekly hardware inspection and cleaning หรือการตรวจสอบฮาร์ดแวร์และทำความสะอาดรายสัปดาห์ ส่วนช่วง Morning 09:00–12:00 ยังเป็นรอบเช้า

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_friday_maintenance_general / Reservation, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance, Weekly hardware inspection, cleaning
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.123` sec


---

## 40. [PASS] ถูก

**คำถาม:** วันศุกร์มีตรวจอุปกรณ์ไหม

**คำตอบ(จาก AI):**

คำตอบ:
วันศุกร์ช่วง Afternoon 13:00–16:00 เป็น Maintenance** หมายถึง Weekly hardware inspection and cleaning หรือการตรวจสอบฮาร์ดแวร์และทำความสะอาดรายสัปดาห์ ส่วนช่วง Morning 09:00–12:00 ยังเป็นรอบเช้า

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_friday_maintenance_general / Reservation, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance, Weekly hardware inspection, cleaning
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.040` sec


---

## 41. [PASS] ถูก

**คำถาม:** ศุกร์ทำความสะอาดอุปกรณ์ใช่ไหม

**คำตอบ(จาก AI):**

คำตอบ:
วันศุกร์ช่วง Afternoon 13:00–16:00 เป็น Maintenance** หมายถึง Weekly hardware inspection and cleaning หรือการตรวจสอบฮาร์ดแวร์และทำความสะอาดรายสัปดาห์ ส่วนช่วง Morning 09:00–12:00 ยังเป็นรอบเช้า

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_friday_maintenance_general / Reservation, curated_reservation_schedule_friday_maintenance, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance, Weekly hardware inspection, cleaning
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.052` sec


---

## 42. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษา มอ จอง PS5 ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.132` sec


---

## 43. [PASS] ถูก

**คำถาม:** PlayStation 5 สำหรับนักศึกษา มอ ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.156` sec


---

## 44. [PASS] ถูก

**คำถาม:** PS5 สำหรับนักเรียน ม.อ. ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.106` sec


---

## 45. [PASS] ถูก

**คำถาม:** นักเรียน ม.อ. เล่น PlayStation 5 กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.127` sec


---

## 46. [PASS] ถูก

**คำถาม:** PS5 สำหรับเด็ก PSU ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.113` sec


---

## 47. [PASS] ถูก

**คำถาม:** เด็ก PSU เล่น PlayStation 5 กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.116` sec


---

## 48. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษาทั่วไป จอง PS5 ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 50 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 50
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.049` sec


---

## 49. [PASS] ถูก

**คำถาม:** PlayStation 5 สำหรับนักศึกษาทั่วไป ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 50 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 50
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.053` sec


---

## 50. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษาต่างมหาลัย จอง PS5 ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 50 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 50
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.067` sec


---

## 51. [PASS] ถูก

**คำถาม:** PlayStation 5 สำหรับนักศึกษาต่างมหาลัย ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 50 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 50
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.075` sec


---

## 52. [PASS] ถูก

**คำถาม:** ถ้าเป็นศิษย์เก่า PSU จอง PS5 ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 50 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 50
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.043` sec


---

## 53. [PASS] ถูก

**คำถาม:** PlayStation 5 สำหรับศิษย์เก่า PSU ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 50 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 50
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.048` sec


---

## 54. [PASS] ถูก

**คำถาม:** ถ้าเป็นบุคคลทั่วไป จอง PS5 ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 150 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 150
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.119` sec


---

## 55. [PASS] ถูก

**คำถาม:** PlayStation 5 สำหรับบุคคลทั่วไป ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 150 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 150
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.159` sec


---

## 56. [PASS] ถูก

**คำถาม:** คนนอก เล่น PS5 กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 150 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 150
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.057` sec


---

## 57. [PASS] ถูก

**คำถาม:** ถ้าเป็นคนนอก จอง PlayStation 5 ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 150 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 150
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.132` sec


---

## 58. [PASS] ถูก

**คำถาม:** ถ้าเป็นGeneral Adult จอง PS5 ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 150 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 150
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.145` sec


---

## 59. [PASS] ถูก

**คำถาม:** PlayStation 5 สำหรับGeneral Adult ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 150 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, 60, 150
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.169` sec


---

## 60. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษา มอ จอง Nintendo 1-2 คน ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (Nintendo Switch 1-2 คน)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.215` sec


---

## 61. [PASS] ถูก

**คำถาม:** Switch 1-2 สำหรับนักศึกษา มอ ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาท (0 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU
- ช่วงเวลาที่ถามคือ 01:00-02:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 1-2 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.152` sec


---

## 62. [PASS] ถูก

**คำถาม:** Nintendo 1-2 คน สำหรับนักเรียน ม.อ. ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (Nintendo Switch 1-2 คน)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.166` sec


---

## 63. [PASS] ถูก

**คำถาม:** นักเรียน ม.อ. เล่น Switch 1-2 กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาท (0 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU
- ช่วงเวลาที่ถามคือ 01:00-02:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 1-2 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.122` sec


---

## 64. [PASS] ถูก

**คำถาม:** Nintendo 1-2 คน สำหรับเด็ก PSU ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (Nintendo Switch 1-2 คน)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.172` sec


---

## 65. [PASS] ถูก

**คำถาม:** เด็ก PSU เล่น Switch 1-2 กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาท (0 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU
- ช่วงเวลาที่ถามคือ 01:00-02:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 1-2 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.141` sec


---

## 66. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษาทั่วไป จอง Nintendo 1-2 คน ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 50 บาทต่อ 60 นาที (Nintendo Switch 1-2 คน)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 50
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.084` sec


---

## 67. [PASS] ถูก

**คำถาม:** Switch 1-2 สำหรับนักศึกษาทั่วไป ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 50 บาท (50 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)
- ช่วงเวลาที่ถามคือ 01:00-02:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 1-2 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 50
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.055` sec


---

## 68. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษาต่างมหาลัย จอง Nintendo 1-2 คน ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 50 บาทต่อ 60 นาที (Nintendo Switch 1-2 คน)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 50
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.108` sec


---

## 69. [PASS] ถูก

**คำถาม:** Switch 1-2 สำหรับนักศึกษาต่างมหาลัย ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 50 บาท (50 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)
- ช่วงเวลาที่ถามคือ 01:00-02:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 1-2 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 50
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.071` sec


---

## 70. [PASS] ถูก

**คำถาม:** ถ้าเป็นศิษย์เก่า PSU จอง Nintendo 1-2 คน ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 50 บาทต่อ 60 นาที (Nintendo Switch 1-2 คน)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 50
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.096` sec


---

## 71. [PASS] ถูก

**คำถาม:** Switch 1-2 สำหรับศิษย์เก่า PSU ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 50 บาท (50 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)
- ช่วงเวลาที่ถามคือ 01:00-02:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 1-2 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 50
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.069` sec


---

## 72. [PASS] ถูก

**คำถาม:** ถ้าเป็นบุคคลทั่วไป จอง Nintendo 1-2 คน ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 140 บาทต่อ 60 นาที (Nintendo Switch 1-2 คน)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 140
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.162` sec


---

## 73. [PASS] ถูก

**คำถาม:** Switch 1-2 สำหรับบุคคลทั่วไป ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 140 บาท (140 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: บุคคลทั่วไป
- ช่วงเวลาที่ถามคือ 01:00-02:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 1-2 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 140
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.153` sec


---

## 74. [PASS] ถูก

**คำถาม:** คนนอก เล่น Nintendo 1-2 คน กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 140 บาทต่อ 60 นาที (Nintendo Switch 1-2 คน)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 140
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.110` sec


---

## 75. [PASS] ถูก

**คำถาม:** ถ้าเป็นคนนอก จอง Switch 1-2 ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 140 บาท (140 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: บุคคลทั่วไป
- ช่วงเวลาที่ถามคือ 01:00-02:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 1-2 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 140
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.150` sec


---

## 76. [PASS] ถูก

**คำถาม:** ถ้าเป็นGeneral Adult จอง Nintendo 1-2 คน ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 140 บาทต่อ 60 นาที (Nintendo Switch 1-2 คน)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 140
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.210` sec


---

## 77. [PASS] ถูก

**คำถาม:** Switch 1-2 สำหรับGeneral Adult ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 140 บาท (140 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: บุคคลทั่วไป
- ช่วงเวลาที่ถามคือ 01:00-02:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 1-2 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 1-2, 140
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.177` sec


---

## 78. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษา มอ จอง Nintendo 3-4 คน ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (Nintendo Switch 3-4 คน)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.175` sec


---

## 79. [PASS] ถูก

**คำถาม:** Switch 3-4 สำหรับนักศึกษา มอ ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาท (0 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU
- ช่วงเวลาที่ถามคือ 03:00-04:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 3-4 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.136` sec


---

## 80. [PASS] ถูก

**คำถาม:** Nintendo 3-4 คน สำหรับนักเรียน ม.อ. ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (Nintendo Switch 3-4 คน)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.161` sec


---

## 81. [PASS] ถูก

**คำถาม:** นักเรียน ม.อ. เล่น Switch 3-4 กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาท (0 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU
- ช่วงเวลาที่ถามคือ 03:00-04:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 3-4 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.136` sec


---

## 82. [PASS] ถูก

**คำถาม:** Nintendo 3-4 คน สำหรับเด็ก PSU ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (Nintendo Switch 3-4 คน)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.161` sec


---

## 83. [PASS] ถูก

**คำถาม:** เด็ก PSU เล่น Switch 3-4 กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาท (0 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU
- ช่วงเวลาที่ถามคือ 03:00-04:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 3-4 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.120` sec


---

## 84. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษาทั่วไป จอง Nintendo 3-4 คน ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 100 บาทต่อ 60 นาที (Nintendo Switch 3-4 คน)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 100
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.078` sec


---

## 85. [PASS] ถูก

**คำถาม:** Switch 3-4 สำหรับนักศึกษาทั่วไป ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 100 บาท (100 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)
- ช่วงเวลาที่ถามคือ 03:00-04:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 3-4 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 100
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.059` sec


---

## 86. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษาต่างมหาลัย จอง Nintendo 3-4 คน ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 100 บาทต่อ 60 นาที (Nintendo Switch 3-4 คน)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 100
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.133` sec


---

## 87. [PASS] ถูก

**คำถาม:** Switch 3-4 สำหรับนักศึกษาต่างมหาลัย ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 100 บาท (100 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)
- ช่วงเวลาที่ถามคือ 03:00-04:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 3-4 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 100
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.088` sec


---

## 88. [PASS] ถูก

**คำถาม:** ถ้าเป็นศิษย์เก่า PSU จอง Nintendo 3-4 คน ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 100 บาทต่อ 60 นาที (Nintendo Switch 3-4 คน)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 100
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.077` sec


---

## 89. [PASS] ถูก

**คำถาม:** Switch 3-4 สำหรับศิษย์เก่า PSU ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 100 บาท (100 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)
- ช่วงเวลาที่ถามคือ 03:00-04:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 3-4 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 100
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.060` sec


---

## 90. [PASS] ถูก

**คำถาม:** ถ้าเป็นบุคคลทั่วไป จอง Nintendo 3-4 คน ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 280 บาทต่อ 60 นาที (Nintendo Switch 3-4 คน)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 280
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.187` sec


---

## 91. [PASS] ถูก

**คำถาม:** Switch 3-4 สำหรับบุคคลทั่วไป ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 280 บาท (280 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: บุคคลทั่วไป
- ช่วงเวลาที่ถามคือ 03:00-04:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 3-4 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 280
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.147` sec


---

## 92. [PASS] ถูก

**คำถาม:** คนนอก เล่น Nintendo 3-4 คน กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 280 บาทต่อ 60 นาที (Nintendo Switch 3-4 คน)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 280
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.127` sec


---

## 93. [PASS] ถูก

**คำถาม:** ถ้าเป็นคนนอก จอง Switch 3-4 ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 280 บาท (280 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: บุคคลทั่วไป
- ช่วงเวลาที่ถามคือ 03:00-04:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 3-4 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 280
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.142` sec


---

## 94. [PASS] ถูก

**คำถาม:** ถ้าเป็นGeneral Adult จอง Nintendo 3-4 คน ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 280 บาทต่อ 60 นาที (Nintendo Switch 3-4 คน)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 280
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.194` sec


---

## 95. [PASS] ถูก

**คำถาม:** Switch 3-4 สำหรับGeneral Adult ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 280 บาท (280 บาท/session × 1 session(s))
- กลุ่มผู้ใช้: บุคคลทั่วไป
- ช่วงเวลาที่ถามคือ 03:00-04:00 = 1 ชั่วโมง
- บริการ Nintendo Switch 3-4 คน คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, 3-4, 280
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.158` sec


---

## 96. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษา มอ จอง Cockpit ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.194` sec


---

## 97. [PASS] ถูก

**คำถาม:** พวงมาลัยขับรถ สำหรับนักศึกษา มอ ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.182` sec


---

## 98. [PASS] ถูก

**คำถาม:** Cockpit สำหรับนักเรียน ม.อ. ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.170` sec


---

## 99. [PASS] ถูก

**คำถาม:** นักเรียน ม.อ. เล่น พวงมาลัยขับรถ กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.170` sec


---

## 100. [PASS] ถูก

**คำถาม:** Cockpit สำหรับเด็ก PSU ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.142` sec


---

## 101. [PASS] ถูก

**คำถาม:** เด็ก PSU เล่น พวงมาลัยขับรถ กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.156` sec


---

## 102. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษาทั่วไป จอง Cockpit ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 65 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 65
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.092` sec


---

## 103. [PASS] ถูก

**คำถาม:** พวงมาลัยขับรถ สำหรับนักศึกษาทั่วไป ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 65 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 65
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.108` sec


---

## 104. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษาต่างมหาลัย จอง Cockpit ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 65 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 65
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.107` sec


---

## 105. [PASS] ถูก

**คำถาม:** พวงมาลัยขับรถ สำหรับนักศึกษาต่างมหาลัย ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 65 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 65
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.108` sec


---

## 106. [PASS] ถูก

**คำถาม:** ถ้าเป็นศิษย์เก่า PSU จอง Cockpit ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 65 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 65
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.092` sec


---

## 107. [PASS] ถูก

**คำถาม:** พวงมาลัยขับรถ สำหรับศิษย์เก่า PSU ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 65 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 65
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.097` sec


---

## 108. [PASS] ถูก

**คำถาม:** ถ้าเป็นบุคคลทั่วไป จอง Cockpit ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 200 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 200
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.188` sec


---

## 109. [PASS] ถูก

**คำถาม:** พวงมาลัยขับรถ สำหรับบุคคลทั่วไป ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 200 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 200
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.169` sec


---

## 110. [PASS] ถูก

**คำถาม:** คนนอก เล่น Cockpit กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 200 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 200
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.099` sec


---

## 111. [PASS] ถูก

**คำถาม:** ถ้าเป็นคนนอก จอง พวงมาลัยขับรถ ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 200 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 200
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.195` sec


---

## 112. [PASS] ถูก

**คำถาม:** ถ้าเป็นGeneral Adult จอง Cockpit ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 200 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 200
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.206` sec


---

## 113. [PASS] ถูก

**คำถาม:** พวงมาลัยขับรถ สำหรับGeneral Adult ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 200 บาทต่อ 60 นาที (Cockpit)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, 60, 200
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.216` sec


---

## 114. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษา มอ จอง VR 30 นาที ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.199` sec


---

## 115. [PASS] ถูก

**คำถาม:** VR ครึ่งชั่วโมง สำหรับนักศึกษา มอ ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.172` sec


---

## 116. [PASS] ถูก

**คำถาม:** VR 30 นาที สำหรับนักเรียน ม.อ. ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.182` sec


---

## 117. [PASS] ถูก

**คำถาม:** นักเรียน ม.อ. เล่น VR ครึ่งชั่วโมง กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.163` sec


---

## 118. [PASS] ถูก

**คำถาม:** VR 30 นาที สำหรับเด็ก PSU ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.201` sec


---

## 119. [PASS] ถูก

**คำถาม:** เด็ก PSU เล่น VR ครึ่งชั่วโมง กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.139` sec


---

## 120. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษาทั่วไป จอง VR 30 นาที ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 190
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.110` sec


---

## 121. [PASS] ถูก

**คำถาม:** VR ครึ่งชั่วโมง สำหรับนักศึกษาทั่วไป ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 190
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.099` sec


---

## 122. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษาต่างมหาลัย จอง VR 30 นาที ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 190
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.140` sec


---

## 123. [PASS] ถูก

**คำถาม:** VR ครึ่งชั่วโมง สำหรับนักศึกษาต่างมหาลัย ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 190
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.124` sec


---

## 124. [PASS] ถูก

**คำถาม:** ถ้าเป็นศิษย์เก่า PSU จอง VR 30 นาที ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 190
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.101` sec


---

## 125. [PASS] ถูก

**คำถาม:** VR ครึ่งชั่วโมง สำหรับศิษย์เก่า PSU ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 190
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.089` sec


---

## 126. [PASS] ถูก

**คำถาม:** ถ้าเป็นบุคคลทั่วไป จอง VR 30 นาที ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 525 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 525
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.189` sec


---

## 127. [PASS] ถูก

**คำถาม:** VR ครึ่งชั่วโมง สำหรับบุคคลทั่วไป ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 525 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 525
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.176` sec


---

## 128. [PASS] ถูก

**คำถาม:** คนนอก เล่น VR 30 นาที กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 525 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 525
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.109` sec


---

## 129. [PASS] ถูก

**คำถาม:** ถ้าเป็นคนนอก จอง VR ครึ่งชั่วโมง ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 525 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 525
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.237` sec


---

## 130. [PASS] ถูก

**คำถาม:** ถ้าเป็นGeneral Adult จอง VR 30 นาที ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 525 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 525
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.206` sec


---

## 131. [PASS] ถูก

**คำถาม:** VR ครึ่งชั่วโมง สำหรับGeneral Adult ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 525 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 30, 525
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.206` sec


---

## 132. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษา มอ จอง VR 1 ชั่วโมง ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.201` sec


---

## 133. [PASS] ถูก

**คำถาม:** VR 60 นาที สำหรับนักศึกษา มอ ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.180` sec


---

## 134. [PASS] ถูก

**คำถาม:** VR 1 ชั่วโมง สำหรับนักเรียน ม.อ. ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.177` sec


---

## 135. [PASS] ถูก

**คำถาม:** นักเรียน ม.อ. เล่น VR 60 นาที กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.177` sec


---

## 136. [PASS] ถูก

**คำถาม:** VR 1 ชั่วโมง สำหรับเด็ก PSU ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.160` sec


---

## 137. [PASS] ถูก

**คำถาม:** เด็ก PSU เล่น VR 60 นาที กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 0, 0, บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.152` sec


---

## 138. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษาทั่วไป จอง VR 1 ชั่วโมง ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 375 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 375
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.102` sec


---

## 139. [PASS] ถูก

**คำถาม:** VR 60 นาที สำหรับนักศึกษาทั่วไป ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 375 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 375
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.101` sec


---

## 140. [PASS] ถูก

**คำถาม:** ถ้าเป็นนักศึกษาต่างมหาลัย จอง VR 1 ชั่วโมง ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 375 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 375
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.140` sec


---

## 141. [PASS] ถูก

**คำถาม:** VR 60 นาที สำหรับนักศึกษาต่างมหาลัย ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 375 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 375
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.125` sec


---

## 142. [PASS] ถูก

**คำถาม:** ถ้าเป็นศิษย์เก่า PSU จอง VR 1 ชั่วโมง ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 375 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 375
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.101` sec


---

## 143. [PASS] ถูก

**คำถาม:** VR 60 นาที สำหรับศิษย์เก่า PSU ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 375 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 375
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.103` sec


---

## 144. [PASS] ถูก

**คำถาม:** ถ้าเป็นบุคคลทั่วไป จอง VR 1 ชั่วโมง ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 1,050 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 1050
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.162` sec


---

## 145. [PASS] ถูก

**คำถาม:** VR 60 นาที สำหรับบุคคลทั่วไป ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 1,050 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 1050
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.192` sec


---

## 146. [PASS] ถูก

**คำถาม:** คนนอก เล่น VR 1 ชั่วโมง กี่บาทต่อชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 1,050 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 1050
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.088` sec


---

## 147. [PASS] ถูก

**คำถาม:** ถ้าเป็นคนนอก จอง VR 60 นาที ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 1,050 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 1050
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.194` sec


---

## 148. [PASS] ถูก

**คำถาม:** ถ้าเป็นGeneral Adult จอง VR 1 ชั่วโมง ราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 1,050 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 1050
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.186` sec


---

## 149. [PASS] ถูก

**คำถาม:** VR 60 นาที สำหรับGeneral Adult ต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 1,050 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, 1 ชั่วโมง, 1050
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.222` sec


---

## 150. [PASS] ถูก

**คำถาม:** ราคา PC ต่อชั่วโมงเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา PC: ยังไม่พบราคาค่าบริการ PC ในฐานข้อมูล/Service Fee 2026 ที่ดึงมา จึงยังไม่ควรคำนวณยอดเงินบาทแบบฟันธง
- คำถามเป็นราคาแบบต่อรอบ/ต่อชั่วโมง จึงคิดเป็น 1 session = 1 ชั่วโมง
- บริการ PC คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)
- ยังไม่ทราบกลุ่มผู้ใช้ จึงยังเทียบเรตราคาเฉพาะกลุ่มไม่ได้
- จากภาพ Service Fee 2026 ที่มีตอนนี้ มีราคา PlayStation 5, Nintendo Switch, Cockpit และ VR แต่ไม่ปรากฏราคา PC
- ถ้าได้รับราคา PC ต่อ 1 session แล้ว ระบบจะคำนวณได้ทันทีด้วยสูตร: จำนวน session × ราคาต่อ session

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-json/wbk/v2/get-preset (service duration)
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (service fee image; PC price not shown)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PC, ไม่พบ, Service Fee
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.055` sec


---

## 151. [PASS] ถูก

**คำถาม:** นักเรียน มอ เล่น PC ฟรีไหม

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา PC: ยังไม่พบราคาค่าบริการ PC ในฐานข้อมูล/Service Fee 2026 ที่ดึงมา จึงยังไม่ควรคำนวณยอดเงินบาทแบบฟันธง
- คำถามเป็นราคาแบบต่อรอบ/ต่อชั่วโมง จึงคิดเป็น 1 session = 1 ชั่วโมง
- บริการ PC คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)
- กลุ่มผู้ใช้ที่ถาม: นักศึกษา/บุคลากร PSU
- จากภาพ Service Fee 2026 ที่มีตอนนี้ มีราคา PlayStation 5, Nintendo Switch, Cockpit และ VR แต่ไม่ปรากฏราคา PC
- ถ้าได้รับราคา PC ต่อ 1 session แล้ว ระบบจะคำนวณได้ทันทีด้วยสูตร: จำนวน session × ราคาต่อ session

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-json/wbk/v2/get-preset (service duration)
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (service fee image; PC price not shown)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PC, ไม่พบ, Service Fee
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.071` sec


---

## 152. [PASS] ถูก

**คำถาม:** คนนอกเล่นคอมต้องจ่ายเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา PC: ยังไม่พบราคาค่าบริการ PC ในฐานข้อมูล/Service Fee 2026 ที่ดึงมา จึงยังไม่ควรคำนวณยอดเงินบาทแบบฟันธง
- คำถามเป็นราคาแบบต่อรอบ/ต่อชั่วโมง จึงคิดเป็น 1 session = 1 ชั่วโมง
- บริการ PC คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)
- กลุ่มผู้ใช้ที่ถาม: บุคคลทั่วไป
- จากภาพ Service Fee 2026 ที่มีตอนนี้ มีราคา PlayStation 5, Nintendo Switch, Cockpit และ VR แต่ไม่ปรากฏราคา PC
- ถ้าได้รับราคา PC ต่อ 1 session แล้ว ระบบจะคำนวณได้ทันทีด้วยสูตร: จำนวน session × ราคาต่อ session

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-json/wbk/v2/get-preset (service duration)
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (service fee image; PC price not shown)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PC, ไม่พบ, Service Fee
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.113` sec


---

## 153. [PASS] ถูก

**คำถาม:** PC มีราคาใน service fee ไหม

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา PC: ยังไม่พบราคาค่าบริการ PC ในฐานข้อมูล/Service Fee 2026 ที่ดึงมา จึงยังไม่ควรคำนวณยอดเงินบาทแบบฟันธง
- คำถามเป็นราคาแบบต่อรอบ/ต่อชั่วโมง จึงคิดเป็น 1 session = 1 ชั่วโมง
- บริการ PC คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)
- ยังไม่ทราบกลุ่มผู้ใช้ จึงยังเทียบเรตราคาเฉพาะกลุ่มไม่ได้
- จากภาพ Service Fee 2026 ที่มีตอนนี้ มีราคา PlayStation 5, Nintendo Switch, Cockpit และ VR แต่ไม่ปรากฏราคา PC
- ถ้าได้รับราคา PC ต่อ 1 session แล้ว ระบบจะคำนวณได้ทันทีด้วยสูตร: จำนวน session × ราคาต่อ session

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-json/wbk/v2/get-preset (service duration)
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (service fee image; PC price not shown)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PC, ไม่พบ, Service Fee
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.079` sec


---

## 154. [PASS] ถูก

**คำถาม:** ทำไมคำนวณ PC ไม่ได้

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา PC: ยังไม่พบราคาค่าบริการ PC ในฐานข้อมูล/Service Fee 2026 ที่ดึงมา จึงยังไม่ควรคำนวณยอดเงินบาทแบบฟันธง
- คำถามเป็นราคาแบบต่อรอบ/ต่อชั่วโมง จึงคิดเป็น 1 session = 1 ชั่วโมง
- บริการ PC คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)
- ยังไม่ทราบกลุ่มผู้ใช้ จึงยังเทียบเรตราคาเฉพาะกลุ่มไม่ได้
- จากภาพ Service Fee 2026 ที่มีตอนนี้ มีราคา PlayStation 5, Nintendo Switch, Cockpit และ VR แต่ไม่ปรากฏราคา PC
- ถ้าได้รับราคา PC ต่อ 1 session แล้ว ระบบจะคำนวณได้ทันทีด้วยสูตร: จำนวน session × ราคาต่อ session

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-json/wbk/v2/get-preset (service duration)
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (service fee image; PC price not shown)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PC, ไม่พบ, Service Fee
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.052` sec


---

## 155. [PASS] ถูก

**คำถาม:** จอง PC 9 ถึง 11 ต้องเสียกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา PC: ยังไม่พบราคาค่าบริการ PC ในฐานข้อมูล/Service Fee 2026 ที่ดึงมา จึงยังไม่ควรคำนวณยอดเงินบาทแบบฟันธง
- ช่วงเวลาที่ถามคือ 09:00-11:00 = 2 ชั่วโมง
- บริการ PC คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 2 session(s)
- ยังไม่ทราบกลุ่มผู้ใช้ จึงยังเทียบเรตราคาเฉพาะกลุ่มไม่ได้
- จากภาพ Service Fee 2026 ที่มีตอนนี้ มีราคา PlayStation 5, Nintendo Switch, Cockpit และ VR แต่ไม่ปรากฏราคา PC
- ถ้าได้รับราคา PC ต่อ 1 session แล้ว ระบบจะคำนวณได้ทันทีด้วยสูตร: จำนวน session × ราคาต่อ session

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-json/wbk/v2/get-preset (service duration)
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (service fee image; PC price not shown)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PC, ไม่พบ, Service Fee
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.077` sec


---

## 156. [PASS] ถูก

**คำถาม:** ถ้าเล่น PC 2 ชั่วโมงคิดเงินยังไง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา PC: ยังไม่พบราคาค่าบริการ PC ในฐานข้อมูล/Service Fee 2026 ที่ดึงมา จึงยังไม่ควรคำนวณยอดเงินบาทแบบฟันธง
- คำถามเป็นราคาแบบต่อรอบ/ต่อชั่วโมง จึงคิดเป็น 1 session = 1 ชั่วโมง
- บริการ PC คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)
- ยังไม่ทราบกลุ่มผู้ใช้ จึงยังเทียบเรตราคาเฉพาะกลุ่มไม่ได้
- จากภาพ Service Fee 2026 ที่มีตอนนี้ มีราคา PlayStation 5, Nintendo Switch, Cockpit และ VR แต่ไม่ปรากฏราคา PC
- ถ้าได้รับราคา PC ต่อ 1 session แล้ว ระบบจะคำนวณได้ทันทีด้วยสูตร: จำนวน session × ราคาต่อ session

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-json/wbk/v2/get-preset (service duration)
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (service fee image; PC price not shown)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PC, ไม่พบ, Service Fee
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.104` sec


---

## 157. [PASS] ถูก

**คำถาม:** PC price for general student?

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา PC: ยังไม่พบราคาค่าบริการ PC ในฐานข้อมูล/Service Fee 2026 ที่ดึงมา จึงยังไม่ควรคำนวณยอดเงินบาทแบบฟันธง
- คำถามเป็นราคาแบบต่อรอบ/ต่อชั่วโมง จึงคิดเป็น 1 session = 1 ชั่วโมง
- บริการ PC คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)
- กลุ่มผู้ใช้ที่ถาม: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)
- จากภาพ Service Fee 2026 ที่มีตอนนี้ มีราคา PlayStation 5, Nintendo Switch, Cockpit และ VR แต่ไม่ปรากฏราคา PC
- ถ้าได้รับราคา PC ต่อ 1 session แล้ว ระบบจะคำนวณได้ทันทีด้วยสูตร: จำนวน session × ราคาต่อ session

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-json/wbk/v2/get-preset (service duration)
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (service fee image; PC price not shown)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PC, ไม่พบ, Service Fee
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.034` sec


---

## 158. [PASS] ถูก

**คำถาม:** ราคาค่าบริการทั้งหมดมีอะไรบ้าง

**คำตอบ(จาก AI):**

คำตอบ:
- ตารางค่าบริการ PSU Esports Studio - Phuket Service Fee 2026 ตามภาพ: PlayStation 5 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 150 บาท. Nintendo Switch 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 140 บาท. Nintendo Switch 1 ชั่วโมง 3-4 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 100 บาท, General Adult 280 บาท. Cockpit 1 ชั่วโมง 1 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 65 บาท, General Adult 200 บาท. VR 30 นาที 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 190 บาท, General Adult 525 บาท. VR 1 ชั่วโมง 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 375 บาท, General Adult 1050 บาท. ภาพนี้ระบุว่าค่าบริการมีผลตั้งแต่ 1 February 2026 ถึง 31 March 2026 และไม่ปรากฏราคา PC

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (curated_service_fee_2026_full_table)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, Nintendo Switch, Cockpit, VR
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.120` sec


---

## 159. [PASS] ถูก

**คำถาม:** สรุป service fee ให้หน่อย

**คำตอบ(จาก AI):**

คำตอบ:
- ตารางค่าบริการ PSU Esports Studio - Phuket Service Fee 2026 ตามภาพ: PlayStation 5 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 150 บาท. Nintendo Switch 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 140 บาท. Nintendo Switch 1 ชั่วโมง 3-4 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 100 บาท, General Adult 280 บาท. Cockpit 1 ชั่วโมง 1 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 65 บาท, General Adult 200 บาท. VR 30 นาที 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 190 บาท, General Adult 525 บาท. VR 1 ชั่วโมง 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 375 บาท, General Adult 1050 บาท. ภาพนี้ระบุว่าค่าบริการมีผลตั้งแต่ 1 February 2026 ถึง 31 March 2026 และไม่ปรากฏราคา PC

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (curated_service_fee_2026_full_table)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, Nintendo Switch, Cockpit, VR
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.110` sec


---

## 160. [PASS] ถูก

**คำถาม:** ตารางราคา 2026 มีอะไร

**คำตอบ(จาก AI):**

คำตอบ:
- ตารางค่าบริการ PSU Esports Studio - Phuket Service Fee 2026 ตามภาพ: PlayStation 5 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 150 บาท. Nintendo Switch 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 140 บาท. Nintendo Switch 1 ชั่วโมง 3-4 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 100 บาท, General Adult 280 บาท. Cockpit 1 ชั่วโมง 1 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 65 บาท, General Adult 200 บาท. VR 30 นาที 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 190 บาท, General Adult 525 บาท. VR 1 ชั่วโมง 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 375 บาท, General Adult 1050 บาท. ภาพนี้ระบุว่าค่าบริการมีผลตั้งแต่ 1 February 2026 ถึง 31 March 2026 และไม่ปรากฏราคา PC

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (curated_service_fee_2026_full_table)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, Nintendo Switch, Cockpit, VR
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.093` sec


---

## 161. [PASS] ถูก

**คำถาม:** ค่าเล่นแต่ละเครื่องเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ตารางค่าบริการ PSU Esports Studio - Phuket Service Fee 2026 ตามภาพ: PlayStation 5 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 150 บาท. Nintendo Switch 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 140 บาท. Nintendo Switch 1 ชั่วโมง 3-4 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 100 บาท, General Adult 280 บาท. Cockpit 1 ชั่วโมง 1 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 65 บาท, General Adult 200 บาท. VR 30 นาที 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 190 บาท, General Adult 525 บาท. VR 1 ชั่วโมง 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 375 บาท, General Adult 1050 บาท. ภาพนี้ระบุว่าค่าบริการมีผลตั้งแต่ 1 February 2026 ถึง 31 March 2026 และไม่ปรากฏราคา PC

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (curated_service_fee_2026_full_table)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, Nintendo Switch, Cockpit, VR
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.115` sec


---

## 162. [PASS] ถูก

**คำถาม:** service fee table includes what

**คำตอบ(จาก AI):**

Answer:
- ตารางค่าบริการ PSU Esports Studio - Phuket Service Fee 2026 ตามภาพ: PlayStation 5 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 150 บาท. Nintendo Switch 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 140 บาท. Nintendo Switch 1 ชั่วโมง 3-4 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 100 บาท, General Adult 280 บาท. Cockpit 1 ชั่วโมง 1 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 65 บาท, General Adult 200 บาท. VR 30 นาที 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 190 บาท, General Adult 525 บาท. VR 1 ชั่วโมง 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 375 บาท, General Adult 1050 บาท. ภาพนี้ระบุว่าค่าบริการมีผลตั้งแต่ 1 February 2026 ถึง 31 March 2026 และไม่ปรากฏราคา PC

Sources:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (curated_service_fee_2026_full_table)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, Nintendo Switch, Cockpit, VR
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.127` sec


---

## 163. [PASS] ถูก

**คำถาม:** ขอเรทราคา PS5 Switch Cockpit VR

**คำตอบ(จาก AI):**

คำตอบ:
- ตารางค่าบริการ PSU Esports Studio - Phuket Service Fee 2026 ตามภาพ: PlayStation 5 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 150 บาท. Nintendo Switch 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 140 บาท. Nintendo Switch 1 ชั่วโมง 3-4 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 100 บาท, General Adult 280 บาท. Cockpit 1 ชั่วโมง 1 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 65 บาท, General Adult 200 บาท. VR 30 นาที 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 190 บาท, General Adult 525 บาท. VR 1 ชั่วโมง 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 375 บาท, General Adult 1050 บาท. ภาพนี้ระบุว่าค่าบริการมีผลตั้งแต่ 1 February 2026 ถึง 31 March 2026 และไม่ปรากฏราคา PC

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (curated_service_fee_2026_full_table)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, Nintendo Switch, Cockpit, VR
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.092` sec


---

## 164. [PASS] ถูก

**คำถาม:** จองต้องล่วงหน้ากี่ชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
ต้องจองล่วงหน้าผ่านระบบออนไลน์ก่อนเวลาใช้งานอย่างน้อย 1 ชั่วโมง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_booking_advance / Reservation, curated_reservation_advance_time)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 1 ชั่วโมง
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.027` sec


---

## 165. [PASS] ถูก

**คำถาม:** ถ้าจะเล่นบ่ายสองต้องจองก่อนอย่างน้อยกี่โมง

**คำตอบ(จาก AI):**

คำตอบ:
ต้องจองล่วงหน้าผ่านระบบออนไลน์ก่อนเวลาใช้งานอย่างน้อย 1 ชั่วโมง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_booking_advance / Reservation, curated_reservation_advance_time)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 1 ชั่วโมง
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.063` sec


---

## 166. [PASS] ถูก

**คำถาม:** ต้อง book ล่วงหน้ากี่ hr

**คำตอบ(จาก AI):**

คำตอบ:
ต้องจองล่วงหน้าผ่านระบบออนไลน์ก่อนเวลาใช้งานอย่างน้อย 1 ชั่วโมง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_booking_advance / Reservation, curated_reservation_advance_time)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 1 ชั่วโมง
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.025` sec


---

## 167. [PASS] ถูก

**คำถาม:** จองก่อน 1 ชั่วโมงใช่ไหม

**คำตอบ(จาก AI):**

คำตอบ:
- การยกเลิกการจองต้องทำล่วงหน้าอย่างน้อย 1 ชั่วโมง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_cancel_1_hour)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 1 ชั่วโมง
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.053` sec


---

## 168. [PASS] ถูก

**คำถาม:** walk in ได้ไหมหรือต้องจองก่อน

**คำตอบ(จาก AI):**

คำตอบ:
ต้องจองล่วงหน้าผ่านระบบออนไลน์ก่อนเวลาใช้งานอย่างน้อย 1 ชั่วโมง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_booking_advance / Reservation, curated_reservation_advance_time)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 1 ชั่วโมง
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.038` sec


---

## 169. [PASS] ถูก

**คำถาม:** จองได้สูงสุดกี่ session

**คำตอบ(จาก AI):**

คำตอบ:
การจอง 1 ครั้งสามารถจองได้สูงสุด 3 Sessions

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_booking_max_sessions / curated_reservation_max_sessions)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 3 Sessions
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.024` sec


---

## 170. [PASS] ถูก

**คำถาม:** ครั้งนึงจองได้กี่รอบ

**คำตอบ(จาก AI):**

คำตอบ:
- การจอง 1 ครั้งสามารถจองได้สูงสุด 3 Sessions

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_reservation_max_sessions)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 3 Sessions
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.073` sec


---

## 171. [PASS] ถูก

**คำถาม:** one booking max sessions?

**คำตอบ(จาก AI):**

คำตอบ:
One booking can include a maximum of 3 sessions. This information is found in both SOURCE 1 and SOURCE 2, which are identical except for the language used.

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 3 Sessions
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_llm` | model: `qwen2.5:3b` | elapsed: `10.106` sec


---

## 172. [PASS] ถูก

**คำถาม:** จองทีเดียว 4 sessions ได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
- การจอง 1 ครั้งสามารถจองได้สูงสุด 3 Sessions

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_reservation_max_sessions)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 3 Sessions
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.184` sec


---

## 173. [PASS] ถูก

**คำถาม:** จองสามรอบได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
- การจอง 1 ครั้งสามารถจองได้สูงสุด 3 Sessions

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_reservation_max_sessions)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 3 Sessions
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.092` sec


---

## 174. [PASS] ถูก

**คำถาม:** หลังจองต้องจ่ายภายในกี่นาที

**คำตอบ(จาก AI):**

คำตอบ:
หลังจองต้องชำระเงินทันที หากไม่ชำระภายใน 10 นาที การจองจะถูกยกเลิก และหากต้องการใช้บริการต้องทำรายการจองใหม่

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_payment_10_minutes / Reservation, curated_payment_10_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 10 นาที, ยกเลิก, จองใหม่
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.061` sec


---

## 175. [PASS] ถูก

**คำถาม:** ไม่จ่ายใน 10 นาทีจะเกิดอะไร

**คำตอบ(จาก AI):**

คำตอบ:
หลังจองต้องชำระเงินทันที หากไม่ชำระภายใน 10 นาที การจองจะถูกยกเลิก และหากต้องการใช้บริการต้องทำรายการจองใหม่

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_payment_10_minutes / Reservation, curated_payment_10_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 10 นาที, ยกเลิก, จองใหม่
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.032` sec


---

## 176. [PASS] ถูก

**คำถาม:** payment timeout กี่นาที

**คำตอบ(จาก AI):**

คำตอบ:
หลังจองต้องชำระเงินทันที หากไม่ชำระภายใน 10 นาที การจองจะถูกยกเลิก และหากต้องการใช้บริการต้องทำรายการจองใหม่

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_payment_10_minutes / Reservation, curated_payment_10_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 10 นาที, ยกเลิก, จองใหม่
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.052` sec


---

## 177. [PASS] ถูก

**คำถาม:** จองแล้วลืมจ่ายเงิน ระบบจะยกเลิกไหม

**คำตอบ(จาก AI):**

คำตอบ:
หลังจองต้องชำระเงินทันที หากไม่ชำระภายใน 10 นาที การจองจะถูกยกเลิก และหากต้องการใช้บริการต้องทำรายการจองใหม่

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_payment_10_minutes / Reservation, curated_payment_10_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 10 นาที, ยกเลิก, จองใหม่
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.054` sec


---

## 178. [PASS] ถูก

**คำถาม:** ชำระเงินหลัง booking ต้องเร็วแค่ไหน

**คำตอบ(จาก AI):**

คำตอบ:
หลังจองต้องชำระเงินทันที หากไม่ชำระภายใน 10 นาที การจองจะถูกยกเลิก และหากต้องการใช้บริการต้องทำรายการจองใหม่

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_payment_10_minutes / Reservation, curated_payment_10_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 10 นาที, ยกเลิก, จองใหม่
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.075` sec


---

## 179. [PASS] ถูก

**คำถาม:** กดจองแล้วแก้ไขได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
- เมื่อกดจองแล้วจะไม่สามารถแก้ไขข้อมูลได้ หากต้องการแก้ไขต้องยกเลิกการจองผ่านทางอีเมลก่อนเวลาใช้งานอย่างน้อย 1 ชั่วโมง แล้วจองใหม่อีกครั้ง พร้อมแนบสลิปการโอนเงินเดิม

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_booking_no_edit)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ยกเลิก, 1 ชั่วโมง, จองใหม่, สลิป
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.092` sec


---

## 180. [PASS] ถูก

**คำถาม:** ถ้ากรอกข้อมูลผิดหลังจองต้องทำยังไง

**คำตอบ(จาก AI):**

คำตอบ:
เมื่อกดจองแล้วจะไม่สามารถแก้ไขข้อมูลได้ หากต้องการแก้ไขต้องยกเลิกการจองผ่านทางอีเมลก่อนเวลาใช้งานอย่างน้อย 1 ชั่วโมง แล้วจองใหม่ พร้อมแนบสลิปเดิม

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_edit_booking / Reservation, curated_booking_no_edit)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ยกเลิก, 1 ชั่วโมง, จองใหม่, สลิป
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.067` sec


---

## 181. [PASS] ถูก

**คำถาม:** แก้เวลา booking ได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
เมื่อกดจองแล้วจะไม่สามารถแก้ไขข้อมูลได้ หากต้องการแก้ไขต้องยกเลิกการจองผ่านทางอีเมลก่อนเวลาใช้งานอย่างน้อย 1 ชั่วโมง แล้วจองใหม่ พร้อมแนบสลิปเดิม

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_edit_booking / Reservation, curated_booking_no_edit)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ยกเลิก, 1 ชั่วโมง, จองใหม่, สลิป
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.036` sec


---

## 182. [PASS] ถูก

**คำถาม:** ต้องยกเลิกแล้วจองใหม่ใช่ไหม

**คำตอบ(จาก AI):**

คำตอบ:
- เมื่อกดจองแล้วจะไม่สามารถแก้ไขข้อมูลได้ หากต้องการแก้ไขต้องยกเลิกการจองผ่านทางอีเมลก่อนเวลาใช้งานอย่างน้อย 1 ชั่วโมง แล้วจองใหม่อีกครั้ง พร้อมแนบสลิปการโอนเงินเดิม

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_booking_no_edit)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ยกเลิก, 1 ชั่วโมง, จองใหม่, สลิป
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.113` sec


---

## 183. [PASS] ถูก

**คำถาม:** แนบสลิปเดิมตอนจองใหม่ได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
เมื่อกดจองแล้วจะไม่สามารถแก้ไขข้อมูลได้ หากต้องการแก้ไขต้องยกเลิกการจองผ่านทางอีเมลก่อนเวลาใช้งานอย่างน้อย 1 ชั่วโมง แล้วจองใหม่ พร้อมแนบสลิปเดิม

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_edit_booking / Reservation, curated_booking_no_edit)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ยกเลิก, 1 ชั่วโมง, จองใหม่, สลิป
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.044` sec


---

## 184. [PASS] ถูก

**คำถาม:** สิทธิ์การจองโอนให้เพื่อนได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
- ไม่สามารถโอนสิทธิ์การจองให้กับผู้อื่นได้

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_booking_non_transferable)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่สามารถโอนสิทธิ์
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.125` sec


---

## 185. [PASS] ถูก

**คำถาม:** ให้คนอื่นมาใช้ booking แทนได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ไม่สามารถโอนสิทธิ์การจองให้กับผู้อื่นได้

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_booking_non_transferable / Reservation, curated_booking_non_transferable)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่สามารถโอนสิทธิ์
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.064` sec


---

## 186. [PASS] ถูก

**คำถาม:** transfer booking ได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ไม่สามารถโอนสิทธิ์การจองให้กับผู้อื่นได้

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_booking_non_transferable / Reservation, curated_booking_non_transferable)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่สามารถโอนสิทธิ์
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.042` sec


---

## 187. [PASS] ถูก

**คำถาม:** จองแทนกันได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ไม่สามารถโอนสิทธิ์การจองให้กับผู้อื่นได้

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_booking_non_transferable / Reservation, curated_booking_non_transferable)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่สามารถโอนสิทธิ์
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.019` sec


---

## 188. [PASS] ถูก

**คำถาม:** เช็คอินล่วงหน้าได้กี่นาที

**คำตอบ(จาก AI):**

คำตอบ:
เช็คอินได้ล่วงหน้าสูงสุด 30 นาที และต้องเช็คอินก่อนเวลาเริ่มต้นของรอบที่จอง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_checkin_advance / curated_checkin_30_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 30 นาที
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.035` sec


---

## 189. [PASS] ถูก

**คำถาม:** check in ได้เร็วสุดกี่นาที

**คำตอบ(จาก AI):**

คำตอบ: Check-in สามารถทำได้ล่วงหน้าสูงสุดถึง 30 นาทีเท่านั้น ตามข้อมูลจากแหล่งที่มา [SOURCE 2]

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 30 นาที
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_llm` | model: `qwen2.5:3b` | elapsed: `3.607` sec


---

## 190. [PASS] ถูก

**คำถาม:** เช็คอินก่อนเวลาได้ 1800 วินาทีไหม

**คำตอบ(จาก AI):**

คำตอบ:
- ผู้ใช้งานต้องเช็คอินก่อนเวลาเริ่มต้นของรอบที่จอง โดยสามารถเช็คอินได้ล่วงหน้าสูงสุด 30 นาที และต้องเช็คอินก่อนถึงเวลาเริ่มต้นของรอบ

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_checkin_30_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 30 นาที
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.113` sec


---

## 191. [PASS] ถูก

**คำถาม:** ต้องเช็คอินก่อนเริ่มรอบใช่ไหม

**คำตอบ(จาก AI):**

คำตอบ:
- ผู้ใช้งานต้องเช็คอินก่อนเวลาเริ่มต้นของรอบที่จอง โดยสามารถเช็คอินได้ล่วงหน้าสูงสุด 30 นาที และต้องเช็คอินก่อนถึงเวลาเริ่มต้นของรอบ

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_checkin_30_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 30 นาที
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.126` sec


---

## 192. [PASS] ถูก

**คำถาม:** ถ้า check-in ไม่ทันจะโดนอะไร

**คำตอบ(จาก AI):**

คำตอบ:
ถ้าไม่เช็คอินก่อนเวลาเริ่มต้นของรอบ ระบบจะยกเลิกการจองทันที และไม่มีการคืนเงิน

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_checkin_late / curated_checkin_late_cancel)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ยกเลิก, ไม่มีการคืนเงิน
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.045` sec


---

## 193. [PASS] ถูก

**คำถาม:** เช็คอินต้องใช้บัตรอะไร

**คำตอบ(จาก AI):**

คำตอบ:
- เมื่อเช็คอินเข้าใช้บริการ ต้องนำบัตรประจำตัวนักศึกษา บัตรประจำตัวบุคลากร หรือบัตรประชาชนมาแสดง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_checkin_id_required)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: บัตรประชาชน
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.093` sec


---

## 194. [PASS] ถูก

**คำถาม:** แสดงบัตรประชาชนได้ไหมตอนเช็คอิน

**คำตอบ(จาก AI):**

คำตอบ:
- เมื่อเช็คอินเข้าใช้บริการ ต้องนำบัตรประจำตัวนักศึกษา บัตรประจำตัวบุคลากร หรือบัตรประชาชนมาแสดง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_checkin_id_required)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: บัตรประชาชน
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.103` sec


---

## 195. [PASS] ถูก

**คำถาม:** ชำระเงินผ่านอะไร

**คำตอบ(จาก AI):**

คำตอบ:
- ชำระเงินโดยโอนเข้าบัญชี Siam Commercial Bank (ธนาคารไทยพาณิชย์) ชื่อบัญชี PSU Esports Studio - Phuket เลขบัญชี 795-276244-1 และแนบสลิปการโอนเงิน

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_payment_bank)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ธนาคารไทยพาณิชย์, 795-276244-1
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.069` sec


---

## 196. [PASS] ถูก

**คำถาม:** โอนเงินไปบัญชีไหน

**คำตอบ(จาก AI):**

คำตอบ:
ชำระเงินโดยโอนผ่าน Siam Commercial Bank (ธนาคารไทยพาณิชย์) ชื่อบัญชี PSU Esports Studio - Phuket เลขบัญชี 795-276244-1 และต้องแนบสลิปการโอนเงิน

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_payment_bank / Reservation, curated_payment_bank)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ธนาคารไทยพาณิชย์, 795-276244-1
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.021` sec


---

## 197. [PASS] ถูก

**คำถาม:** ชื่อบัญชีจ่ายค่าจองคืออะไร

**คำตอบ(จาก AI):**

คำตอบ:
ชำระเงินโดยโอนผ่าน Siam Commercial Bank (ธนาคารไทยพาณิชย์) ชื่อบัญชี PSU Esports Studio - Phuket เลขบัญชี 795-276244-1 และต้องแนบสลิปการโอนเงิน

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_payment_bank / Reservation, curated_payment_bank)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ธนาคารไทยพาณิชย์, 795-276244-1
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.042` sec


---

## 198. [PASS] ถูก

**คำถาม:** เลขบัญชีธนาคารสำหรับจองคืออะไร

**คำตอบ(จาก AI):**

คำตอบ:
ชำระเงินโดยโอนผ่าน Siam Commercial Bank (ธนาคารไทยพาณิชย์) ชื่อบัญชี PSU Esports Studio - Phuket เลขบัญชี 795-276244-1 และต้องแนบสลิปการโอนเงิน

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_payment_bank / Reservation, curated_payment_bank)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ธนาคารไทยพาณิชย์, 795-276244-1
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.040` sec


---

## 199. [PASS] ถูก

**คำถาม:** ธนาคารที่ใช้รับเงินคืออะไร

**คำตอบ(จาก AI):**

คำตอบ:
ชำระเงินโดยโอนผ่าน Siam Commercial Bank (ธนาคารไทยพาณิชย์) ชื่อบัญชี PSU Esports Studio - Phuket เลขบัญชี 795-276244-1 และต้องแนบสลิปการโอนเงิน

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_payment_bank / Reservation, curated_payment_bank)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ธนาคารไทยพาณิชย์, 795-276244-1
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.039` sec


---

## 200. [PASS] ถูก

**คำถาม:** PS5 มี Spider-Man 2 ไหม

**คำตอบ(จาก AI):**

คำตอบ:
เกม PlayStation 5 ที่มีในรายการรวมถึง Marvel’s Spider-Man 2, Fortnite, God of War Ragnarok และ TEKKEN 8

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_ps5_specific_games / Reservation, curated_games_ps5, reservation-service-ps5)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, Spider-Man 2
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.029` sec


---

## 201. [PASS] ถูก

**คำถาม:** เพลย์ห้ามี tekken 8 หรือเปล่า

**คำตอบ(จาก AI):**

คำตอบ:
เกม PlayStation 5 ที่มีในรายการรวมถึง Marvel’s Spider-Man 2, Fortnite, God of War Ragnarok และ TEKKEN 8

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_ps5_specific_games / Reservation, curated_games_ps5, reservation-service-ps5)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, TEKKEN 8
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.038` sec


---

## 202. [PASS] ถูก

**คำถาม:** เกมบน PlayStation มีอะไรบ้าง

**คำตอบ(จาก AI):**

คำตอบ:
เกม PlayStation 5 ที่มีในรายการ ได้แก่ Call of Duty: Modern Warfare III, Delta Force, EA Sports FC 24, eFootball, FINAL FANTASY XVI, Fortnite, God of War Ragnarok, Hogwarts Legacy, Marvel’s Spider-Man 2, Naruto X Boruto Ultimate Ninja Storm Connections, Resident Evil 4, Resident Evil Village, TEKKEN 8, THE FINALS, The Last of Us Part I, The Last of Us Part II Remastered และ Uncharted: Legacy of Thieves Collection

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_ps5_games / Reservation, curated_games_ps5, reservation-service-ps5)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, TEKKEN 8
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.052` sec


---

## 203. [PASS] ถูก

**คำถาม:** PS5 เล่น Fortnite ได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
เกม PlayStation 5 ที่มีในรายการรวมถึง Marvel’s Spider-Man 2, Fortnite, God of War Ragnarok และ TEKKEN 8

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_ps5_specific_games / Reservation, curated_games_ps5, reservation-service-ps5)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, Fortnite
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.041` sec


---

## 204. [PASS] ถูก

**คำถาม:** มี God of War Ragnarok ไหม

**คำตอบ(จาก AI):**

คำตอบ:
เกม PlayStation 5 ที่มีในรายการรวมถึง Marvel’s Spider-Man 2, Fortnite, God of War Ragnarok และ TEKKEN 8

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_ps5_specific_games / Reservation, curated_games_ps5, reservation-service-ps5)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PlayStation 5, God of War Ragnarok
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.041` sec


---

## 205. [PASS] ถูก

**คำถาม:** Switch มี Mario Kart ไหม

**คำตอบ(จาก AI):**

คำตอบ:
เกม Nintendo Switch ที่มีในรายการรวมถึง Mario Kart 8 Deluxe, Overcooked 2, Nintendo Switch Sports และ Super Smash Bros Ultimate

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_switch_specific_games / Reservation, curated_games_switch, reservation-service-nintendo)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, Mario Kart
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.042` sec


---

## 206. [PASS] ถูก

**คำถาม:** นินเทนโดมี Overcooked 2 ไหม

**คำตอบ(จาก AI):**

คำตอบ:
เกม Nintendo Switch ที่มีในรายการรวมถึง Mario Kart 8 Deluxe, Overcooked 2, Nintendo Switch Sports และ Super Smash Bros Ultimate

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_switch_specific_games / Reservation, curated_games_switch, reservation-service-nintendo)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, Overcooked 2
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.046` sec


---

## 207. [PASS] ถูก

**คำถาม:** เกม Nintendo มีอะไรบ้าง

**คำตอบ(จาก AI):**

คำตอบ:
เกม Nintendo Switch ที่มีในรายการ ได้แก่ Pokémon Champions, Animal Crossing: New Horizon, It Takes Two, Little Nightmares II, Luigi’s Mansion 3, Mario Kart 8 Deluxe, Mario Party Superstars, Monster Hunter Rise, Moving Out 2, New Super Mario Bros. U Deluxe, Nintendo Switch Sports, Overcooked, Overcooked 2, Ring Fit Adventure, Super Mario Odyssey, Super Smash Bros Ultimate และ The Legend of Zelda: Breath of The Wild

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_switch_games / Reservation, curated_games_switch, reservation-service-nintendo)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, Mario Kart
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.044` sec


---

## 208. [PASS] ถูก

**คำถาม:** เล่น Super Smash Bros ที่ศูนย์ได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
เกม Nintendo Switch ที่มีในรายการรวมถึง Mario Kart 8 Deluxe, Overcooked 2, Nintendo Switch Sports และ Super Smash Bros Ultimate

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_switch_specific_games / Reservation, curated_games_switch, reservation-service-nintendo)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, Super Smash
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.072` sec


---

## 209. [PASS] ถูก

**คำถาม:** Switch Sports มีไหม

**คำตอบ(จาก AI):**

คำตอบ:
เกม Nintendo Switch ที่มีในรายการรวมถึง Mario Kart 8 Deluxe, Overcooked 2, Nintendo Switch Sports และ Super Smash Bros Ultimate

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_switch_specific_games / Reservation, curated_games_switch, reservation-service-nintendo)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Nintendo Switch, Switch Sports
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.043` sec


---

## 210. [PASS] ถูก

**คำถาม:** PC มี valorant ไหม

**คำตอบ(จาก AI):**

คำตอบ:
เกม PC ที่มีในรายการ ได้แก่ Tekken 8, Counter-Strike 2, League of Legends, PUBG: BATTLEGROUNDS, VALORANT และ Call of Duty: Warzone

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_pc_specific_games / Reservation, curated_games_pc, reservation-service-pc)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PC, VALORANT
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.019` sec


---

## 211. [PASS] ถูก

**คำถาม:** คอมเล่น CS2 ได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
เกม PC ที่มีในรายการ ได้แก่ Tekken 8, Counter-Strike 2, League of Legends, PUBG: BATTLEGROUNDS, VALORANT และ Call of Duty: Warzone

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_pc_specific_games / Reservation, curated_games_pc, reservation-service-pc)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PC, Counter-Strike 2
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.024` sec


---

## 212. [PASS] ถูก

**คำถาม:** PC games list

**คำตอบ(จาก AI):**

Answer:
PC games include Tekken 8, Counter-Strike 2, League of Legends, PUBG: BATTLEGROUNDS, VALORANT, and Call of Duty: Warzone.

Sources:
- https://esports.computing.psu.ac.th/ (rule_pc_games / Reservation, curated_games_pc, reservation-service-pc)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PC, VALORANT
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.009` sec


---

## 213. [PASS] ถูก

**คำถาม:** มี PUBG บน PC ไหม

**คำตอบ(จาก AI):**

คำตอบ: ในรายการเกมที่ปรากฏในรายการเกมบน PC (SOURCE 1) มี PUBG: BATTLEGROUNDS เป็นหนึ่งในเกมที่แสดงให้เห็นว่ามี PUBG บน PC

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PC, PUBG
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_llm` | model: `qwen2.5:3b` | elapsed: `3.630` sec


---

## 214. [PASS] ถูก

**คำถาม:** Warzone อยู่เครื่อง PC ไหน

**คำตอบ(จาก AI):**

คำตอบ:
เกม PC ที่มีในรายการ ได้แก่ Tekken 8, Counter-Strike 2, League of Legends, PUBG: BATTLEGROUNDS, VALORANT และ Call of Duty: Warzone

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_pc_specific_games / Reservation, curated_games_pc, reservation-service-pc)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PC, Warzone
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.051` sec


---

## 215. [PASS] ถูก

**คำถาม:** VR เล่นเกมอะไร

**คำตอบ(จาก AI):**

คำตอบ:
เกม VR Station ที่มีในรายการ ได้แก่ Beat Saber และ Horizon Call of the Mountain

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_vr_games / Reservation, curated_games_vr, reservation-service-vr)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, Beat Saber
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.022` sec


---

## 216. [PASS] ถูก

**คำถาม:** Beat Saber มีไหม

**คำตอบ(จาก AI):**

คำตอบ:
เกม VR Station ที่มีในรายการ ได้แก่ Beat Saber และ Horizon Call of the Mountain

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_vr_specific_games / Reservation, curated_games_vr, reservation-service-vr)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, Beat Saber
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.016` sec


---

## 217. [PASS] ถูก

**คำถาม:** แว่น VR มี Horizon ไหม

**คำตอบ(จาก AI):**

คำตอบ:
เกม VR Station ที่มีในรายการ ได้แก่ Beat Saber และ Horizon Call of the Mountain

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_vr_specific_games / Reservation, curated_games_vr, reservation-service-vr)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VR, Horizon
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.046` sec


---

## 218. [PASS] ถูก

**คำถาม:** Cockpit เล่นเกมอะไร

**คำตอบ(จาก AI):**

คำตอบ:
Cockpit ใช้เล่นเกม Gran Turismo 7 (Single Player) ได้

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_cockpit_games / Reservation, curated_games_cockpit, reservation-service-cockpit)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Cockpit, Gran Turismo 7
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.040` sec


---

## 219. [PASS] ถูก

**คำถาม:** พวงมาลัยใช้เล่น Gran Turismo ใช่ไหม

**คำตอบ(จาก AI):**

คำตอบ:
Cockpit หรือพวงมาลัยใช้เล่นเกม Gran Turismo 7 (Single Player) ได้

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_cockpit_gran_turismo / Reservation, curated_games_cockpit, reservation-service-cockpit)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Gran Turismo 7
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.087` sec


---

## 220. [PASS] ถูก

**คำถาม:** PC Zone มีอุปกรณ์อะไรบ้าง

**คำตอบ(จาก AI):**

คำตอบ:
ใน PC Zone มี Gaming Monitor 10 Units, Gaming Chair 10 Units, Gaming Keyboard, Gaming Headset, Gaming Mouse และ Gaming PC รุ่น MSI MAG Infinite S3 14th 10 Units

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (rule_equipment_pc_zone / home, curated_home_equipment_pc_zone)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Gaming PC, Gaming Monitor, Gaming Chair
Expected source keywords: home
หมวด: equipment
ชนิดคำตอบ: list

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.044` sec


---

## 221. [PASS] ถูก

**คำถาม:** คอมที่ศูนย์มีทั้งหมดกี่เครื่อง

**คำตอบ(จาก AI):**

คำตอบ:
- ใน PC Zone มี Gaming Monitor 10 Units, Gaming Chair 10 Units, Gaming Keyboard, Gaming Headset, Gaming Mouse และ Gaming PC รุ่น MSI MAG Infinite S3 14th 10 Units
- Cockpit Zone มี TV 65" 2 Units, Logitech G923 TRUEFORCE Racing wheel + Driving Force Shifter 2 Units, Racezone Full Cockpit V3 Steering Wheel Stand with Seat Set 2 Units และ Pulse Elite Wireless Headset 2 Units
- VR Zone มี PlayStation 5 Slim With Ultra HD Blu-Ray Disc Drive 1 Units และ PlayStation VR รุ่น Sony PlayStation VR2 1 Unit

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (curated_home_equipment_pc_zone)
- https://esports.phuket.psu.ac.th/home (curated_home_equipment_cockpit_zone)
- https://esports.phuket.psu.ac.th/home (curated_home_equipment_vr_zone)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Gaming PC, 10 Units
Expected source keywords: home
หมวด: equipment
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.127` sec


---

## 222. [PASS] ถูก

**คำถาม:** Gaming PC รุ่นอะไร

**คำตอบ(จาก AI):**

คำตอบ:
- ใน PC Zone มี Gaming Monitor 10 Units, Gaming Chair 10 Units, Gaming Keyboard, Gaming Headset, Gaming Mouse และ Gaming PC รุ่น MSI MAG Infinite S3 14th 10 Units

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (curated_home_equipment_pc_zone)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: MSI MAG Infinite S3, 10 Units
Expected source keywords: home
หมวด: equipment
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.083` sec


---

## 223. [PASS] ถูก

**คำถาม:** เก้าอี้เกมมิ่งมีไหมใน PC zone

**คำตอบ(จาก AI):**

คำตอบ:
ใน PC Zone มี Gaming Monitor 10 Units, Gaming Chair 10 Units, Gaming Keyboard, Gaming Headset, Gaming Mouse และ Gaming PC รุ่น MSI MAG Infinite S3 14th 10 Units

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (rule_equipment_pc_zone / home, curated_home_equipment_pc_zone)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Gaming Chair, 10 Units
Expected source keywords: home
หมวด: equipment
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.069` sec


---

## 224. [PASS] ถูก

**คำถาม:** เมาส์กับหูฟังมีให้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ใน PC Zone มี Gaming Monitor 10 Units, Gaming Chair 10 Units, Gaming Keyboard, Gaming Headset, Gaming Mouse และ Gaming PC รุ่น MSI MAG Infinite S3 14th 10 Units

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (rule_equipment_pc_zone / home, curated_home_equipment_pc_zone)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Gaming Mouse, Gaming Headset
Expected source keywords: home
หมวด: equipment
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.028` sec


---

## 225. [PASS] ถูก

**คำถาม:** Cockpit zone มีทีวีขนาดกี่นิ้ว

**คำตอบ(จาก AI):**

คำตอบ:
- Cockpit Zone มี TV 65" 2 Units, Logitech G923 TRUEFORCE Racing wheel + Driving Force Shifter 2 Units, Racezone Full Cockpit V3 Steering Wheel Stand with Seat Set 2 Units และ Pulse Elite Wireless Headset 2 Units

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (curated_home_equipment_cockpit_zone)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Units
Expected source keywords: home
หมวด: equipment
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.129` sec


---

## 226. [PASS] ถูก

**คำถาม:** พวงมาลัยใช้รุ่นอะไร

**คำตอบ(จาก AI):**

คำตอบ:
- Cockpit Zone มี TV 65" 2 Units, Logitech G923 TRUEFORCE Racing wheel + Driving Force Shifter 2 Units, Racezone Full Cockpit V3 Steering Wheel Stand with Seat Set 2 Units และ Pulse Elite Wireless Headset 2 Units

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (curated_home_equipment_cockpit_zone)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Units
Expected source keywords: home
หมวด: equipment
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.101` sec


---

## 227. [PASS] ถูก

**คำถาม:** Nintendo zone มีทีวีกี่นิ้ว

**คำตอบ(จาก AI):**

คำตอบ:
- Nintendo Switch Zone มี TV 86" 1 Units, Sofa 2 seats 2 Units และ Nintendo Switch (OLED model) Neon Red Neon Blue set 1 Units

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (curated_home_equipment_nintendo_zone)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Units
Expected source keywords: home
หมวด: equipment
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.118` sec


---

## 228. [PASS] ถูก

**คำถาม:** PS5 zone มีเครื่องกี่เครื่อง

**คำตอบ(จาก AI):**

คำตอบ:
- PLAYstation5 Zone มี PlayStation 5 Slim With Ultra HD Blu-Ray Disc Drive 2 Units

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (curated_home_equipment_ps5_zone)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Units
Expected source keywords: home
หมวด: equipment
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.125` sec


---

## 229. [PASS] ถูก

**คำถาม:** VR zone ใช้แว่นรุ่นอะไร

**คำตอบ(จาก AI):**

คำตอบ:
- VR Zone มี PlayStation 5 Slim With Ultra HD Blu-Ray Disc Drive 1 Units และ PlayStation VR รุ่น Sony PlayStation VR2 1 Unit

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (curated_home_equipment_vr_zone)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Units
Expected source keywords: home
หมวด: equipment
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.123` sec


---

## 230. [PASS] ถูก

**คำถาม:** เอาขนมเข้าไปกินตรงโต๊ะได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
อนุญาตให้รับประทานอาหารและเครื่องดื่มเฉพาะในพื้นที่ที่กำหนดเท่านั้น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_food_drink / Reservation, curated_rule_food_drinks)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: เฉพาะ, พื้นที่ที่กำหนด
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.052` sec


---

## 231. [PASS] ถูก

**คำถาม:** กินน้ำในพื้นที่เล่นได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
อนุญาตให้รับประทานอาหารและเครื่องดื่มเฉพาะในพื้นที่ที่กำหนดเท่านั้น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_food_drink / Reservation, curated_rule_food_drinks)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: เฉพาะ, พื้นที่ที่กำหนด
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.035` sec


---

## 232. [PASS] ถูก

**คำถาม:** ต้องฝากกระเป๋าก่อนไหม

**คำตอบ(จาก AI):**

คำตอบ: ในแหล่งข้อมูลที่ให้มีคำแนะนำเกี่ยวกับการฝากสัมภาระว่า "กรุณาฝากสัมภาระก่อนเข้าใช้บริการ" ซึ่งหมายความว่าผู้ใช้งานต้องฝากกระเป๋าหรือสัมภาระก่อนที่จะเข้าใช้บริการ

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ฝากสัมภาระ
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_llm` | model: `qwen2.5:3b` | elapsed: `4.082` sec


---

## 233. [PASS] ถูก

**คำถาม:** ใช้เสียงดังได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
กรุณางดส่งเสียงดังเกินควร และห้ามพูดจาดูหมิ่นหรือเสียดสีผู้อื่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_noise_language / Reservation, curated_rule_noise_language)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: งด, เสียงดัง
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.018` sec


---

## 234. [PASS] ถูก

**คำถาม:** พูดจาเสียดสีคนอื่นได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
กรุณางดส่งเสียงดังเกินควร และห้ามพูดจาดูหมิ่นหรือเสียดสีผู้อื่น

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_noise_language / Reservation, curated_rule_noise_language)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม, เสียดสี
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.032` sec


---

## 235. [PASS] ถูก

**คำถาม:** ทิ้งขยะไว้ในโซนเล่นได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
- ห้ามทิ้งขยะหรือสิ่งของใด ๆ ในบริเวณที่ไม่ได้กำหนด

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_rule_trash_personal_items)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม, ทิ้งขยะ
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.099` sec


---

## 236. [PASS] ถูก

**คำถาม:** สูบบุหรี่ในศูนย์ได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ศูนย์ห้ามสูบบุหรี่ เสพสารเสพติด หรือดื่มเครื่องดื่มแอลกอฮอล์ภายในศูนย์

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_smoking_alcohol / Reservation, curated_rule_smoking_alcohol_drugs)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.039` sec


---

## 237. [PASS] ถูก

**คำถาม:** เอาแอลกอฮอล์เข้าได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ศูนย์ห้ามสูบบุหรี่ เสพสารเสพติด หรือดื่มเครื่องดื่มแอลกอฮอล์ภายในศูนย์

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_smoking_alcohol / Reservation, curated_rule_smoking_alcohol_drugs)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.036` sec


---

## 238. [PASS] ถูก

**คำถาม:** พกมีดเข้าไปได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ห้ามพกอาวุธหรือของมีคม ห้ามทะเลาะวิวาท และห้ามเล่นการพนัน

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_weapons_gambling / Reservation, curated_rule_weapons_gambling)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.014` sec


---

## 239. [PASS] ถูก

**คำถาม:** เล่นพนันในห้องได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ห้ามพกอาวุธหรือของมีคม ห้ามทะเลาะวิวาท และห้ามเล่นการพนัน

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_weapons_gambling / Reservation, curated_rule_weapons_gambling)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.024` sec


---

## 240. [PASS] ถูก

**คำถาม:** เอาปลั๊กไฟส่วนตัวมาใช้ได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
- ห้ามนำอุปกรณ์อิเล็กทรอนิกส์ส่วนตัวมาใช้ปลั๊กไฟโดยไม่ได้รับอนุญาต

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_rule_power_outlet)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.103` sec


---

## 241. [PASS] ถูก

**คำถาม:** ย้ายอุปกรณ์เองได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
- ห้ามเคลื่อนย้ายอุปกรณ์หรือสิ่งของใด ๆ โดยไม่ได้รับอนุญาต

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_rule_move_equipment)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.091` sec


---

## 242. [PASS] ถูก

**คำถาม:** ทำอุปกรณ์เสียหายต้องจ่ายไหม

**คำตอบ(จาก AI):**

คำตอบ:
ถ้าทำอุปกรณ์เสียหาย ผู้ใช้งานต้องรับผิดชอบค่าปรับหรือค่าชดเชยตามระดับความเสียหาย ตั้งแต่ความเสียหายเล็กน้อย 100–500 บาท ความเสียหายปานกลางตามราคาจริงหรือ 500–2,000 บาท ไปจนถึงความเสียหายร้ายแรงที่ต้องชดเชยราคาทรัพย์สินเต็มจำนวน

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_damage_responsibility / Reservation, curated_damage_minor, curated_damage_moderate, curated_damage_severe)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: รับผิดชอบ, ค่าปรับ
Expected source keywords: Reservation
หมวด: penalty
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.001` sec


---

## 243. [PASS] ถูก

**คำถาม:** รอยขีดข่วนเล็กน้อยโดนปรับเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
ความเสียหายเล็กน้อย เช่น รอยเปื้อน คราบน้ำ รอยขีดข่วน ฝาปิดหลุด หรือปุ่มหลวม มีค่าปรับ 100–500 บาท

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_minor_damage / curated_damage_minor)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: บาท
Expected source keywords: Reservation
หมวด: penalty
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.001` sec


---

## 244. [PASS] ถูก

**คำถาม:** เบาะขาดต้องจ่ายกี่บาท

**คำตอบ(จาก AI):**

คำตอบ:
ความเสียหายปานกลาง เช่น เบาะขาด รอยขีดข่วนลึก โครงเฟอร์นิเจอร์เสียหาย คอนโทรลเลอร์ปุ่มค้าง หรือหูฟังสายขาด ต้องชำระค่าซ่อมตามราคาจริง หรือ 500–2,000 บาท

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_moderate_damage / curated_damage_moderate)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: บาท
Expected source keywords: Reservation
หมวด: penalty
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.000` sec


---

## 245. [PASS] ถูก

**คำถาม:** หูฟังสายขาดค่าปรับเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
ความเสียหายปานกลาง เช่น เบาะขาด รอยขีดข่วนลึก โครงเฟอร์นิเจอร์เสียหาย คอนโทรลเลอร์ปุ่มค้าง หรือหูฟังสายขาด ต้องชำระค่าซ่อมตามราคาจริง หรือ 500–2,000 บาท

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_moderate_damage / curated_damage_moderate)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: บาท
Expected source keywords: Reservation
หมวด: penalty
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.001` sec


---

## 246. [PASS] ถูก

**คำถาม:** จอแตกต้องชดเชยยังไง

**คำตอบ(จาก AI):**

คำตอบ:
ความเสียหายร้ายแรง เช่น จอแตก คอมพิวเตอร์พัง เฟอร์นิเจอร์เสียหายจนใช้ไม่ได้ หรืออุปกรณ์ใช้งานไม่ได้ ต้องชดเชยราคาทรัพย์สินเต็มจำนวนตามราคากลาง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_severe_damage / Reservation, curated_damage_severe)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ชดเชย, เต็มจำนวน
Expected source keywords: Reservation
หมวด: penalty
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.000` sec


---

## 247. [PASS] ถูก

**คำถาม:** คอมพังต้องจ่ายเต็มไหม

**คำตอบ(จาก AI):**

คำตอบ:
ความเสียหายร้ายแรง เช่น จอแตก คอมพิวเตอร์พัง เฟอร์นิเจอร์เสียหายจนใช้ไม่ได้ หรืออุปกรณ์ใช้งานไม่ได้ ต้องชดเชยราคาทรัพย์สินเต็มจำนวนตามราคากลาง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_severe_damage / Reservation, curated_damage_severe)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ชดเชย, เต็มจำนวน
Expected source keywords: Reservation
หมวด: penalty
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.001` sec


---

## 248. [PASS] ถูก

**คำถาม:** ละเมิดกฎจะโดนระงับสิทธิ์กี่วัน

**คำตอบ(จาก AI):**

คำตอบ:
หากผู้ใช้งานละเมิดกฎซ้ำหรือกระทำการรุนแรง อาจถูกระงับสิทธิ์การใช้งานชั่วคราวเป็นระยะเวลา 1-7 วัน ขึ้นอยู่กับลักษณะของการละเมิด

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_penalty_temp_suspension / Reservation, curated_penalty_temp_suspension, reservation-penalty-001)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ระงับสิทธิ์
Expected source keywords: Reservation
หมวด: penalty
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.044` sec


---

## 249. [PASS] ถูก

**คำถาม:** โดนแบนชั่วคราวกี่วัน

**คำตอบ(จาก AI):**

คำตอบ:
หากผู้ใช้งานละเมิดกฎซ้ำหรือกระทำการรุนแรง อาจถูกระงับสิทธิ์การใช้งานชั่วคราวเป็นระยะเวลา 1-7 วัน ขึ้นอยู่กับลักษณะของการละเมิด

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_penalty_temp_suspension / Reservation, curated_penalty_temp_suspension, reservation-penalty-001)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ระงับสิทธิ์, 1-7 วัน
Expected source keywords: Reservation
หมวด: penalty
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.027` sec


---

## 250. [PASS] ถูก

**คำถาม:** กรณีไหนแบนถาวร

**คำตอบ(จาก AI):**

คำตอบ: การระงับสิทธิ์ถาวรเกิดขึ้นเมื่อผู้ใช้งานกระทำการละเมิดกฎอย่างร้ายแรง เช่น การทำลายทรัพย์สินหรือพฤติกรรมที่เป็นภัยต่อความปลอดภัยของผู้อื่น ซึ่งสามารถดูได้จากข้อมูลใน [SOURCE 1]

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ระงับสิทธิ์, ถาวร
Expected source keywords: Reservation
หมวด: penalty
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_llm` | model: `qwen2.5:3b` | elapsed: `4.243` sec


---

## 251. [PASS] ถูก

**คำถาม:** อุทธรณ์การลงโทษได้ภายในกี่วัน

**คำตอบ(จาก AI):**

คำตอบ:
- หากผู้ใช้งานละเมิดกฎซ้ำหรือกระทำการรุนแรง อาจถูกระงับสิทธิ์การใช้งานเป็นระยะเวลา 1-7 วัน ขึ้นอยู่กับลักษณะของการละเมิด

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (curated_penalty_temp_suspension)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 7 วัน
Expected source keywords: Reservation
หมวด: penalty
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.111` sec


---

## 252. [PASS] ถูก

**คำถาม:** ศูนย์เก็บประวัติคนทำผิดไหม

**คำตอบ(จาก AI):**

คำตอบ: ศูนย์จะบันทึกประวัติการละเมิดผ่านการใช้บริการ และอาจส่งผลต่อสิทธิ์ในการใช้บริการในอนาคต. ดังนี้คือข้อมูลที่เกี่ยวข้อง: [SOURCE 1]

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: บันทึก, ประวัติ
Expected source keywords: Reservation
หมวด: penalty
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rag_llm` | model: `qwen2.5:3b` | elapsed: `4.104` sec


---

## 253. [PASS] ถูก

**คำถาม:** ศูนย์นี้คืออะไรแบบสั้นๆ

**คำตอบ(จาก AI):**

คำตอบ:
PSU Esports Studio - Phuket คือศูนย์พัฒนาการเรียนรู้ด้านอีสปอร์ตเพื่อความเป็นเลิศและขับเคลื่อนเศรษฐกิจในพื้นที่ภาคใต้ สาขาภูเก็ต ของมหาวิทยาลัยสงขลานครินทร์ และดำเนินการโดยวิทยาลัยการคอมพิวเตอร์

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (rule_overview_identity / curated_overview_identity)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: มหาวิทยาลัยสงขลานครินทร์, วิทยาลัยการคอมพิวเตอร์
Expected source keywords: home
หมวด: overview
ชนิดคำตอบ: summary

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.027` sec


---

## 254. [PASS] ถูก

**คำถาม:** PSU Esports Studio Phuket คืออะไร

**คำตอบ(จาก AI):**

คำตอบ:
PSU Esports Studio - Phuket คือศูนย์พัฒนาการเรียนรู้ด้านอีสปอร์ตเพื่อความเป็นเลิศและขับเคลื่อนเศรษฐกิจในพื้นที่ภาคใต้ สาขาภูเก็ต ของมหาวิทยาลัยสงขลานครินทร์ และดำเนินการโดยวิทยาลัยการคอมพิวเตอร์

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (rule_overview_identity / curated_overview_identity)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: มหาวิทยาลัยสงขลานครินทร์, วิทยาลัยการคอมพิวเตอร์
Expected source keywords: home
หมวด: overview
ชนิดคำตอบ: summary

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.068` sec


---

## 255. [PASS] ถูก

**คำถาม:** ใครเป็นคนก่อตั้งศูนย์นี้

**คำตอบ(จาก AI):**

คำตอบ:
ศูนย์ก่อตั้งโดยมหาวิทยาลัยสงขลานครินทร์ และดำเนินการโดยวิทยาลัยการคอมพิวเตอร์

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (rule_overview_founder / home, curated_overview_mission, curated_overview_identity)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: มหาวิทยาลัยสงขลานครินทร์, วิทยาลัยการคอมพิวเตอร์
Expected source keywords: home
หมวด: overview
ชนิดคำตอบ: summary

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.033` sec


---

## 256. [PASS] ถูก

**คำถาม:** หน่วยงานที่ดำเนินการคือใคร

**คำตอบ(จาก AI):**

คำตอบ:
ศูนย์ก่อตั้งโดยมหาวิทยาลัยสงขลานครินทร์ และดำเนินการโดยวิทยาลัยการคอมพิวเตอร์

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (rule_overview_founder / home, curated_overview_mission, curated_overview_identity)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: มหาวิทยาลัยสงขลานครินทร์, วิทยาลัยการคอมพิวเตอร์
Expected source keywords: home
หมวด: overview
ชนิดคำตอบ: summary

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.032` sec


---

## 257. [PASS] ถูก

**คำถาม:** mission ของศูนย์คืออะไร

**คำตอบ(จาก AI):**

คำตอบ:
PSU Esports Studio - Phuket คือศูนย์พัฒนาการเรียนรู้ด้านอีสปอร์ตเพื่อความเป็นเลิศและขับเคลื่อนเศรษฐกิจในพื้นที่ภาคใต้ สาขาภูเก็ต ของมหาวิทยาลัยสงขลานครินทร์ และดำเนินการโดยวิทยาลัยการคอมพิวเตอร์

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (rule_overview_identity / curated_overview_identity)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: มหาวิทยาลัยสงขลานครินทร์, วิทยาลัยการคอมพิวเตอร์
Expected source keywords: home
หมวด: overview
ชนิดคำตอบ: summary

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['home']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.030` sec


---

## 258. [PASS] ถูก

**คำถาม:** ศูนย์อยู่ตรงไหน

**คำตอบ(จาก AI):**

คำตอบ: ศูนย์ PSU Esports Studio - Phuket ตั้งอยู่ที่มหาวิทยาลัยสงขลานครินทร์ วิทยาเขตภูเก็ต 80 หมู่ 1 ถ.วิชิตสงคราม อ.กะทู้ จ.ภูเก็ต 83120

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: Contact
หมวด: contact
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Contact']`
- route: `rag_llm` | model: `qwen2.5:3b` | elapsed: `4.335` sec


---

## 259. [PASS] ถูก

**คำถาม:** ที่ตั้งของ studio คือที่ไหน

**คำตอบ(จาก AI):**

คำตอบ:
PSU Esports Studio - Phuket ตั้งอยู่ที่มหาวิทยาลัยสงขลานครินทร์ วิทยาเขตภูเก็ต 80 หมู่ 1 ถ.วิชิตสงคราม อ.กะทู้ จ.ภูเก็ต 83120

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/Contact-Us (rule_contact_location / curated_contact_location)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: Contact
หมวด: contact
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Contact']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.053` sec


---

## 260. [PASS] ถูก

**คำถาม:** ขอ email ติดต่อ

**คำตอบ(จาก AI):**

คำตอบ:
อีเมลติดต่อศูนย์คือ psuesportspkt@gmail.com

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/Contact-Us (rule_contact_email / curated_contact_email)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: Contact
หมวด: contact
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Contact']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.016` sec


---

## 261. [PASS] ถูก

**คำถาม:** Facebook ศูนย์ชื่ออะไร

**คำตอบ(จาก AI):**

คำตอบ:
Facebook ของศูนย์คือ https://www.facebook.com/psuesportsphuket

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/Contact-Us (rule_contact_facebook / curated_contact_facebook)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: Contact
หมวด: contact
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Contact']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.034` sec


---

## 262. [PASS] ถูก

**คำถาม:** เบอร์โทรระบบจองมีเบอร์อะไร

**คำตอบ(จาก AI):**

คำตอบ:
เบอร์ติดต่อที่ปรากฏในระบบจองคือ +66 7627 6004 และ +66 7627 6045

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_contact_phone / curated_contact_phone)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: Contact
หมวด: contact
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Contact']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.039` sec


---

## 263. [PASS] ถูก

**คำถาม:** อีสปอร์ตคืออะไรแบบเข้าใจง่าย

**คำตอบ(จาก AI):**

คำตอบ:
- อีสปอร์ต (Esports) หรือกีฬาอิเล็กทรอนิกส์ (Electronic Sports) เป็นการแข่งขันกีฬาที่ใช้ทักษะและความสามารถในการเล่นวิดีโอเกมในรูปแบบต่าง ๆ

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/Knowledge (curated_knowledge_esports_definition)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: อีสปอร์ต
Expected source keywords: Knowledge
หมวด: knowledge
ชนิดคำตอบ: summary

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Knowledge']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.107` sec


---

## 264. [PASS] ถูก

**คำถาม:** esports เริ่มครั้งแรกที่ไหน

**คำตอบ(จาก AI):**

คำตอบ:
Esports เริ่มขึ้นครั้งแรกที่มหาวิทยาลัยสแตนฟอร์ด (Stanford University) ในวันที่ 19 ตุลาคม ค.ศ. 1972 โดยการแข่งขันเกม Spacewar ภายใต้ชื่องาน Intergalactic Spacewar Olympics

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Stanford, 1972
Expected source keywords: Knowledge
หมวด: knowledge
ชนิดคำตอบ: summary

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Knowledge']`
- route: `rag_llm` | model: `qwen2.5:3b` | elapsed: `4.305` sec


---

## 265. [PASS] ถูก

**คำถาม:** Spacewar เกี่ยวกับประวัติอีสปอร์ตยังไง

**คำตอบ(จาก AI):**

คำตอบ:
- อีสปอร์ตเกิดขึ้นครั้งแรกในวันที่ 19 ตุลาคม ค.ศ. 1972 ณ มหาวิทยาลัยสแตนฟอร์ด (Standford University) จากการแข่งขันเกม Spacewar ภายใต้ชื่องาน Intergalactic Spacewar Olympics

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/Knowledge (curated_knowledge_esports_origin)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: Spacewar, 1972
Expected source keywords: Knowledge
หมวด: knowledge
ชนิดคำตอบ: summary

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Knowledge']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.128` sec


---

## 266. [PASS] ถูก

**คำถาม:** เกมประเภท MOBA คืออะไร

**คำตอบ(จาก AI):**

คำตอบ:
- ประเภทเกมที่นิยมในการแข่งขันอีสปอร์ต ได้แก่ Multiplayer Online Battle Arena (MOBA), First-Person Shooter (FPS), Battle Royale, Fighting Games, Sports Games, Racing Games, Digital Card Games และ Real-Time Strategy (RTS)

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/Knowledge (curated_knowledge_esports_categories)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: MOBA
Expected source keywords: Knowledge
หมวด: knowledge
ชนิดคำตอบ: summary

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Knowledge']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.090` sec


---

## 267. [PASS] ถูก

**คำถาม:** อาชีพในวงการ esports มีอะไรบ้าง

**คำตอบ(จาก AI):**

คำตอบ:
- อีสปอร์ต (Esports) หรือกีฬาอิเล็กทรอนิกส์ (Electronic Sports) เป็นการแข่งขันกีฬาที่ใช้ทักษะและความสามารถในการเล่นวิดีโอเกมในรูปแบบต่าง ๆ
- อาชีพที่เกี่ยวข้องกับกีฬาอีสปอร์ต ได้แก่ นักกีฬาอีสปอร์ต, โค้ชอีสปอร์ต, ผู้จัดการทีมอีสปอร์ต, นักพากย์การแข่งขันอีสปอร์ต, ผู้จัดการแข่งขันอีสปอร์ต, นักวิเคราะห์อีสปอร์ต, นักเขียนข่าวอีสปอร์ต, นักจิตวิทยาอีสปอร์ต, นักโภชนาการอีสปอร์ต และนักกายภาพบำบัดอีสปอร์ต

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/Knowledge (curated_knowledge_esports_definition)
- https://esports.phuket.psu.ac.th/Knowledge (curated_knowledge_esports_careers)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: อีสปอร์ต
Expected source keywords: Knowledge
หมวด: knowledge
ชนิดคำตอบ: summary

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Knowledge']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.109` sec


---

## 268. [PASS] ถูก

**คำถาม:** Overcooked 2 ฝึกทักษะอะไร

**คำตอบ(จาก AI):**

คำตอบ:
Overcooked! 2 ฝึกทักษะการทำงานเป็นทีม, การจัดการสถานการณ์กดดัน, การสื่อสารอย่างชัดเจน, และการวางแผน.

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: การทำงานเป็นทีม, สื่อสาร
Expected source keywords: Knowledge
หมวด: knowledge
ชนิดคำตอบ: summary

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Knowledge']`
- route: `rag_llm` | model: `qwen2.5:3b` | elapsed: `3.623` sec


---

## 269. [PASS] ถูก

**คำถาม:** Mario Kart 8 Deluxe ฝึกอะไร

**คำตอบ(จาก AI):**

Mario Kart 8 Deluxe ฝึกไหวพริบและการตัดสินใจในการแข่งขัน. ในเกมนี้ผู้เล่นต้องควบคุมตัวละครและรถบนสนามที่เต็มไปด้วยทางโค้ง, สิ่งกีดขวาง และไอเทมหลากหลายรูปแบบ ทำให้จำเป็นต้องใช้วาทีและการคิดอย่างรวดเร็ว. นอกจากนี้ยังฝึกทักษะการประเมินสถานการณ์แบบเรียล

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไหวพริบ, การตัดสินใจ
Expected source keywords: Knowledge
หมวด: knowledge
ชนิดคำตอบ: summary

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Knowledge']`
- route: `rag_llm` | model: `qwen2.5:3b` | elapsed: `4.936` sec


---

## 270. [PASS] ถูก

**คำถาม:** วันที่ 25 เมษายน 2569 แข่งเกมอะไร

**คำตอบ(จาก AI):**

คำตอบ:
- เมื่อวันที่ 25 เมษายน 2569 PSU Esports Studio - Phuket จัดการแข่งขัน PSU Phuket CS 2 2026 Tournament ในเกม Counter-Strike 2

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/events-news/news (curated_news_cs2_2026)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: News
หมวด: events_news
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['News']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.125` sec


---

## 271. [PASS] ถูก

**คำถาม:** VALORANT 2026 จัดวันไหน

**คำตอบ(จาก AI):**

คำตอบ:
- PSU Phuket VALORANT 2026 Tournament จัดขึ้นเมื่อวันที่ 21 กุมภาพันธ์ 2569 โดยเป็นการแข่งขันเกม VALORANT ณ PSU Esports Studio - Phuket

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/events-news/news (curated_news_valorant_2026)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: News
หมวด: events_news
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['News']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.112` sec


---

## 272. [PASS] ถูก

**คำถาม:** SURAT SMASH ส่งตัวแทนกี่คน

**คำตอบ(จาก AI):**

คำตอบ:
- ตัวแทน PSU Esports Studio - Phuket จำนวน 4 คน เข้าร่วมการแข่งขันเกม Tekken 8 ในรายการ SURAT SMASH TEKKEN 8 ONLINE TOURNAMENT Season 3

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/events-news/news (curated_news_surat_smash_tekken8)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: News
หมวด: events_news
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['News']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.103` sec


---

## 273. [PASS] ถูก

**คำถาม:** นักศึกษาชาวจีนมีกี่คน

**คำตอบ(จาก AI):**

คำตอบ:
- เมื่อวันที่ 27 กุมภาพันธ์ 2569 นักศึกษาชาวจีนระดับปริญญาโท คณะการบริการและการท่องเที่ยว จำนวน 11 คน เข้าร่วมกิจกรรมการเรียนรู้โดยใช้เกมเป็นฐาน (Game-based Learning) ณ PSU Esports Studio - Phuket

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/events-news/news (curated_news_chinese_students_game_based_learning)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: News
หมวด: events_news
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['News']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.091` sec


---

## 274. [PASS] ถูก

**คำถาม:** GAME ON เปิดโลกอีสปอร์ตจัดให้ใคร

**คำตอบ(จาก AI):**

คำตอบ:
- กิจกรรม GAME ON! เปิดโลกอีสปอร์ต จัดสำหรับนักเรียนชั้นมัธยมศึกษาปีที่ 3 (ม.3) จากโรงเรียนท้ายเหมืองวิทยา จำนวน 21 คน

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/events-news/news (curated_news_game_on_thaimuang)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: News
หมวด: events_news
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['News']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.107` sec


---

## 275. [PASS] ถูก

**คำถาม:** อธิการบดีในหน้าสมาชิกคือใคร

**คำตอบ(จาก AI):**

คำตอบ:
- หน้าสมาชิกระบุว่า ผศ.ดร.นิวัติ แก้วประดับ เป็นอธิการบดี มหาวิทยาลัยสงขลานครินทร์

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/about-us/Members (curated_about_members_president)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: Members
หมวด: about_us
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Members']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.091` sec


---

## 276. [PASS] ถูก

**คำถาม:** คณบดีวิทยาลัยการคอมพิวเตอร์คือใคร

**คำตอบ(จาก AI):**

คำตอบ:
- หน้าสมาชิกระบุว่า รศ.ดร.อซีส นันทอมรพงศ์ เป็นคณบดี วิทยาลัยการคอมพิวเตอร์

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/about-us/Members (curated_about_members_dean)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: Members
หมวด: about_us
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Members']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.126` sec


---

## 277. [PASS] ถูก

**คำถาม:** ผู้จัดการศูนย์คือใคร

**คำตอบ(จาก AI):**

คำตอบ:
- หน้าสมาชิกระบุว่า นายชนะชัย สิริพันธ์วราภรณ์ เป็นผู้จัดการ PSU Esports Studio - Phuket

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/about-us/Members (curated_about_members_manager)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: Members
หมวด: about_us
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Members']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.090` sec


---

## 278. [PASS] ถูก

**คำถาม:** ประธาน PSU Phuket Esports Club คือใคร

**คำตอบ(จาก AI):**

คำตอบ:
- หน้าสมาชิกระบุว่า นายษุภากรณ์ จิราจินดากุล เป็นประธาน PSU Phuket Esports Club - PSU Phuket

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/about-us/Members (curated_about_members_club_president)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: Members
หมวด: about_us
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Members']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.133` sec


---

## 279. [PASS] ถูก

**คำถาม:** Gallery มีหมวดภาพอะไร

**คำตอบ(จาก AI):**

คำตอบ:
- หน้า Gallery มีหมวดภาพ Nintendo Switch และ PlayStation 5

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/about-us/Gallery (curated_about_gallery_categories)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: PSU
Expected source keywords: Members
หมวด: about_us
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Members']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.087` sec


---

## 280. [PASS] ถูก

**คำถาม:** มีบริการซ่อมคอมส่วนตัวไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลว่าศูนย์มีบริการซ่อมคอมพิวเตอร์ส่วนตัวในฐานข้อมูลที่มี ข้อมูลที่มีตอนนี้เน้นการจองใช้อุปกรณ์ภายในศูนย์และกฎการใช้งาน หากต้องการยืนยันเรื่องซ่อมอุปกรณ์ส่วนตัวควรติดต่อเจ้าหน้าที่ศูนย์โดยตรง

แหล่งข้อมูล:
- ฐานข้อมูลที่มี (rule_no_answer_repair_personal_pc)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.000` sec


---

## 281. [PASS] ถูก

**คำถาม:** ส่งอาหารถึงโต๊ะเกมได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลว่ามีบริการจัดส่งอาหารถึงโต๊ะเล่นเกมในฐานข้อมูลที่มี และตามกฎที่มี อาหารและเครื่องดื่มควรรับประทานเฉพาะในพื้นที่ที่กำหนดเท่านั้น จึงไม่ควรสรุปว่าสามารถส่งหรือทานที่โต๊ะเล่นเกมได้

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_food_delivery / Reservation, curated_rule_food_drinks)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.037` sec


---

## 282. [PASS] ถูก

**คำถาม:** เอาแมวเข้าได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องการนำสัตว์เลี้ยงเข้าศูนย์ในฐานข้อมูลที่มี จึงยังไม่ควรตอบว่าอนุญาต หากจำเป็นต้องนำสัตว์เลี้ยงมาด้วยควรติดต่อเจ้าหน้าที่ศูนย์ก่อนเข้าใช้บริการ

แหล่งข้อมูล:
- ฐานข้อมูลที่มี (rule_no_answer_pets)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.027` sec


---

## 283. [PASS] ถูก

**คำถาม:** สมัครสมาชิกรายปีราคาเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องค่าสมัครสมาชิกศูนย์รายปีในฐานข้อมูลที่มี ข้อมูลราคาที่มีตอนนี้เป็นค่าบริการตามประเภทอุปกรณ์และกลุ่มผู้ใช้ เช่น PSU Student and Staff, PSU Alumni and General Student และ General Adult

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (rule_no_answer_annual_membership / service_fee)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.050` sec


---

## 284. [PASS] ถูก

**คำถาม:** เช่าโน้ตบุ๊กกลับบ้านได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องบริการเช่าโน้ตบุ๊กกลับบ้านในฐานข้อมูลที่มี ข้อมูลที่มีระบุการจองใช้อุปกรณ์ภายในศูนย์เป็นรอบเวลา จึงยังไม่ควรสรุปว่าสามารถนำอุปกรณ์กลับไปใช้นอกสถานที่ได้

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_laptop_rental_home / Reservation)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.034` sec


---

## 285. [PASS] ถูก

**คำถาม:** มีห้องนอนพักค้างคืนไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องห้องนอนหรือบริการพักค้างคืนในฐานข้อมูลที่มี ข้อมูลเวลาที่มีระบุการใช้งานเป็นรอบเวลาในช่วง 09:00-16:00 และมีช่วง Maintenance ตามตาราง จึงไม่ควรสรุปว่ามีบริการพักค้างคืน

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_overnight_room / Reservation, curated_schedule_weekly)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.029` sec


---

## 286. [PASS] ถูก

**คำถาม:** ขายคีย์บอร์ดเกมมิ่งไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลว่าศูนย์มีบริการขายคีย์บอร์ดเกมมิ่งในฐานข้อมูลที่มี ข้อมูลอุปกรณ์ที่มีเป็นรายการอุปกรณ์สำหรับใช้งานภายในศูนย์ เช่น Gaming Keyboard, Gaming Mouse, Gaming Headset และ Gaming PC

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (rule_no_answer_keyboard_sales / home, curated_home_equipment_pc_zone)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.028` sec


---

## 287. [PASS] ถูก

**คำถาม:** รับซ่อมจอย PS5 ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลว่าศูนย์รับซ่อมจอย PS5 หรือคอนโทรลเลอร์ส่วนตัวในฐานข้อมูลที่มี ข้อมูลที่มีเป็นการจองใช้อุปกรณ์และกฎการรับผิดชอบเมื่อทำอุปกรณ์ของศูนย์เสียหาย หากต้องการซ่อมอุปกรณ์ส่วนตัวควรสอบถามเจ้าหน้าที่โดยตรง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_controller_repair / Reservation, curated_damage_moderate, curated_damage_severe)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.000` sec


---

## 288. [PASS] ถูก

**คำถาม:** มีบริการส่งเครื่องเกมไปบ้านไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องบริการส่งหรือให้เช่าเครื่องเกม/จอกลับไปใช้ที่บ้านในฐานข้อมูลที่มี ข้อมูลที่มีเป็นการจองใช้อุปกรณ์ภายใน PSU Esports Studio ตามรอบเวลา ดังนั้นควรจองเพื่อใช้งานที่ศูนย์ หรือสอบถามเจ้าหน้าที่หากต้องการบริการนอกสถานที่

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_home_delivery_console / Reservation)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.044` sec


---

## 289. [PASS] ถูก

**คำถาม:** ซื้อเกม Steam ผ่านศูนย์ได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลว่าศูนย์มีบริการซื้อเกม Steam ผ่านศูนย์ในฐานข้อมูลที่มี ข้อมูลที่มีตอนนี้เป็นรายการเกมและบริการใช้อุปกรณ์ภายในศูนย์ ไม่ใช่บริการจำหน่ายเกม

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_steam_purchase / Reservation)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.051` sec


---

## 290. [PASS] ถูก

**คำถาม:** มีคอร์สสอนเล่น Valorant ส่วนตัวไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องคอร์สสอนเล่นเกมแบบส่วนตัวในฐานข้อมูลที่มี ข้อมูลที่มีระบุรายการเกมและกิจกรรม/การแข่งขันของศูนย์ แต่ยังไม่มีรายละเอียดบริการสอนส่วนตัว หากสนใจควรถามเจ้าหน้าที่หรือเพจของศูนย์อีกครั้ง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_private_course / Reservation, Events_and_News)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.070` sec


---

## 291. [PASS] ถูก

**คำถาม:** จ่ายด้วยคริปโตได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลว่าศูนย์รับชำระด้วยคริปโตในฐานข้อมูลที่มี ข้อมูลการชำระเงินที่ยืนยันได้คือชำระโดยโอนเข้าบัญชีธนาคารและแนบสลิปการโอนเงินในระบบจอง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_crypto_payment / Reservation, curated_payment_bank)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.025` sec


---

## 292. [PASS] ถูก

**คำถาม:** ผ่อนชำระค่าเล่นได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องการผ่อนชำระค่าใช้บริการในฐานข้อมูลที่มี ข้อมูลที่มีระบุว่าหลังจองต้องชำระเงินทันที และหากไม่ชำระภายใน 10 นาที การจองจะถูกยกเลิก จึงควรยึดตามการชำระเงินตามรอบจองก่อน

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_installment / Reservation, curated_payment_10_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.028` sec


---

## 293. [PASS] ถูก

**คำถาม:** มีส่วนลดวันเกิดไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องส่วนลดวันเกิดในฐานข้อมูลที่มี ข้อมูลราคาที่มีตอนนี้เป็นตารางค่าบริการตามประเภทอุปกรณ์และกลุ่มผู้ใช้ หากมีโปรโมชันพิเศษควรตรวจสอบกับเพจหรือเจ้าหน้าที่ศูนย์ก่อนจอง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (rule_no_answer_birthday_discount / service_fee)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.026` sec


---

## 294. [PASS] ถูก

**คำถาม:** จองแบบเหมาทั้งวันได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องการจองแบบเหมาทั้งวันในฐานข้อมูลที่มี ข้อมูลการจองที่มีระบุว่าการจอง 1 ครั้งสามารถจองได้สูงสุด 3 Sessions และต้องจองล่วงหน้าอย่างน้อย 1 ชั่วโมง จึงควรใช้เงื่อนไขนี้ก่อนหากไม่มีประกาศอื่นเพิ่มเติม

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_full_day_booking / Reservation, curated_reservation_max_sessions, curated_reservation_advance_time)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.035` sec


---

## 295. [PASS] ถูก

**คำถาม:** เอาเครื่อง PC ตัวเองมาตั้งได้ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลว่าสามารถนำ PC หรือคอมพิวเตอร์ส่วนตัวมาตั้งใช้งานในศูนย์ได้ในฐานข้อมูลที่มี ข้อมูลที่มีระบุบริการและอุปกรณ์ของศูนย์สำหรับจองใช้งานภายในสถานที่ จึงควรสอบถามเจ้าหน้าที่ก่อนนำอุปกรณ์ส่วนตัวเข้ามา

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (rule_no_answer_own_pc_setup / home, Reservation, curated_home_equipment_pc_zone)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.063` sec


---

## 296. [PASS] ถูก

**คำถาม:** มีบริการถ่ายรูปโปรไฟล์เกมเมอร์ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องบริการถ่ายรูปโปรไฟล์เกมเมอร์ในฐานข้อมูลที่มี ข้อมูลที่มีครอบคลุมบริการใช้อุปกรณ์ พื้นที่เล่นเกม และกิจกรรมของศูนย์ แต่ยังไม่พบรายละเอียดบริการถ่ายภาพ

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (rule_no_answer_photo_profile / Services, Events_and_News)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.053` sec


---

## 297. [PASS] ถูก

**คำถาม:** มีอาหารบุฟเฟต์ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลว่าศูนย์มีบริการอาหารบุฟเฟต์ในฐานข้อมูลที่มี และกฎที่มีระบุว่าอาหารและเครื่องดื่มควรรับประทานเฉพาะในพื้นที่ที่กำหนดเท่านั้น จึงไม่ควรสรุปว่ามีบริการบุฟเฟต์ภายในศูนย์

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_food_buffet / Reservation, curated_rule_food_drinks)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.018` sec


---

## 298. [PASS] ถูก

**คำถาม:** รับจัดงานแต่งในศูนย์ไหม

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลว่าศูนย์รับจัดงานแต่งในฐานข้อมูลที่มี ข้อมูลที่มีเกี่ยวกับบริการศูนย์อีสปอร์ต การจองใช้อุปกรณ์ กิจกรรม และการแข่งขัน หากเป็นงานพิเศษนอกเหนือจากนี้ควรติดต่อเจ้าหน้าที่ศูนย์โดยตรง

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (rule_no_answer_wedding_event / Services, Events_and_News, Reservation)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.031` sec


---

## 299. [PASS] ถูก

**คำถาม:** ถ้าจองแล้วไม่จ่ายใน 10 นาที แล้วไปเช็คอินช้าจะเกิดอะไรบ้าง

**คำตอบ(จาก AI):**

คำตอบ:
ถ้าหลังจองไม่ชำระเงินภายใน 10 นาที การจองจะถูกยกเลิกและต้องจองใหม่ หากมีการจองสำเร็จแล้วแต่เช็คอินไม่ทันก่อนเริ่มรอบ ระบบจะยกเลิกการจองทันทีและไม่มีการคืนเงิน

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_payment_and_late_checkin / Reservation, curated_payment_10_minutes, curated_checkin_late_cancel)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 10 นาที, ยกเลิก, ไม่มีการคืนเงิน
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: multi_fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.086` sec


---

## 300. [PASS] ถูก

**คำถาม:** ช่วยบอกทั้งจองล่วงหน้า เช็คอิน และยกเลิกแบบสั้นๆ

**คำตอบ(จาก AI):**

คำตอบ:
สรุปสั้นๆ: ต้องจองล่วงหน้าอย่างน้อย 1 ชั่วโมง, เช็คอินได้ล่วงหน้าสูงสุด 30 นาทีและต้องเช็คอินก่อนเริ่มรอบ, หากต้องยกเลิกหรือแก้ไขการจองควรดำเนินการล่วงหน้าอย่างน้อย 1 ชั่วโมง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_reservation_quick_summary / Reservation, curated_reservation_advance_time, curated_checkin_30_minutes, curated_cancel_1_hour)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 1 ชั่วโมง, 30 นาที, ยกเลิก
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: multi_fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.077` sec


---

## 301. [PASS] ถูก

**คำถาม:** จอง 1 ครั้งได้กี่ session แล้วต้องจ่ายภายในกี่นาที

**คำตอบ(จาก AI):**

คำตอบ:
การจอง 1 ครั้งจองได้สูงสุด 3 Sessions และหลังจองต้องชำระเงินทันที หากไม่ชำระภายใน 10 นาที การจองจะถูกยกเลิกและต้องจองใหม่

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_booking_session_and_payment / Reservation, curated_reservation_max_sessions, curated_payment_10_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 3 Sessions, 10 นาที
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: multi_fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.087` sec


---

## 302. [PASS] ถูก

**คำถาม:** กรอกข้อมูลอะไรบ้าง แล้วต้องแนบสลิปไหม

**คำตอบ(จาก AI):**

คำตอบ:
ตอนจองต้องกรอก Student ID/Staff ID/National ID, ชื่อ, นามสกุล, อีเมล, เบอร์โทรศัพท์ และคอมเมนต์ถ้ามี จากนั้นชำระเงินโดยโอนเข้าบัญชีธนาคารและแนบสลิปการโอนเงิน

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_user_info_and_slip / Reservation, curated_user_info_required, curated_booking_steps, curated_payment_bank)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: สลิป, โอนเงิน
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: multi_fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.060` sec


---

## 303. [PASS] ถูก

**คำถาม:** ถ้าจองผิดเวลา ต้องแก้ยังไงและต้องแจ้งก่อนกี่ชั่วโมง

**คำตอบ(จาก AI):**

คำตอบ:
ต้องจองล่วงหน้าผ่านระบบออนไลน์ก่อนเวลาใช้งานอย่างน้อย 1 ชั่วโมง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_booking_advance / Reservation, curated_reservation_advance_time)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 1 ชั่วโมง
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: multi_fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.072` sec


---

## 304. [PASS] ถูก

**คำถาม:** นักเรียน มอ เล่น PS5 แล้วต่อด้วย VR 30 นาที ค่าใช้จ่ายเป็นยังไง

**คำตอบ(จาก AI):**

คำตอบ:
- ตารางค่าบริการ PSU Esports Studio - Phuket Service Fee 2026 ตามภาพ: PlayStation 5 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 150 บาท. Nintendo Switch 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 140 บาท. Nintendo Switch 1 ชั่วโมง 3-4 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 100 บาท, General Adult 280 บาท. Cockpit 1 ชั่วโมง 1 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 65 บาท, General Adult 200 บาท. VR 30 นาที 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 190 บาท, General Adult 525 บาท. VR 1 ชั่วโมง 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 375 บาท, General Adult 1050 บาท. ภาพนี้ระบุว่าค่าบริการมีผลตั้งแต่ 1 February 2026 ถึง 31 March 2026 และไม่ปรากฏราคา PC

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (curated_service_fee_2026_full_table)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.140` sec


---

## 305. [PASS] ถูก

**คำถาม:** คนนอกเล่น Switch 3-4 คนกับ Cockpit ราคาอะไรแพงกว่า

**คำตอบ(จาก AI):**

คำตอบ:
- ค่าบริการ Nintendo Switch 3-4 คน ต่อ 60 นาที: นักศึกษา/บุคลากร PSU 0 บาท, ศิษย์เก่า PSU หรือ General Student 100 บาท, บุคคลทั่วไป 280 บาท
- ค่าบริการ Nintendo Switch 1-2 คน ต่อ 60 นาที: นักศึกษา/บุคลากร PSU 0 บาท, ศิษย์เก่า PSU หรือ General Student 50 บาท, บุคคลทั่วไป 140 บาท
- ตารางค่าบริการ PSU Esports Studio - Phuket Service Fee 2026 ตามภาพ: PlayStation 5 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 150 บาท. Nintendo Switch 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 140 บาท. Nintendo Switch 1 ชั่วโมง 3-4 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 100 บาท, General Adult 280 บาท. Cockpit 1 ชั่วโมง 1 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 65 บาท, General Adult 200 บาท. VR 30 นาที 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 190 บาท, General Adult 525 บาท. VR 1 ชั่วโมง 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 375 บาท, General Adult 1050 บาท. ภาพนี้ระบุว่าค่าบริการมีผลตั้งแต่ 1 February 2026 ถึง 31 March 2026 และไม่ปรากฏราคา PC
- ค่าบริการ Cockpit ต่อ 60 นาที: นักศึกษา/บุคลากร PSU 0 บาท, ศิษย์เก่า PSU หรือ General Student 65 บาท, บุคคลทั่วไป 200 บาท

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (curated_service_fee_nintendo_3_4_rates)
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (curated_service
... [trimmed output]

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.105` sec


---

## 306. [PASS] ถูก

**คำถาม:** General Student เล่น PS5 กับ Nintendo 1-2 ราคาเท่ากันไหม

**คำตอบ(จาก AI):**

คำตอบ:
- ค่าบริการ Nintendo Switch 1-2 คน ต่อ 60 นาที: นักศึกษา/บุคลากร PSU 0 บาท, ศิษย์เก่า PSU หรือ General Student 50 บาท, บุคคลทั่วไป 140 บาท
- ตารางค่าบริการ PSU Esports Studio - Phuket Service Fee 2026 ตามภาพ: PlayStation 5 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 150 บาท. Nintendo Switch 1 ชั่วโมง 1-2 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 50 บาท, General Adult 140 บาท. Nintendo Switch 1 ชั่วโมง 3-4 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 100 บาท, General Adult 280 บาท. Cockpit 1 ชั่วโมง 1 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 65 บาท, General Adult 200 บาท. VR 30 นาที 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 190 บาท, General Adult 525 บาท. VR 1 ชั่วโมง 1-5 คน: PSU Student and Staff ฟรี 0 บาท, PSU Alumni and General Student 375 บาท, General Adult 1050 บาท. ภาพนี้ระบุว่าค่าบริการมีผลตั้งแต่ 1 February 2026 ถึง 31 March 2026 และไม่ปรากฏราคา PC
- ค่าบริการ Nintendo Switch 3-4 คน ต่อ 60 นาที: นักศึกษา/บุคลากร PSU 0 บาท, ศิษย์เก่า PSU หรือ General Student 100 บาท, บุคคลทั่วไป 280 บาท
- ค่าบริการ PlayStation 5 ต่อ 60 นาที: นักศึกษา/บุคลากร PSU 0 บาท, ศิษย์เก่า PSU หรือ General Student 50 บาท, บุคคลทั่วไป 150 บาท

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (curated_service_fee_nintendo_1_2_rates)
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (curated_s
... [trimmed output]

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `rag_direct_curated` | model: `direct_from_retrieved_curated_fact` | elapsed: `0.120` sec


---

## 307. [PASS] ถูก

**คำถาม:** ถ้าเป็น PSU staff เล่น VR 1 ชั่วโมงต้องจ่ายไหม

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (VR 1 ชั่วโมง)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.188` sec


---

## 308. [PASS] ถูก

**คำถาม:** ต่างมหาลัยเล่น VR 30 นาที กับ VR 1 ชั่วโมงต่างกันเท่าไหร่

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: บาท
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: calculation

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.191` sec


---

## 309. [PASS] ถูก

**คำถาม:** ถ้ากินข้าวเสียงดังแล้วทำจอยพัง จะเกี่ยวกับกฎไหนบ้าง

**คำตอบ(จาก AI):**

คำตอบ:
กรณีนี้เกี่ยวกับหลายกฎพร้อมกัน: อาหารและเครื่องดื่มรับประทานได้เฉพาะพื้นที่ที่กำหนด, ควรงดส่งเสียงดังเกินควร, และถ้าทำอุปกรณ์เสียหายผู้ใช้งานต้องรับผิดชอบค่าปรับหรือค่าชดเชยตามระดับความเสียหาย

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_mixed_food_noise_damage / Reservation, curated_rule_food_drinks, curated_rule_noise_language, curated_damage_minor, curated_damage_moderate, curated_damage_severe)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: เฉพาะ, เสียงดัง
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: multi_fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.003` sec


---

## 310. [PASS] ถูก

**คำถาม:** ถ้าของหายกับทำอุปกรณ์เปียก ศูนย์รับผิดชอบไหม

**คำตอบ(จาก AI):**

คำตอบ:
กรณีทรัพย์สินส่วนตัวสูญหาย ศูนย์ขอสงวนสิทธิ์ไม่รับผิดชอบในทุกกรณี ส่วนกรณีทำอุปกรณ์เสียหายหรือเปียก ผู้ใช้งานต้องรับผิดชอบค่าปรับหรือค่าชดเชยตามระดับความเสียหาย

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_mixed_lost_and_damage / Reservation, curated_rule_lost_items, curated_damage_minor, curated_damage_moderate, curated_damage_severe)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่รับผิดชอบ, รับผิดชอบ
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: multi_fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.001` sec


---

## 311. [PASS] ถูก

**คำถาม:** ถ้าสูบบุหรี่และเล่นพนันในศูนย์ผิดกฎไหม

**คำตอบ(จาก AI):**

คำตอบ:
ห้ามพกอาวุธหรือของมีคม ห้ามทะเลาะวิวาท และห้ามเล่นการพนัน

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_weapons_gambling / Reservation, curated_rule_weapons_gambling)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: multi_fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.060` sec


---

## 312. [PASS] ถูก

**คำถาม:** ถ้ายืมแผ่นเกมแล้วไม่คืน หลังใช้งานต้องทำยังไง

**คำตอบ(จาก AI):**

คำตอบ:
กรุณานำอุปกรณ์และแผ่นเกมที่เบิกไปใช้งานมาคืนหลังจากใช้งานเสร็จ

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_return_equipment_games / Reservation, curated_rule_return_equipment, reservation-studio-rules)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: คืน, หลังจากใช้งานเสร็จ
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: multi_fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.074` sec


---

## 313. [PASS] ถูก

**คำถาม:** ถ้าพบปัญหาเครื่องตอนเล่นควรแจ้งใคร

**คำตอบ(จาก AI):**

คำตอบ:
หากพบปัญหาการใช้งาน พฤติกรรมที่ไม่เหมาะสม หรือข้อกังวลใด ๆ ควรแจ้งเจ้าหน้าที่ทันที

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_report_problem / Reservation, curated_rule_report_problem)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: แจ้งเจ้าหน้าที่
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: multi_fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.053` sec


---

## 314. [PASS] ถูก

**คำถาม:** เปิดปิด วันจัน กีโมงอะ ตอบสั้นๆ

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* จึงยังไม่ใช่ช่วงเล่น/เปิดให้บริการ ส่วนช่วง Afternoon 13:00–16:00 เป็น Open for Service ดังนั้นวันจันทร์เปิดให้เล่นช่วง 13:00 น. และปิด 16:00 น.

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_open_close / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.044` sec


---

## 315. [PASS] ถูก

**คำถาม:** เด็ก มอ เล่นเพลย์ห้าเสียกี่บาท ตอบสั้นๆ

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 0, PlayStation 5
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.108` sec


---

## 316. [PASS] ถูก

**คำถาม:** ต่างมหาลัย เล่น vr ครึ่ง ชม เท่าไหร่ ตอบสั้นๆ

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 190, VR
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.165` sec


---

## 317. [PASS] ถูก

**คำถาม:** คอมมีวาโลไหม ตอบสั้นๆ

**คำตอบ(จาก AI):**

คำตอบ:
เกม PC ที่มีในรายการ ได้แก่ Tekken 8, Counter-Strike 2, League of Legends, PUBG: BATTLEGROUNDS, VALORANT และ Call of Duty: Warzone

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_pc_specific_games / Reservation, curated_games_pc, reservation-service-pc)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VALORANT
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.037` sec


---

## 318. [PASS] ถูก

**คำถาม:** เชคอินก่อนกี่นาที ตอบสั้นๆ

**คำตอบ(จาก AI):**

คำตอบ:
เช็คอินได้ล่วงหน้าสูงสุด 30 นาที และต้องเช็คอินก่อนเวลาเริ่มต้นของรอบที่จอง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_checkin_advance / curated_checkin_30_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 30 นาที
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.036` sec


---

## 319. [PASS] ถูก

**คำถาม:** สูบบุหรี่ได้ปะ ตอบสั้นๆ

**คำตอบ(จาก AI):**

คำตอบ:
ศูนย์ห้ามสูบบุหรี่ เสพสารเสพติด หรือดื่มเครื่องดื่มแอลกอฮอล์ภายในศูนย์

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_smoking_alcohol / Reservation, curated_rule_smoking_alcohol_drugs)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.025` sec


---

## 320. [PASS] ถูก

**คำถาม:** มีให้เช่าจอไปบ้านไหม ตอบสั้นๆ

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องบริการส่งหรือให้เช่าเครื่องเกม/จอกลับไปใช้ที่บ้านในฐานข้อมูลที่มี ข้อมูลที่มีเป็นการจองใช้อุปกรณ์ภายใน PSU Esports Studio ตามรอบเวลา ดังนั้นควรจองเพื่อใช้งานที่ศูนย์ หรือสอบถามเจ้าหน้าที่หากต้องการบริการนอกสถานที่

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_home_delivery_console / Reservation)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.042` sec


---

## 321. [PASS] ถูก

**คำถาม:** ขอเฟสศูนย์หน่อย ตอบสั้นๆ

**คำตอบ(จาก AI):**

คำตอบ:
Facebook ของศูนย์คือ https://www.facebook.com/psuesportsphuket

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/Contact-Us (rule_contact_facebook / curated_contact_facebook)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: facebook.com/psuesportsphuket
Expected source keywords: Contact
หมวด: contact
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Contact']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.035` sec


---

## 322. [PASS] ถูก

**คำถาม:** เปิดปิด วันจัน กีโมงอะ ในเว็บบอกว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* จึงยังไม่ใช่ช่วงเล่น/เปิดให้บริการ ส่วนช่วง Afternoon 13:00–16:00 เป็น Open for Service ดังนั้นวันจันทร์เปิดให้เล่นช่วง 13:00 น. และปิด 16:00 น.

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_open_close / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.054` sec


---

## 323. [PASS] ถูก

**คำถาม:** เด็ก มอ เล่นเพลย์ห้าเสียกี่บาท ในเว็บบอกว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 0, PlayStation 5
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.139` sec


---

## 324. [PASS] ถูก

**คำถาม:** ต่างมหาลัย เล่น vr ครึ่ง ชม เท่าไหร่ ในเว็บบอกว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 190, VR
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.208` sec


---

## 325. [PASS] ถูก

**คำถาม:** คอมมีวาโลไหม ในเว็บบอกว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
เกม PC ที่มีในรายการ ได้แก่ Tekken 8, Counter-Strike 2, League of Legends, PUBG: BATTLEGROUNDS, VALORANT และ Call of Duty: Warzone

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_pc_specific_games / Reservation, curated_games_pc, reservation-service-pc)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VALORANT
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.051` sec


---

## 326. [PASS] ถูก

**คำถาม:** เชคอินก่อนกี่นาที ในเว็บบอกว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
เช็คอินได้ล่วงหน้าสูงสุด 30 นาที และต้องเช็คอินก่อนเวลาเริ่มต้นของรอบที่จอง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_checkin_advance / curated_checkin_30_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 30 นาที
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.046` sec


---

## 327. [PASS] ถูก

**คำถาม:** สูบบุหรี่ได้ปะ ในเว็บบอกว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
ศูนย์ห้ามสูบบุหรี่ เสพสารเสพติด หรือดื่มเครื่องดื่มแอลกอฮอล์ภายในศูนย์

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_smoking_alcohol / Reservation, curated_rule_smoking_alcohol_drugs)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.044` sec


---

## 328. [PASS] ถูก

**คำถาม:** มีให้เช่าจอไปบ้านไหม ในเว็บบอกว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องบริการส่งหรือให้เช่าเครื่องเกม/จอกลับไปใช้ที่บ้านในฐานข้อมูลที่มี ข้อมูลที่มีเป็นการจองใช้อุปกรณ์ภายใน PSU Esports Studio ตามรอบเวลา ดังนั้นควรจองเพื่อใช้งานที่ศูนย์ หรือสอบถามเจ้าหน้าที่หากต้องการบริการนอกสถานที่

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_home_delivery_console / Reservation)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.051` sec


---

## 329. [PASS] ถูก

**คำถาม:** ขอเฟสศูนย์หน่อย ในเว็บบอกว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
Facebook ของศูนย์คือ https://www.facebook.com/psuesportsphuket

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/Contact-Us (rule_contact_facebook / curated_contact_facebook)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: facebook.com/psuesportsphuket
Expected source keywords: Contact
หมวด: contact
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Contact']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.045` sec


---

## 330. [PASS] ถูก

**คำถาม:** เปิดปิด วันจัน กีโมงอะ ถามแทนเพื่อน

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* จึงยังไม่ใช่ช่วงเล่น/เปิดให้บริการ ส่วนช่วง Afternoon 13:00–16:00 เป็น Open for Service ดังนั้นวันจันทร์เปิดให้เล่นช่วง 13:00 น. และปิด 16:00 น.

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_open_close / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.055` sec


---

## 331. [PASS] ถูก

**คำถาม:** เด็ก มอ เล่นเพลย์ห้าเสียกี่บาท ถามแทนเพื่อน

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 0, PlayStation 5
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.122` sec


---

## 332. [PASS] ถูก

**คำถาม:** ต่างมหาลัย เล่น vr ครึ่ง ชม เท่าไหร่ ถามแทนเพื่อน

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 190, VR
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.187` sec


---

## 333. [PASS] ถูก

**คำถาม:** คอมมีวาโลไหม ถามแทนเพื่อน

**คำตอบ(จาก AI):**

คำตอบ:
เกม PC ที่มีในรายการ ได้แก่ Tekken 8, Counter-Strike 2, League of Legends, PUBG: BATTLEGROUNDS, VALORANT และ Call of Duty: Warzone

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_pc_specific_games / Reservation, curated_games_pc, reservation-service-pc)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VALORANT
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.048` sec


---

## 334. [PASS] ถูก

**คำถาม:** เชคอินก่อนกี่นาที ถามแทนเพื่อน

**คำตอบ(จาก AI):**

คำตอบ:
เช็คอินได้ล่วงหน้าสูงสุด 30 นาที และต้องเช็คอินก่อนเวลาเริ่มต้นของรอบที่จอง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_checkin_advance / curated_checkin_30_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 30 นาที
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.044` sec


---

## 335. [PASS] ถูก

**คำถาม:** สูบบุหรี่ได้ปะ ถามแทนเพื่อน

**คำตอบ(จาก AI):**

คำตอบ:
ศูนย์ห้ามสูบบุหรี่ เสพสารเสพติด หรือดื่มเครื่องดื่มแอลกอฮอล์ภายในศูนย์

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_smoking_alcohol / Reservation, curated_rule_smoking_alcohol_drugs)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.037` sec


---

## 336. [PASS] ถูก

**คำถาม:** มีให้เช่าจอไปบ้านไหม ถามแทนเพื่อน

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องบริการส่งหรือให้เช่าเครื่องเกม/จอกลับไปใช้ที่บ้านในฐานข้อมูลที่มี ข้อมูลที่มีเป็นการจองใช้อุปกรณ์ภายใน PSU Esports Studio ตามรอบเวลา ดังนั้นควรจองเพื่อใช้งานที่ศูนย์ หรือสอบถามเจ้าหน้าที่หากต้องการบริการนอกสถานที่

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_home_delivery_console / Reservation)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.054` sec


---

## 337. [PASS] ถูก

**คำถาม:** ขอเฟสศูนย์หน่อย ถามแทนเพื่อน

**คำตอบ(จาก AI):**

คำตอบ:
Facebook ของศูนย์คือ https://www.facebook.com/psuesportsphuket

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/Contact-Us (rule_contact_facebook / curated_contact_facebook)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: facebook.com/psuesportsphuket
Expected source keywords: Contact
หมวด: contact
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Contact']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.050` sec


---

## 338. [PASS] ถูก

**คำถาม:** เปิดปิด วันจัน กีโมงอะ แบบภาษาคนทั่วไป

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* จึงยังไม่ใช่ช่วงเล่น/เปิดให้บริการ ส่วนช่วง Afternoon 13:00–16:00 เป็น Open for Service ดังนั้นวันจันทร์เปิดให้เล่นช่วง 13:00 น. และปิด 16:00 น.

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_open_close / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.059` sec


---

## 339. [PASS] ถูก

**คำถาม:** เด็ก มอ เล่นเพลย์ห้าเสียกี่บาท แบบภาษาคนทั่วไป

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 150 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: บุคคลทั่วไป

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 0, PlayStation 5
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.124` sec


---

## 340. [PASS] ถูก

**คำถาม:** ต่างมหาลัย เล่น vr ครึ่ง ชม เท่าไหร่ แบบภาษาคนทั่วไป

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 190, VR
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.215` sec


---

## 341. [PASS] ถูก

**คำถาม:** คอมมีวาโลไหม แบบภาษาคนทั่วไป

**คำตอบ(จาก AI):**

คำตอบ:
เกม PC ที่มีในรายการ ได้แก่ Tekken 8, Counter-Strike 2, League of Legends, PUBG: BATTLEGROUNDS, VALORANT และ Call of Duty: Warzone

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_pc_specific_games / Reservation, curated_games_pc, reservation-service-pc)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VALORANT
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.052` sec


---

## 342. [PASS] ถูก

**คำถาม:** เชคอินก่อนกี่นาที แบบภาษาคนทั่วไป

**คำตอบ(จาก AI):**

คำตอบ:
เช็คอินได้ล่วงหน้าสูงสุด 30 นาที และต้องเช็คอินก่อนเวลาเริ่มต้นของรอบที่จอง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_checkin_advance / curated_checkin_30_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 30 นาที
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.050` sec


---

## 343. [PASS] ถูก

**คำถาม:** สูบบุหรี่ได้ปะ แบบภาษาคนทั่วไป

**คำตอบ(จาก AI):**

คำตอบ:
ศูนย์ห้ามสูบบุหรี่ เสพสารเสพติด หรือดื่มเครื่องดื่มแอลกอฮอล์ภายในศูนย์

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_smoking_alcohol / Reservation, curated_rule_smoking_alcohol_drugs)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.041` sec


---

## 344. [PASS] ถูก

**คำถาม:** มีให้เช่าจอไปบ้านไหม แบบภาษาคนทั่วไป

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องบริการส่งหรือให้เช่าเครื่องเกม/จอกลับไปใช้ที่บ้านในฐานข้อมูลที่มี ข้อมูลที่มีเป็นการจองใช้อุปกรณ์ภายใน PSU Esports Studio ตามรอบเวลา ดังนั้นควรจองเพื่อใช้งานที่ศูนย์ หรือสอบถามเจ้าหน้าที่หากต้องการบริการนอกสถานที่

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_home_delivery_console / Reservation)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.054` sec


---

## 345. [PASS] ถูก

**คำถาม:** ขอเฟสศูนย์หน่อย แบบภาษาคนทั่วไป

**คำตอบ(จาก AI):**

คำตอบ:
Facebook ของศูนย์คือ https://www.facebook.com/psuesportsphuket

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/Contact-Us (rule_contact_facebook / curated_contact_facebook)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: facebook.com/psuesportsphuket
Expected source keywords: Contact
หมวด: contact
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Contact']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.044` sec


---

## 346. [PASS] ถูก

**คำถาม:** เปิดปิด วันจัน กีโมงอะ ขอสรุปเร็วๆ

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* จึงยังไม่ใช่ช่วงเล่น/เปิดให้บริการ ส่วนช่วง Afternoon 13:00–16:00 เป็น Open for Service ดังนั้นวันจันทร์เปิดให้เล่นช่วง 13:00 น. และปิด 16:00 น.

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_open_close / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.051` sec


---

## 347. [PASS] ถูก

**คำถาม:** เด็ก มอ เล่นเพลย์ห้าเสียกี่บาท ขอสรุปเร็วๆ

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 0, PlayStation 5
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.115` sec


---

## 348. [PASS] ถูก

**คำถาม:** ต่างมหาลัย เล่น vr ครึ่ง ชม เท่าไหร่ ขอสรุปเร็วๆ

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 190, VR
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.184` sec


---

## 349. [PASS] ถูก

**คำถาม:** คอมมีวาโลไหม ขอสรุปเร็วๆ

**คำตอบ(จาก AI):**

คำตอบ:
เกม PC ที่มีในรายการ ได้แก่ Tekken 8, Counter-Strike 2, League of Legends, PUBG: BATTLEGROUNDS, VALORANT และ Call of Duty: Warzone

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_pc_specific_games / Reservation, curated_games_pc, reservation-service-pc)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VALORANT
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.041` sec


---

## 350. [PASS] ถูก

**คำถาม:** เชคอินก่อนกี่นาที ขอสรุปเร็วๆ

**คำตอบ(จาก AI):**

คำตอบ:
เช็คอินได้ล่วงหน้าสูงสุด 30 นาที และต้องเช็คอินก่อนเวลาเริ่มต้นของรอบที่จอง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_checkin_advance / curated_checkin_30_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 30 นาที
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.044` sec


---

## 351. [PASS] ถูก

**คำถาม:** สูบบุหรี่ได้ปะ ขอสรุปเร็วๆ

**คำตอบ(จาก AI):**

คำตอบ:
ศูนย์ห้ามสูบบุหรี่ เสพสารเสพติด หรือดื่มเครื่องดื่มแอลกอฮอล์ภายในศูนย์

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_smoking_alcohol / Reservation, curated_rule_smoking_alcohol_drugs)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.042` sec


---

## 352. [PASS] ถูก

**คำถาม:** มีให้เช่าจอไปบ้านไหม ขอสรุปเร็วๆ

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องบริการส่งหรือให้เช่าเครื่องเกม/จอกลับไปใช้ที่บ้านในฐานข้อมูลที่มี ข้อมูลที่มีเป็นการจองใช้อุปกรณ์ภายใน PSU Esports Studio ตามรอบเวลา ดังนั้นควรจองเพื่อใช้งานที่ศูนย์ หรือสอบถามเจ้าหน้าที่หากต้องการบริการนอกสถานที่

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_home_delivery_console / Reservation)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.044` sec


---

## 353. [PASS] ถูก

**คำถาม:** ขอเฟสศูนย์หน่อย ขอสรุปเร็วๆ

**คำตอบ(จาก AI):**

คำตอบ:
Facebook ของศูนย์คือ https://www.facebook.com/psuesportsphuket

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/Contact-Us (rule_contact_facebook / curated_contact_facebook)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: facebook.com/psuesportsphuket
Expected source keywords: Contact
หมวด: contact
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Contact']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.037` sec


---

## 354. [PASS] ถูก

**คำถาม:** เปิดปิด วันจัน กีโมงอะ ถ้าจะไปวันนี้ต้องรู้ว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
วันจันทร์ช่วง Morning 09:00–12:00 เป็น Maintenance* จึงยังไม่ใช่ช่วงเล่น/เปิดให้บริการ ส่วนช่วง Afternoon 13:00–16:00 เป็น Open for Service ดังนั้นวันจันทร์เปิดให้เล่นช่วง 13:00 น. และปิด 16:00 น.

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_schedule_monday_open_close / Reservation, curated_reservation_schedule_monday_morning, reservation-intro-schedule)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 13:00, 16:00, Maintenance
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.080` sec


---

## 355. [PASS] ถูก

**คำถาม:** เด็ก มอ เล่นเพลย์ห้าเสียกี่บาท ถ้าจะไปวันนี้ต้องรู้ว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 0 บาทต่อ 60 นาที (PlayStation 5)
- กลุ่มผู้ใช้: นักศึกษา/บุคลากร PSU

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 0, PlayStation 5
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.191` sec


---

## 356. [PASS] ถูก

**คำถาม:** ต่างมหาลัย เล่น vr ครึ่ง ชม เท่าไหร่ ถ้าจะไปวันนี้ต้องรู้ว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
- ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 190, VR
Expected source keywords: service_fee
หมวด: service_fee
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['service_fee']`
- route: `deterministic_calculator` | model: `calculator` | elapsed: `0.275` sec


---

## 357. [PASS] ถูก

**คำถาม:** คอมมีวาโลไหม ถ้าจะไปวันนี้ต้องรู้ว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
เกม PC ที่มีในรายการ ได้แก่ Tekken 8, Counter-Strike 2, League of Legends, PUBG: BATTLEGROUNDS, VALORANT และ Call of Duty: Warzone

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_pc_specific_games / Reservation, curated_games_pc, reservation-service-pc)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: VALORANT
Expected source keywords: Reservation
หมวด: games
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.080` sec


---

## 358. [PASS] ถูก

**คำถาม:** เชคอินก่อนกี่นาที ถ้าจะไปวันนี้ต้องรู้ว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
เช็คอินได้ล่วงหน้าสูงสุด 30 นาที และต้องเช็คอินก่อนเวลาเริ่มต้นของรอบที่จอง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_checkin_advance / curated_checkin_30_minutes)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: 30 นาที
Expected source keywords: Reservation
หมวด: reservation
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.068` sec


---

## 359. [PASS] ถูก

**คำถาม:** สูบบุหรี่ได้ปะ ถ้าจะไปวันนี้ต้องรู้ว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
ศูนย์ห้ามสูบบุหรี่ เสพสารเสพติด หรือดื่มเครื่องดื่มแอลกอฮอล์ภายในศูนย์

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_smoking_alcohol / Reservation, curated_rule_smoking_alcohol_drugs)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ห้าม
Expected source keywords: Reservation
หมวด: rules
ชนิดคำตอบ: fact

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True` | matched: `['Reservation']`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.062` sec


---

## 360. [PASS] ถูก

**คำถาม:** มีให้เช่าจอไปบ้านไหม ถ้าจะไปวันนี้ต้องรู้ว่าไง

**คำตอบ(จาก AI):**

คำตอบ:
ยังไม่พบข้อมูลเรื่องบริการส่งหรือให้เช่าเครื่องเกม/จอกลับไปใช้ที่บ้านในฐานข้อมูลที่มี ข้อมูลที่มีเป็นการจองใช้อุปกรณ์ภายใน PSU Esports Studio ตามรอบเวลา ดังนั้นควรจองเพื่อใช้งานที่ศูนย์ หรือสอบถามเจ้าหน้าที่หากต้องการบริการนอกสถานที่

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_no_answer_home_delivery_console / Reservation)

**เฉลย/เกณฑ์ที่ถูก:**

ต้องมีคำสำคัญ: ไม่พบข้อมูล
หมวด: no_answer
ชนิดคำตอบ: no_answer

**ผลตรวจ:**
- สถานะ: **ถูก**
- keyword_ok: `True`
- source_ok: `True`
- route: `rule_fast_path` | model: `rule_based` | elapsed: `0.075` sec


saved markdown: c:\Users\Chokhun\Downloads\Learn-LLM\16_PSU_Esports_RAG_Experiment_Timeline\ground_truth_verbose_v2_full_20260701_233959.md
saved jsonl: c:\Users\Chokhun\Downloads\Learn-LLM\16_PSU_Esports_RAG_Experiment_Timeline\ground_truth_verbose_v2_full_20260701_233959.jsonl


## 7. Next steps

หลังจาก notebook นี้ตอบได้แล้ว ขั้นถัดไปคือ:

1. เพิ่ม PDF/gฎ official เข้า data pipeline
2. เพิ่ม ground truth เป็น 30-50 ข้อ
3. ปรับ prompt ให้ตอบสั้น/ยาวตามที่ต้องการ
4. ทำ FastAPI endpoint `/chat`
5. ทำ UI demo ง่าย ๆ
6. ทำ Docker Compose
7. ต่อ Facebook Messenger Webhook


In [17]:
question = "PS5 มีเกมอะไรบ้าง"
answer, hits, elapsed = answer_question(question)
print(answer)

คำตอบ:
เกม PlayStation 5 ที่มีในรายการ ได้แก่ Call of Duty: Modern Warfare III, Delta Force, EA Sports FC 24, eFootball, FINAL FANTASY XVI, Fortnite, God of War Ragnarok, Hogwarts Legacy, Marvel’s Spider-Man 2, Naruto X Boruto Ultimate Ninja Storm Connections, Resident Evil 4, Resident Evil Village, TEKKEN 8, THE FINALS, The Last of Us Part I, The Last of Us Part II Remastered และ Uncharted: Legacy of Thieves Collection

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_ps5_games / Reservation, curated_games_ps5, reservation-service-ps5)


In [18]:
question1 = "PSU Esports Studio Phuket คืออะไร"
question2 = "Mission ของศูนย์คืออะไร"
question3 = "เช็คอินล่วงหน้าได้กี่นาที"
question4 = "ถ้าทำจอแตกต้องชดเชยยังไง"
question5 = "ติดต่อศูนย์ได้ทางไหน"

answer1, hits, elapsed = answer_question(question1)
print(answer1)
answer2, hits, elapsed = answer_question(question2)
print(answer2)
answer3, hits, elapsed = answer_question(question3)
print(answer3)
answer4, hits, elapsed = answer_question(question4)
print(answer4)
answer5, hits, elapsed = answer_question(question5)
print(answer5)

คำตอบ:
PSU Esports Studio - Phuket คือศูนย์พัฒนาการเรียนรู้ด้านอีสปอร์ตเพื่อความเป็นเลิศและขับเคลื่อนเศรษฐกิจในพื้นที่ภาคใต้ สาขาภูเก็ต ของมหาวิทยาลัยสงขลานครินทร์ และดำเนินการโดยวิทยาลัยการคอมพิวเตอร์

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (rule_overview_identity / curated_overview_identity)
คำตอบ:
PSU Esports Studio - Phuket คือศูนย์พัฒนาการเรียนรู้ด้านอีสปอร์ตเพื่อความเป็นเลิศและขับเคลื่อนเศรษฐกิจในพื้นที่ภาคใต้ สาขาภูเก็ต ของมหาวิทยาลัยสงขลานครินทร์ และดำเนินการโดยวิทยาลัยการคอมพิวเตอร์

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (rule_overview_identity / curated_overview_identity)
คำตอบ:
เช็คอินได้ล่วงหน้าสูงสุด 30 นาที และต้องเช็คอินก่อนเวลาเริ่มต้นของรอบที่จอง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_checkin_advance / curated_checkin_30_minutes)
คำตอบ:
ความเสียหายร้ายแรง เช่น จอแตก คอมพิวเตอร์พัง เฟอร์นิเจอร์เสียหายจนใช้ไม่ได้ หรืออุปกรณ์ใช้งานไม่ได้ ต้องชดเชยราคาทรัพย์สินเต็มจำนวนตามราคากลาง

แหล่งข้อมูล:
- https://esports.computing.ps

In [19]:
[(h["id"], h["metadata"].get("category"), h["metadata"].get("title")) for h in hits]

[('curated_contact_facebook', 'contact', 'Facebook ติดต่อ'),
 ('curated_contact_email', 'contact', 'อีเมลติดต่อ'),
 ('curated_contact_location', 'contact', 'ที่ตั้งศูนย์'),
 ('curated_contact_phone', 'contact', 'เบอร์ติดต่อจากระบบจอง')]

In [20]:
answer, hits, elapsed = answer_question("เช็คอินล่วงหน้าได้กี่นาที")
print(elapsed)
print(answer)

0.046
คำตอบ:
เช็คอินได้ล่วงหน้าสูงสุด 30 นาที และต้องเช็คอินก่อนเวลาเริ่มต้นของรอบที่จอง

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/ (rule_checkin_advance / curated_checkin_30_minutes)


In [21]:
answer, hits, elapsed = answer_question("ศูนย์นี้เกี่ยวกับอะไร", use_rules=False)
print(elapsed)
print(answer)

0.098
คำตอบ:
- PSU Esports Studio - Phuket คือศูนย์พัฒนาการเรียนรู้ด้านอีสปอร์ตเพื่อความเป็นเลิศและขับเคลื่อนเศรษฐกิจในพื้นที่ภาคใต้ สาขาภูเก็ต เป็นศูนย์การเรียนรู้ผ่านเกมและอีสปอร์ตของมหาวิทยาลัยสงขลานครินทร์
- Mission ของ PSU Esports Studio - Phuket คือการยกระดับการศึกษาและความเป็นเลิศด้านอีสปอร์ต ผ่านสิ่งอำนวยความสะดวกและอุปกรณ์ที่ช่วยเสริมสร้างการเรียนรู้ให้กับนักเล่นเกม นักศึกษา และผู้สนใจ โดยก่อตั้งโดยมหาวิทยาลัยสงขลานครินทร์และดำเนินการโดยวิทยาลัยการคอมพิวเตอร์

แหล่งข้อมูล:
- https://esports.phuket.psu.ac.th/home (curated_overview_identity)
- https://esports.phuket.psu.ac.th/home (curated_overview_mission)


In [22]:
print(len(RULES))

77


In [23]:
answer, hits, elapsed = answer_question("ถ้าจอง 9โมงถึง11โมง แบบเล่น PC ต้องเสียกี่บาท", use_rules=True, use_direct=False)
print(elapsed)
print([h["id"] for h in hits])
print(answer)

0.214
['calculator_pc']
คำตอบ:
- ราคา PC: ยังไม่พบราคาค่าบริการ PC ในฐานข้อมูล/Service Fee 2026 ที่ดึงมา จึงยังไม่ควรคำนวณยอดเงินบาทแบบฟันธง
- ช่วงเวลาที่ถามคือ 09:00-11:00 = 2 ชั่วโมง
- บริการ PC คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 2 session(s)
- ยังไม่ทราบกลุ่มผู้ใช้ จึงยังเทียบเรตราคาเฉพาะกลุ่มไม่ได้
- จากภาพ Service Fee 2026 ที่มีตอนนี้ มีราคา PlayStation 5, Nintendo Switch, Cockpit และ VR แต่ไม่ปรากฏราคา PC
- ถ้าได้รับราคา PC ต่อ 1 session แล้ว ระบบจะคำนวณได้ทันทีด้วยสูตร: จำนวน session × ราคาต่อ session

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-json/wbk/v2/get-preset (service duration)
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (service fee image; PC price not shown)


In [24]:
# Manual Ask Cell: ใช้สำหรับพิมพ์ถามเอง และดูว่า Router เลือก Rule Base / RAG / LLM ทางไหน
from IPython.display import Markdown, display
import json
from pathlib import Path

ROUTE_EXPLAIN = {
    "deterministic_calculator": "คำนวณด้วย logic ตรงๆ เช่น เวลา/session ก่อนตอบ",
    "rule_fast_path": "Rule Base: คำถามตรง pattern ชัดเจน ตอบเร็วและคุมคำตอบได้",
    "rag_direct_curated": "RAG Direct: ดึงข้อมูล curated/chunk ที่มั่นใจ แล้วตอบจากข้อมูลตรงๆ",
    "rag_llm": "RAG + LLM: ดึงข้อมูลก่อน แล้วให้ LLM เรียบเรียงคำตอบ",
    "unknown": "ยังอ่าน route ไม่ได้ อาจเกิดจาก log ยังไม่ถูกเขียน",
}


def _read_last_chat_log():
    path = Path(LOG_PATH)
    if not path.exists():
        return {}
    lines = [line for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not lines:
        return {}
    try:
        return json.loads(lines[-1])
    except json.JSONDecodeError:
        return {}


def _hit_preview(hit, limit=350):
    text = hit.get("text") or hit.get("document") or ""
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[:limit].rstrip() + "..."


def _format_sources_markdown(hits, max_sources=5):
    if not hits:
        return "ไม่มี retrieved sources"

    lines = []
    for i, hit in enumerate(hits[:max_sources], 1):
        meta = hit.get("metadata", {})
        score = hit.get("score", "-")
        lines.append(
            f"{i}. `{hit.get('id', '-')}` | category: `{meta.get('category', '-')}` | score: `{score}`  \n"
            f"   title: {meta.get('title', '-')}  \n"
            f"   url: {meta.get('source_url', '-')}  \n"
            f"   preview: {_hit_preview(hit)}"
        )
    return "\n\n".join(lines)


def ask_manual(question=None, use_rules=True, use_direct=True, top_k=TOP_K, show_sources=True):
    """ถาม AI เองใน notebook พร้อมดู route ที่ระบบเลือกใช้"""
    if question is None or not str(question).strip():
        question = input("พิมพ์คำถาม: ").strip()
    if not question:
        print("ยังไม่ได้ใส่คำถาม")
        return None

    answer, hits, elapsed = answer_question(
        question,
        top_k=top_k,
        use_rules=use_rules,
        use_direct=use_direct,
    )

    log_row = _read_last_chat_log()
    mode = log_row.get("mode", "unknown")
    model = log_row.get("model", "-")
    route_detail = ROUTE_EXPLAIN.get(mode, ROUTE_EXPLAIN["unknown"])

    md = [
        f"### คำถาม\n{question}",
        f"### Route ที่ใช้\n`{mode}` - {route_detail}\n\nmodel: `{model}` | elapsed: `{elapsed:.3f}` sec | hits: `{len(hits)}`",
        f"### คำตอบจาก AI\n{answer}",
    ]

    if show_sources:
        md.append(f"### Retrieved Sources\n{_format_sources_markdown(hits)}")

    display(Markdown("\n\n---\n\n".join(md)))

    return {
        "question": question,
        "answer": answer,
        "hits": hits,
        "elapsed": elapsed,
        "mode": mode,
        "model": model,
        "log": log_row,
    }


def ask_loop(use_rules=True, use_direct=True, top_k=TOP_K, show_sources=False):
    """โหมดถามต่อเนื่องใน notebook: พิมพ์ exit/quit/q เพื่อออก"""
    print("พิมพ์คำถามได้เลย | พิมพ์ exit, quit หรือ q เพื่อออก")
    while True:
        question = input("ถาม> ").strip()
        if question.lower() in {"exit", "quit", "q"}:
            print("จบการถาม")
            break
        if not question:
            continue
        ask_manual(
            question,
            use_rules=use_rules,
            use_direct=use_direct,
            top_k=top_k,
            show_sources=show_sources,
        )


# ตัวอย่างใช้งาน
# result = ask_manual("ศูนย์เปิดกี่โมงปิดกี่โมง")
# result = ask_manual("ช่วยสรุปขั้นตอนการจองให้หน่อย", use_rules=False, use_direct=False)
# ask_loop(show_sources=False)


In [25]:
result = ask_manual("ถ้าเป็นนักศึกษาจากกระบี่ ต้องเสียค่าเล่น PC เท่าไหร่ต่อชั่วโมง")

### คำถาม
ถ้าเป็นนักศึกษาจากกระบี่ ต้องเสียค่าเล่น PC เท่าไหร่ต่อชั่วโมง

---

### Route ที่ใช้
`deterministic_calculator` - คำนวณด้วย logic ตรงๆ เช่น เวลา/session ก่อนตอบ

model: `calculator` | elapsed: `0.088` sec | hits: `1`

---

### คำตอบจาก AI
คำตอบ:
- ราคา PC: ยังไม่พบราคาค่าบริการ PC ในฐานข้อมูล/Service Fee 2026 ที่ดึงมา จึงยังไม่ควรคำนวณยอดเงินบาทแบบฟันธง
- คำถามเป็นราคาแบบต่อรอบ/ต่อชั่วโมง จึงคิดเป็น 1 session = 1 ชั่วโมง
- บริการ PC คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s)
- กลุ่มผู้ใช้ที่ถาม: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)
- จากภาพ Service Fee 2026 ที่มีตอนนี้ มีราคา PlayStation 5, Nintendo Switch, Cockpit และ VR แต่ไม่ปรากฏราคา PC
- ถ้าได้รับราคา PC ต่อ 1 session แล้ว ระบบจะคำนวณได้ทันทีด้วยสูตร: จำนวน session × ราคาต่อ session

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-json/wbk/v2/get-preset (service duration)
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png (service fee image; PC price not shown)

---

### Retrieved Sources
1. `calculator_pc` | category: `service_fee` | score: `999.0`  
   title: reservation_price_calculator  
   url: https://esports.computing.psu.ac.th/wp-json/wbk/v2/get-preset  
   preview: คำตอบ: - ราคา PC: ยังไม่พบราคาค่าบริการ PC ในฐานข้อมูล/Service Fee 2026 ที่ดึงมา จึงยังไม่ควรคำนวณยอดเงินบาทแบบฟันธง - คำถามเป็นราคาแบบต่อรอบ/ต่อชั่วโมง จึงคิดเป็น 1 session = 1 ชั่วโมง - บริการ PC คิดเป็นรอบละ 60 นาที ดังนั้นต้องใช้ 1 session(s) - กลุ่มผู้ใช้ที่ถาม: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student) - จากภาพ Service Fee 2026 ที่มีตอ...

In [26]:
result = ask_manual(
    "วันจันทร์เปิดให้เล่นกีโมง ปิดกี่โมง",
    use_rules=False,
    use_direct=False
)

### คำถาม
วันจันทร์เปิดให้เล่นกีโมง ปิดกี่โมง

---

### Route ที่ใช้
`rag_llm` - RAG + LLM: ดึงข้อมูลก่อน แล้วให้ LLM เรียบเรียงคำตอบ

model: `qwen2.5:3b` | elapsed: `4.037` sec | hits: `4`

---

### คำตอบจาก AI
คำตอบ: ช่วงบ่ายของวันจันทร์เปิดบริการที่ 13:00 - 16:00

โดยข้อมูลจาก [SOURCE 2] และ [SOURCE 3], ช่วงเวลาเช้าและช่วงเวลาบ่ายได้ระบุไว้อย่างชัดเจนว่าเปิดให้บริการในช่วงเวลาที่กำหนด.

---

### Retrieved Sources
1. `curated_reservation_schedule_monday_morning` | category: `reservation` | score: `2.9746668112277983`  
   title: Reservation Schedule - Monday Morning Maintenance  
   url: https://esports.computing.psu.ac.th/  
   preview: Reservation Schedule - Monday Morning Maintenance ในตารางบริการ วัน Monday ช่วง Morning 09:00 – 12:00 เป็น Maintenance* และช่วง Afternoon 13:00 – 16:00 เปิด Open for Service

2. `curated_schedule_morning` | category: `reservation` | score: `2.9250688540935514`  
   title: ช่วงเวลาเช้า  
   url: https://esports.computing.psu.ac.th/  
   preview: ช่วงเวลาเช้า ตารางบริการช่วง Morning คือ 09:00 – 12:00

3. `curated_schedule_afternoon` | category: `reservation` | score: `2.916080354452133`  
   title: ช่วงเวลาบ่าย  
   url: https://esports.computing.psu.ac.th/  
   preview: ช่วงเวลาบ่าย ตารางบริการช่วง Afternoon คือ 13:00 – 16:00

4. `curated_time_change_policy` | category: `reservation` | score: `-0.01104646205902099`  
   title: เปลี่ยนเวลาใช้งาน  
   url: https://esports.computing.psu.ac.th/  
   preview: เปลี่ยนเวลาใช้งาน สามารถเปลี่ยนแปลงเวลาใช้งานได้ โดยต้องแจ้งล่วงหน้าก่อนเวลาที่จองไว้อย่างน้อย 1 ชั่วโมง หากแจ้งล่าช้าหรือไม่แจ้ง ศูนย์สงวนสิทธิ์ไม่คืนเงินและไม่ชดเชยเวลา

In [27]:
from IPython.display import Markdown, display
import time

def ask_one(question: str, show_sources: bool = True, show_hits: bool = True):
    start = time.time()
    answer, hits, elapsed = answer_question(question)
    wall = round(time.time() - start, 3)

    # อ่าน log ล่าสุดเพื่อดู route/mode
    mode = "-"
    model = "-"
    try:
        if LOG_PATH.exists():
            lines = [line for line in LOG_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
            if lines:
                last = json.loads(lines[-1])
                mode = last.get("mode", "-")
                model = last.get("model", "-")
    except Exception:
        pass

    md = []
    md.append("## คำถาม")
    md.append(question)
    md.append("")
    md.append("---")
    md.append("")
    md.append("## Route ที่ใช้")
    md.append(f"`{mode}`")
    md.append("")
    md.append(f"model: `{model}` | elapsed: `{elapsed}` sec | wall: `{wall}` sec | hits: `{len(hits)}`")
    md.append("")
    md.append("---")
    md.append("")
    md.append("## คำตอบจาก AI")
    md.append(answer)

    if show_sources and hits:
        md.append("")
        md.append("---")
        md.append("")
        md.append("## Retrieved Sources")
        for i, h in enumerate(hits, 1):
            meta = h.get("metadata", {})
            md.append(f"{i}. `{h.get('id')}`")
            md.append(f"   - category: `{meta.get('category', '-')}`")
            md.append(f"   - source_type: `{meta.get('source_type', '-')}`")
            md.append(f"   - url: {meta.get('source_url', '-')}")
            if show_hits:
                text = " ".join((h.get("text") or "").split())
                md.append(f"   - preview: {text[:350]}")

    display(Markdown("\n".join(md)))

    return {
        "question": question,
        "answer": answer,
        "hits": hits,
        "mode": mode,
        "model": model,
        "elapsed": elapsed,
        "wall": wall,
    }


# ใช้งานแบบนี้
result = ask_one("ต่างมหาลัย เล่น vr ครึ่ง ชม เท่าไหร่")

## คำถาม
ต่างมหาลัย เล่น vr ครึ่ง ชม เท่าไหร่

---

## Route ที่ใช้
`deterministic_calculator`

model: `calculator` | elapsed: `0.119` sec | wall: `0.119` sec | hits: `1`

---

## คำตอบจาก AI
คำตอบ:
- ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที)
- กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student)

แหล่งข้อมูล:
- https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

---

## Retrieved Sources
1. `calculator_vr_30`
   - category: `service_fee`
   - source_type: `calculator`
   - url: https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png
   - preview: คำตอบ: - ราคา: 190 บาทต่อ 30 นาที (VR 30 นาที) - กลุ่มผู้ใช้: ศิษย์เก่า PSU / นักศึกษาทั่วไป (General Student) แหล่งข้อมูล: - https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

In [28]:
result = ask_one("Tekken 8 ค้องเล่นในไหน")

## คำถาม
Tekken 8 ค้องเล่นในไหน

---

## Route ที่ใช้
`rag_llm`

model: `qwen2.5:3b` | elapsed: `4.086` sec | wall: `4.088` sec | hits: `4`

---

## คำตอบจาก AI
คำตอบ:
Tekken 8 เล่นบนคอนโซลหรือเครื่องเกมต่างๆ เช่น PC โดยดูจากแหล่งข้อมูลที่ให้รายละเอียดเกี่ยวกับ Tekken 8 ในหน้า "Services - Our Games" ของเว็บไซต์ esports.phuket.psu.ac.th

---

## Retrieved Sources
1. `curated_home_popular_games_list`
   - category: `games`
   - source_type: `curated_fact`
   - url: https://esports.phuket.psu.ac.th/home
   - preview: Home Popular Games - รายชื่อเกมยอดนิยม เกมยอดนิยมที่ปรากฏบนหน้า Home ได้แก่ Gran Turismo 7, Mario Kart 8 Deluxe, Tekken 8 และ Beat Saber
2. `curated_games_pc`
   - category: `games`
   - source_type: `curated_fact`
   - url: https://esports.computing.psu.ac.th/
   - preview: เกมบน PC เกมที่ปรากฏในรายการ PC ได้แก่ Tekken 8, Counter-Strike 2, League of Legends, PUBG: BATTLEGROUNDS, VALORANT และ Call of Duty: Warzone
3. `curated_games_popular`
   - category: `games`
   - source_type: `curated_fact`
   - url: https://esports.phuket.psu.ac.th/home
   - preview: เกมยอดนิยมบนหน้า Home เกมยอดนิยมที่ปรากฏบนหน้า Home ได้แก่ Gran Turismo 7, Mario Kart 8 Deluxe, Tekken 8 และ Beat Saber
4. `services-our-games-01-035`
   - category: `games`
   - source_type: `webscraping_structured`
   - url: https://esports.phuket.psu.ac.th/Services/our-games
   - preview: สัมผัสถึงพลังของทุกการโจมตีใน TEKKEN 8 ภาคล่าสุดของแฟรนไชส์เกมต่อสู้ในตำนานจาก Bandai Namco ใช้ขุมพลังและความสมจริงของ Unreal Engine 5, TEKKEN 8 ผลักดันขีดจำกัดของเกมต่อสู้ด้วยการใช้พลังของคอนโซลยุคใหม่อย่างเต็มที่ คุณสมบัติใหม่ๆ สุดล้ำ รูปร่างตัวละครที่มีรายละเอียดอย่างมากและสภาพแวดล้อมที่โดดเด่นทำให้เกมนี้เป็นเกมที่มีงานภาพที่น่าตื่นตาและชวนดื่มด

In [29]:
# FAST_RUNTIME_UPDATE_20260701
# Optional fast runtime from the Update folder.
# This path answers common PSU Esports FAQ/price/schedule/game questions without loading LLM.

import sys
from pathlib import Path

FAST_RUNTIME_DIR = Path(r"C:\Users\Chokhun\Downloads\Learn-LLM\18_PSU_Esports_Update_Route_Data")
if str(FAST_RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(FAST_RUNTIME_DIR))

from app.runtime.fast_answer import answer_question_fast as _answer_question_fast_update


def answer_question_fast_runtime(question: str):
    answer, hits, elapsed, mode = _answer_question_fast_update(question)
    return answer, hits, elapsed


def ask_fast(question: str):
    answer, hits, elapsed, mode = _answer_question_fast_update(question)
    print("คำถาม:", question)
    print("Route:", mode)
    print("Elapsed:", elapsed, "sec")
    print()
    print(answer)
    print()
    print("Sources:", [hit.get("id") for hit in hits])
    return answer, hits, elapsed, mode


# ถ้าต้องการให้ notebook ใช้ fast runtime เป็น answer_question หลัก ให้เปิดบรรทัดนี้:
# answer_question = answer_question_fast_runtime

ask_fast("ต่างมหาลัย เล่น vr ครึ่ง ชม เท่าไหร่")


คำถาม: ต่างมหาลัย เล่น vr ครึ่ง ชม เท่าไหร่
Route: deterministic_calculator_fast
Elapsed: 0.0003 sec

ราคา 190 บาท สำหรับกลุ่ม PSU Alumni and General Student / นักศึกษาหรือนักเรียนต่างสถาบัน
- VR 30 นาที ราคา 190 บาท

รายละเอียดจากตาราง:
VR 30 นาที (1-5 คน): PSU Student and Staff 0 บาท, PSU Alumni and General Student 190 บาท, General Adult 525 บาท
แหล่งข้อมูล: https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png

Sources: ['service_fee_image_2026']


('ราคา 190 บาท สำหรับกลุ่ม PSU Alumni and General Student / นักศึกษาหรือนักเรียนต่างสถาบัน\n- VR 30 นาที ราคา 190 บาท\n\nรายละเอียดจากตาราง:\nVR 30 นาที (1-5 คน): PSU Student and Staff 0 บาท, PSU Alumni and General Student 190 บาท, General Adult 525 บาท\nแหล่งข้อมูล: https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png',
 [{'id': 'service_fee_image_2026',
   'metadata': {'source_url': 'https://esports.computing.psu.ac.th/wp-content/uploads/2026/01/PSU-Esports-Studio-phuket-SERVICE-FEE-2026.png',
    'category': 'service_fee',
    'title': 'Service Fee 2026',
    'source_ids': ['service_fee_image_2026']}}],
 0.0003,
 'deterministic_calculator_fast')